<div dir="rtl" align="right">

# Notebook 17 — Reranking V1

## هدف این مرحله

NB17 مسئول بازمرتب‌سازی Candidateهای فریز‌شده Hybrid Retrieval است.

در NB16 برای هر Query حداکثر ۱۰۰ Chunk کاندید تولید شد. Hybrid Retriever وظیفه داشت شواهد مرتبط را با Recall مناسب وارد Candidate Pool کند؛ اما ترتیب نهایی این Candidateها هنوز می‌تواند بهبود پیدا کند.

در این Notebook چند Reranker کاندید روی Development مقایسه می‌شوند تا مشخص شود آیا می‌توان مرتبط‌ترین شواهد را با دقت بیشتری در رتبه‌های ابتدایی قرار داد.

## مأموریت‌های رسمی NB17

براساس نقشه ماژولار، این Notebook باید موارد زیر را انجام دهد:

1. ثبت Candidate Rerankerها؛
2. ساخت Pairهای Query و Chunk؛
3. ثبت Baseline بدون Reranker؛
4. ارزیابی Candidate Rerankerها؛
5. تحلیل زبان فارسی و سه منبع Book، Transcript و Website؛
6. اندازه‌گیری Latency و Cost؛
7. تعیین `candidate_k` و `final_k`؛
8. Failure Analysis؛
9. انتخاب تنظیم نهایی با ADR؛
10. اجرای Regression Gate.

## مرز مسئولیت

این Notebook مجاز نیست موارد زیر را تغییر دهد:

- Dense Index فریز‌شده NB14؛
- Sparse Index فریز‌شده NB15؛
- Hybrid Configuration فریز‌شده NB16؛
- وزن‌های Dense و Sparse؛
- Candidateهای ذخیره‌شده NB16؛
- Golden Dataset یا Split توسعه و Test؛
- Corpus Embeddingها.

NB17 فقط Candidateهای فریز‌شده NB16 را امتیازدهی مجدد و مرتب می‌کند.

</div>

<div dir="rtl" align="right">

## قرارداد اجرا، ورودی‌ها و معیار پذیرش

### ورودی‌های Canonical

ورودی اصلی NB17، Handoff نسخه‌دار NB16 است و باید به Artifactهای زیر متصل باشد:

- Hybrid Retrieval Release؛
- NB17 Handoff؛
- Hybrid Configuration فریز‌شده؛
- Hybrid Manifest؛
- Candidate Artifact شامل ۳۸۱۳ Query؛
- Corpus Canonical شامل ۳۴۸۸ Chunk؛
- Golden Dataset شامل ۳۸۱۳ Query؛
- Reload Validation و Acceptance نهایی NB16.

### سیاست جلوگیری از Data Leakage

تقسیم داده NB16 بدون تغییر استفاده می‌شود:

- Development: تعداد ۲۶۶۴ Query؛
- Test: تعداد ۱۱۴۹ Query.

انتخاب مدل، `candidate_k`، `final_k`، Batch Size و تمام تنظیمات Reranking فقط با Development انجام می‌شود.

Test تا زمان Freeze شدن Reranker Configuration و ADR قفل می‌ماند. نتیجه Test فقط برای ارزیابی مستقل نهایی استفاده می‌شود و اجازه تغییر Configuration را ندارد.

### Baseline رسمی

Baseline این Notebook همان ترتیب Hybrid فریز‌شده NB16 است:

- Candidate Depth: ۱۰۰؛
- Fusion: Weighted Score Fusion؛
- Dense Weight: ۰٫۴؛
- Sparse Weight: ۰٫۶؛
- Test `nDCG@10`: برابر 0.521665؛
- Test `Hit@10`: برابر 0.755440.

### شرایط PASS

NB17 فقط زمانی PASS می‌شود که:

- تمام ورودی‌ها با Hash و Status معتبر بارگذاری شوند؛
- Baseline ذخیره‌شده NB16 دقیقاً بازسازی شود؛
- Candidate Rerankerها فقط روی Development مقایسه شوند؛
- Latency، Cost، Slice Metrics و Failure Analysis ثبت شوند؛
- Configuration پیش از مشاهده Test فریز شود؛
- Regression Gate از افت غیرمجاز جلوگیری کند؛
- Test مستقل اجرا شود؛
- Artifactها از Disk مجدداً Load و Validate شوند؛
- `Restart Kernel → Run All` بدون خطا انجام شود؛
- Handoff نسخه‌دار برای NB18 ساخته شود.

</div>

In [1]:
# Code Cell 01 — Imports and runtime bootstrap only

import collections
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import statistics
import sys
import time
import unicodedata

from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import torch
import transformers

from sentence_transformers import (
    CrossEncoder,
    SentenceTransformer,
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Code Cell 02 — Project-root discovery and read-only NB17 preflight

NOTEBOOK_NAME = "17_reranking.ipynb"

EXPECTED_NB16_CONFIGURATION_SHA256 = (
    "e7dc28836b0e6b2f72cf19aa61507af4"
    "1a76df4980b056f39af015ed91ba26d5"
)

EXPECTED_NB16_MANIFEST_SHA256 = (
    "362b9cbfd33e2f8e307ffc6cc7dcffc4"
    "8d2cc308fe74f9bb0ba896d1e080032c"
)

EXPECTED_NB16_RELEASE_SHA256 = (
    "8953c5d4d20a0f0cb2cf9fb35d86c42"
    "1c06cf02b193aa2b19acba46640d0216e"
)

EXPECTED_NB17_HANDOFF_SHA256 = (
    "88e8c325f54f89c4aeed94787bc36048"
    "f9cc1d6c963e781f7254b6c8c04f7113"
)

EXPECTED_NB16_LOGICAL_RETRIEVAL_SHA256 = (
    "f54277680f3d20681b1585915f54bb44"
    "85ecb1fcccb15c27df0061ff0725d19a"
)

EXPECTED_CHUNK_SHA256 = (
    "3891c4ecad9332ba753cd605bbccd99da"
    "9d063f64934a30ee39e495349cffb50"
)

EXPECTED_GOLDEN_DATASET_SHA256 = (
    "b9ca18e710686ddca48a25c637840b7"
    "16adb306717cafd8ec7c96ce9e182e6f2"
)


def sha256_file(path):
    hasher = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(block)

    return hasher.hexdigest()


def discover_project_root():
    current_path = Path.cwd().resolve()

    candidate_roots = (
        current_path,
        *current_path.parents,
    )

    for candidate_root in candidate_roots:
        required_markers = (
            candidate_root
            / "notebooks/phase2"
            / NOTEBOOK_NAME,
            candidate_root
            / "artifacts/phase2"
            / "hybrid_retrieval"
            / "hybrid_retrieval_release_v1.json",
            candidate_root
            / "data/phase2/retrieval"
            / "hybrid_retrieval_v1"
            / "hybrid_candidates_v1.jsonl",
        )

        if all(
            marker.exists()
            for marker in required_markers
        ):
            return candidate_root

    raise FileNotFoundError(
        "Canonical RAG_finance project root "
        "could not be discovered."
    )


PROJECT_ROOT = discover_project_root()

NB16_CONFIGURATION_PATH = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
    / "hybrid_configuration_v1.json"
)

NB16_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/phase2/retrieval"
    / "hybrid_retrieval_v1"
    / "hybrid_retrieval_manifest.json"
)

NB16_RELEASE_PATH = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
    / "hybrid_retrieval_release_v1.json"
)

NB17_HANDOFF_PATH = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
    / "nb17_hybrid_retrieval_handoff_v1.json"
)

NB16_CANDIDATES_PATH = (
    PROJECT_ROOT
    / "data/phase2/retrieval"
    / "hybrid_retrieval_v1"
    / "hybrid_candidates_v1.jsonl"
)

NB16_RELOAD_VALIDATION_PATH = (
    PROJECT_ROOT
    / "data/phase2/retrieval"
    / "hybrid_retrieval_v1"
    / "hybrid_reload_validation_v1.json"
)

NB16_ACCEPTANCE_PATH = (
    PROJECT_ROOT
    / "reports/phase2"
    / "hybrid_retrieval_acceptance_v1.json"
)

CANONICAL_CHUNK_PATH = (
    PROJECT_ROOT
    / "data/phase2/chunks"
    / "nb11_benchmark_v1"
    / "baseline_phase1_char_safe_v1.jsonl"
)

GOLDEN_DATASET_PATH = (
    PROJECT_ROOT
    / "artifacts/phase2"
    / "golden_dataset_v2"
    / "golden_dataset_v2.jsonl"
)

required_paths = {
    "nb16_configuration": (
        NB16_CONFIGURATION_PATH
    ),
    "nb16_manifest": NB16_MANIFEST_PATH,
    "nb16_release": NB16_RELEASE_PATH,
    "nb17_handoff": NB17_HANDOFF_PATH,
    "nb16_candidates": NB16_CANDIDATES_PATH,
    "nb16_reload_validation": (
        NB16_RELOAD_VALIDATION_PATH
    ),
    "nb16_acceptance": NB16_ACCEPTANCE_PATH,
    "canonical_chunks": (
        CANONICAL_CHUNK_PATH
    ),
    "golden_dataset": GOLDEN_DATASET_PATH,
}

missing_required_paths = {
    name: str(path)
    for name, path in required_paths.items()
    if not path.is_file()
}

assert not missing_required_paths, (
    missing_required_paths
)


validated_hashes = {
    "nb16_configuration": sha256_file(
        NB16_CONFIGURATION_PATH
    ),
    "nb16_manifest": sha256_file(
        NB16_MANIFEST_PATH
    ),
    "nb16_release": sha256_file(
        NB16_RELEASE_PATH
    ),
    "nb17_handoff": sha256_file(
        NB17_HANDOFF_PATH
    ),
    "canonical_chunks": sha256_file(
        CANONICAL_CHUNK_PATH
    ),
    "golden_dataset": sha256_file(
        GOLDEN_DATASET_PATH
    ),
}

assert (
    validated_hashes[
        "nb16_configuration"
    ]
    == EXPECTED_NB16_CONFIGURATION_SHA256
)

assert (
    validated_hashes["nb16_manifest"]
    == EXPECTED_NB16_MANIFEST_SHA256
)

assert (
    validated_hashes["nb16_release"]
    == EXPECTED_NB16_RELEASE_SHA256
)

assert (
    validated_hashes["nb17_handoff"]
    == EXPECTED_NB17_HANDOFF_SHA256
)

assert (
    validated_hashes["canonical_chunks"]
    == EXPECTED_CHUNK_SHA256
)

assert (
    validated_hashes["golden_dataset"]
    == EXPECTED_GOLDEN_DATASET_SHA256
)


nb16_configuration = json.loads(
    NB16_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

nb16_manifest = json.loads(
    NB16_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

nb16_release = json.loads(
    NB16_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

nb17_handoff = json.loads(
    NB17_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

nb16_reload_validation = json.loads(
    NB16_RELOAD_VALIDATION_PATH.read_text(
        encoding="utf-8"
    )
)

nb16_acceptance = json.loads(
    NB16_ACCEPTANCE_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    nb16_configuration[
        "configuration_status"
    ] == "FROZEN_BEFORE_TEST_ACCESS"
)

assert (
    nb16_manifest["manifest_status"]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    nb16_release["status"]
    == "FROZEN_FOR_RERANKING"
)

assert (
    nb16_release["final_status"]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    nb17_handoff["status"]
    == "READY_FOR_NB17"
)

assert (
    nb16_reload_validation[
        "reload_validation_status"
    ] == "PASS"
)

assert (
    nb16_acceptance[
        "acceptance_status"
    ] == "PASS_COMPLETE_FROZEN"
)


candidate_path_from_handoff = (
    PROJECT_ROOT
    / nb17_handoff["candidate_path"]
)

assert (
    candidate_path_from_handoff.resolve()
    == NB16_CANDIDATES_PATH.resolve()
)

assert (
    sha256_file(
        candidate_path_from_handoff
    )
    == nb17_handoff[
        "candidate_sha256"
    ]
)

assert (
    nb17_handoff[
        "logical_retrieval_sha256"
    ]
    == EXPECTED_NB16_LOGICAL_RETRIEVAL_SHA256
)

assert (
    nb16_release["candidates"][
        "logical_sha256"
    ]
    == EXPECTED_NB16_LOGICAL_RETRIEVAL_SHA256
)

assert (
    nb17_handoff[
        "hybrid_release"
    ]["sha256"]
    == EXPECTED_NB16_RELEASE_SHA256
)

assert (
    nb16_release["manifest"]["sha256"]
    == EXPECTED_NB16_MANIFEST_SHA256
)

assert (
    nb17_handoff[
        "candidate_record_count"
    ] == 3813
)

assert (
    nb17_handoff[
        "development_query_count"
    ] == 2664
)

assert (
    nb17_handoff["test_query_count"]
    == 1149
)

assert (
    nb17_handoff["candidate_depth"]
    == 100
)

assert (
    nb17_handoff[
        "reranking_scope"
    ][
        "hybrid_configuration_mutation_allowed"
    ] is False
)

assert (
    nb17_handoff[
        "reranking_scope"
    ][
        "dense_index_mutation_allowed"
    ] is False
)

assert (
    nb17_handoff[
        "reranking_scope"
    ][
        "sparse_index_mutation_allowed"
    ] is False
)

assert (
    nb17_handoff[
        "reranking_scope"
    ][
        "corpus_embedding_recomputation_allowed"
    ] is False
)


package_names = (
    "numpy",
    "pandas",
    "scipy",
    "torch",
    "transformers",
    "sentence-transformers",
)

package_versions = {}

for package_name in package_names:
    try:
        package_versions[
            package_name
        ] = importlib.metadata.version(
            package_name
        )
    except (
        importlib.metadata.PackageNotFoundError
    ):
        package_versions[
            package_name
        ] = "NOT_INSTALLED"


nb17_preflight_result = {
    "notebook": NOTEBOOK_NAME,
    "project_root": str(PROJECT_ROOT),
    "python": {
        "version": platform.python_version(),
        "implementation": (
            platform.python_implementation()
        ),
        "platform": platform.platform(),
    },
    "package_versions": package_versions,
    "required_input_count": len(
        required_paths
    ),
    "missing_required_paths": (
        missing_required_paths
    ),
    "validated_hashes": (
        validated_hashes
    ),
    "hybrid_contract": {
        "release_id": (
            nb16_release["release_id"]
        ),
        "candidate_record_count": (
            nb17_handoff[
                "candidate_record_count"
            ]
        ),
        "development_query_count": (
            nb17_handoff[
                "development_query_count"
            ]
        ),
        "test_query_count": (
            nb17_handoff[
                "test_query_count"
            ]
        ),
        "candidate_depth": (
            nb17_handoff[
                "candidate_depth"
            ]
        ),
        "logical_retrieval_sha256": (
            nb17_handoff[
                "logical_retrieval_sha256"
            ]
        ),
        "release_status": (
            nb16_release["status"]
        ),
        "handoff_status": (
            nb17_handoff["status"]
        ),
    },
    "test_access_policy": (
        "LOCKED_UNTIL_RERANKER_CONFIG_"
        "AND_ADR_ARE_FROZEN"
    ),
    "repository_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "preflight_status": "PASS",
}

print(
    json.dumps(
        nb17_preflight_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "notebook": "17_reranking.ipynb",
  "project_root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance",
  "python": {
    "version": "3.11.3",
    "implementation": "CPython",
    "platform": "macOS-12.7.6-x86_64-i386-64bit"
  },
  "package_versions": {
    "numpy": "1.24.3",
    "pandas": "1.5.3",
    "scipy": "1.13.1",
    "torch": "2.2.2",
    "transformers": "4.53.2",
    "sentence-transformers": "5.0.0"
  },
  "required_input_count": 9,
  "missing_required_paths": {},
  "validated_hashes": {
    "nb16_configuration": "e7dc28836b0e6b2f72cf19aa61507af41a76df4980b056f39af015ed91ba26d5",
    "nb16_manifest": "362b9cbfd33e2f8e307ffc6cc7dcffc48d2cc308fe74f9bb0ba896d1e080032c",
    "nb16_release": "8953c5d4d20a0f0cb2cf9fb35d86c421c06cf02b193aa2b19acba46640d0216e",
    "nb17_handoff": "88e8c325f54f89c4aeed94787bc36048f9cc1d6c963e781f7254b6c8c04f7113",
    "canonical_chunks": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    "golden_datase

<div dir="rtl" align="right">

## بارگذاری ورودی‌های Canonical و قفل‌کردن Test

در این مرحله Candidateهای فریز‌شده NB16، Corpus Canonical و Golden Dataset مستقیماً از Artifactهای ذخیره‌شده بارگذاری می‌شوند.

کنترل‌های این مرحله:

- صحت تعداد ۳۸۱۳ Candidate Record؛
- صحت تقسیم ۲۶۶۴ Development و ۱۱۴۹ Test؛
- صحت Candidate Depth حداکثر ۱۰۰؛
- معتبر بودن Rank، Score و Chunk ID؛
- برابری شناسه Queryها با Golden Dataset؛
- بازسازی Logical SHA256 Candidate Artifact؛
- قفل‌ماندن محتوای Queryهای Test.

در این مرحله فقط ساختار و هویت داده‌ها بررسی می‌شود. متن Queryهای Test برای انتخاب مدل یا تنظیمات خوانده نخواهد شد.

</div>

In [3]:
# Code Cell 03 — Load canonical candidates, chunks, and Golden records

RERANKER_CONFIGURATION_FROZEN = False
TEST_SPLIT_LOCKED = True
TEST_QUERIES_ACCESSED = False

EXPECTED_CANDIDATE_RECORD_COUNT = 3813
EXPECTED_DEVELOPMENT_QUERY_COUNT = 2664
EXPECTED_TEST_QUERY_COUNT = 1149
EXPECTED_CHUNK_COUNT = 3488
EXPECTED_CANDIDATE_DEPTH = 100


def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")


def load_jsonl(path):
    records = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        for line_number, line in enumerate(
            file_handle,
            start=1,
        ):
            stripped_line = line.strip()

            assert stripped_line, (
                path,
                line_number,
            )

            records.append(
                json.loads(stripped_line)
            )

    return records


hybrid_candidate_records = load_jsonl(
    NB16_CANDIDATES_PATH
)

chunk_records = load_jsonl(
    CANONICAL_CHUNK_PATH
)

golden_records = load_jsonl(
    GOLDEN_DATASET_PATH
)

assert len(
    hybrid_candidate_records
) == EXPECTED_CANDIDATE_RECORD_COUNT

assert len(
    chunk_records
) == EXPECTED_CHUNK_COUNT

assert len(
    golden_records
) == EXPECTED_CANDIDATE_RECORD_COUNT


chunk_record_by_id = {
    str(record["chunk_id"]): record
    for record in chunk_records
}

assert len(
    chunk_record_by_id
) == EXPECTED_CHUNK_COUNT


golden_record_by_candidate_id = {
    str(record["candidate_id"]): record
    for record in golden_records
}

assert len(
    golden_record_by_candidate_id
) == EXPECTED_CANDIDATE_RECORD_COUNT


candidate_record_by_id = {
    str(record["candidate_id"]): record
    for record
    in hybrid_candidate_records
}

assert len(
    candidate_record_by_id
) == EXPECTED_CANDIDATE_RECORD_COUNT

assert set(
    candidate_record_by_id
) == set(
    golden_record_by_candidate_id
)


development_candidate_ids = sorted(
    candidate_id
    for candidate_id, record
    in candidate_record_by_id.items()
    if record["split"] == "development"
)

test_candidate_ids = sorted(
    candidate_id
    for candidate_id, record
    in candidate_record_by_id.items()
    if record["split"] == "test"
)

assert len(
    development_candidate_ids
) == EXPECTED_DEVELOPMENT_QUERY_COUNT

assert len(
    test_candidate_ids
) == EXPECTED_TEST_QUERY_COUNT

assert not (
    set(development_candidate_ids)
    & set(test_candidate_ids)
)

assert (
    set(development_candidate_ids)
    | set(test_candidate_ids)
) == set(candidate_record_by_id)


candidate_validation_failures = []
candidate_count_distribution = (
    collections.Counter()
)

source_type_counts = (
    collections.Counter(
        str(record["source_type"])
        for record in chunk_records
    )
)

for record in hybrid_candidate_records:
    candidate_id = str(
        record["candidate_id"]
    )

    candidates = record["candidates"]

    if (
        record["candidate_depth"]
        != EXPECTED_CANDIDATE_DEPTH
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "CANDIDATE_DEPTH_MISMATCH",
            )
        )
        continue

    if (
        record["candidate_count"]
        != len(candidates)
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "CANDIDATE_COUNT_MISMATCH",
            )
        )
        continue

    if not (
        1
        <= len(candidates)
        <= EXPECTED_CANDIDATE_DEPTH
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "INVALID_CANDIDATE_COUNT",
            )
        )
        continue

    expected_ranks = list(
        range(
            1,
            len(candidates) + 1,
        )
    )

    actual_ranks = [
        candidate["rank"]
        for candidate in candidates
    ]

    if actual_ranks != expected_ranks:
        candidate_validation_failures.append(
            (
                candidate_id,
                "RANK_MISMATCH",
            )
        )
        continue

    ranked_chunk_ids = [
        str(candidate["chunk_id"])
        for candidate in candidates
    ]

    if len(ranked_chunk_ids) != len(
        set(ranked_chunk_ids)
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "DUPLICATE_CHUNK_ID",
            )
        )
        continue

    if not set(
        ranked_chunk_ids
    ).issubset(
        set(chunk_record_by_id)
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "UNKNOWN_CHUNK_ID",
            )
        )
        continue

    ranked_scores = np.asarray(
        [
            candidate["hybrid_score"]
            for candidate in candidates
        ],
        dtype=np.float64,
    )

    if not np.all(
        np.isfinite(ranked_scores)
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "NON_FINITE_SCORE",
            )
        )
        continue

    if (
        len(ranked_scores) > 1
        and not np.all(
            ranked_scores[:-1]
            >= ranked_scores[1:]
        )
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "SCORE_ORDER_MISMATCH",
            )
        )
        continue

    candidate_count_distribution.update(
        [len(candidates)]
    )


assert not candidate_validation_failures

assert sum(
    candidate_count_distribution.values()
) == EXPECTED_CANDIDATE_RECORD_COUNT


logical_hasher = hashlib.sha256()

for record in hybrid_candidate_records:
    logical_hasher.update(
        canonical_json_bytes(record)
    )
    logical_hasher.update(b"\n")

reloaded_logical_retrieval_sha256 = (
    logical_hasher.hexdigest()
)

assert (
    reloaded_logical_retrieval_sha256
    == EXPECTED_NB16_LOGICAL_RETRIEVAL_SHA256
)

assert (
    reloaded_logical_retrieval_sha256
    == nb17_handoff[
        "logical_retrieval_sha256"
    ]
)

assert (
    sha256_file(
        NB16_CANDIDATES_PATH
    )
    == nb17_handoff[
        "candidate_sha256"
    ]
)


canonical_input_result = {
    "artifact_type": (
        "NB17_CANONICAL_INPUT_CONTRACT"
    ),
    "schema_version": "1.0",
    "hybrid_release_id": (
        nb16_release["release_id"]
    ),
    "candidate_artifact": {
        "path": str(
            NB16_CANDIDATES_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": sha256_file(
            NB16_CANDIDATES_PATH
        ),
        "logical_sha256": (
            reloaded_logical_retrieval_sha256
        ),
        "record_count": len(
            hybrid_candidate_records
        ),
        "candidate_depth": (
            EXPECTED_CANDIDATE_DEPTH
        ),
        "minimum_candidate_count": min(
            candidate_count_distribution
        ),
        "maximum_candidate_count": max(
            candidate_count_distribution
        ),
    },
    "split": {
        "development_query_count": len(
            development_candidate_ids
        ),
        "test_query_count": len(
            test_candidate_ids
        ),
        "cross_split_candidate_overlap": len(
            set(development_candidate_ids)
            & set(test_candidate_ids)
        ),
    },
    "canonical_corpus": {
        "path": str(
            CANONICAL_CHUNK_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            CANONICAL_CHUNK_PATH
        ),
        "record_count": len(
            chunk_records
        ),
        "source_type_counts": dict(
            sorted(
                source_type_counts.items()
            )
        ),
    },
    "golden_dataset": {
        "path": str(
            GOLDEN_DATASET_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            GOLDEN_DATASET_PATH
        ),
        "record_count": len(
            golden_records
        ),
    },
    "configuration": {
        "path": str(
            NB16_CONFIGURATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            NB16_CONFIGURATION_PATH
        ),
        "mutation_allowed": False,
    },
    "test_access": {
        "test_split_locked": (
            TEST_SPLIT_LOCKED
        ),
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
        "unlock_condition": (
            "RERANKER_CONFIGURATION_AND_ADR_FROZEN"
        ),
    },
    "repository_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "canonical_input_status": "PASS",
}

print(
    json.dumps(
        canonical_input_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "artifact_type": "NB17_CANONICAL_INPUT_CONTRACT",
  "schema_version": "1.0",
  "hybrid_release_id": "rag_finance_phase2_hybrid_v1",
  "candidate_artifact": {
    "path": "data/phase2/retrieval/hybrid_retrieval_v1/hybrid_candidates_v1.jsonl",
    "file_sha256": "f54277680f3d20681b1585915f54bb4485ecb1fcccb15c27df0061ff0725d19a",
    "logical_sha256": "f54277680f3d20681b1585915f54bb4485ecb1fcccb15c27df0061ff0725d19a",
    "record_count": 3813,
    "candidate_depth": 100,
    "minimum_candidate_count": 100,
    "maximum_candidate_count": 100
  },
  "split": {
    "development_query_count": 2664,
    "test_query_count": 1149,
    "cross_split_candidate_overlap": 0
  },
  "canonical_corpus": {
    "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    "sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    "record_count": 3488,
    "source_type_counts": {
      "book": 351,
      "transcript": 2041,
      "website": 1096
    }
  }

<div dir="rtl" align="right">

## بازسازی Gold Mapping و Baseline رسمی Development

برای ارزیابی Reranker باید برای هر Query مشخص باشد کدام Chunkها واقعاً مرتبط هستند.

در این مرحله، Evidence Locatorهای Golden Dataset با `member_locators` موجود در Corpus Canonical به‌صورت Exact تطبیق داده می‌شوند. هیچ تطبیق متنی، Fuzzy Matching یا حدس معنایی مجاز نیست.

سپس ترتیب فریز‌شده Hybrid Candidateها روی Development به‌عنوان Baseline رسمی NB17 ارزیابی می‌شود.

به‌دلیل اینکه Artifact تحویلی NB17 شامل Top-100 است، مقدار MRR این Baseline نیز در Scope همان Top-100 محاسبه می‌شود. معیارهای Top-5 و Top-10 باید دقیقاً با NB16 برابر باشند.

Test در این مرحله همچنان قفل است و فقط Candidateهای Development ارزیابی می‌شوند.

</div>

In [4]:
# Code Cell 04 — Rebuild exact Gold mapping and Development baseline

EXPECTED_GOLD_MAPPING_SHA256 = (
    "42216f67665cfc50697789dffe4eb550"
    "839ee22bcbfc0489accc407519dad4d1"
)

EXPECTED_DEVELOPMENT_TOP_K_METRICS = {
    "hit_at_10": 0.7856606606606606,
    "hit_at_5": 0.7117117117117117,
    "ndcg_at_10": 0.5381727779077033,
    "ndcg_at_5": 0.5115595588576384,
    "precision_at_10": (
        0.10762012012012014
    ),
    "precision_at_5": (
        0.18641141141141143
    ),
    "recall_at_10": 0.621242522284189,
    "recall_at_5": 0.5469027062777063,
}

EXPECTED_DEVELOPMENT_TOP_100_MRR = (
    0.6082169141249909
)

assert TEST_SPLIT_LOCKED is True
assert TEST_QUERIES_ACCESSED is False
assert RERANKER_CONFIGURATION_FROZEN is False


def canonical_locator_key(locator):
    assert isinstance(locator, dict)
    assert locator

    return json.dumps(
        locator,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


locator_to_chunk_ids = (
    collections.defaultdict(set)
)

locator_to_source_types = (
    collections.defaultdict(set)
)

chunk_locator_pair_count = 0

for chunk_record in chunk_records:
    chunk_id = str(
        chunk_record["chunk_id"]
    )

    source_type = str(
        chunk_record["source_type"]
    )

    member_locators = chunk_record[
        "member_locators"
    ]

    assert isinstance(
        member_locators,
        list,
    )

    assert member_locators

    for member_locator in member_locators:
        assert isinstance(
            member_locator,
            dict,
        )

        locator = member_locator.get(
            "locator"
        )

        assert isinstance(locator, dict)
        assert locator

        locator_key = (
            canonical_locator_key(locator)
        )

        locator_to_chunk_ids[
            locator_key
        ].add(chunk_id)

        locator_to_source_types[
            locator_key
        ].add(source_type)

        chunk_locator_pair_count += 1


assert locator_to_chunk_ids

chunk_id_set = set(
    chunk_record_by_id
)

gold_mapping_by_candidate_id = {}
gold_mapping_rows = []

unmapped_evidence = []
mapped_evidence_count = 0
evidence_locator_count = 0

evidence_chunk_multiplicity = (
    collections.Counter()
)

query_relevant_chunk_multiplicity = (
    collections.Counter()
)

mapped_query_source_type_counts = (
    collections.Counter()
)


for golden_record in golden_records:
    candidate_id = str(
        golden_record["candidate_id"]
    )

    evidence_locators = (
        golden_record["evidence_locators"]
    )

    record_source_types = {
        str(source_type)
        for source_type
        in golden_record["source_types"]
    }

    relevant_chunk_ids = set()

    evidence_locator_count += len(
        evidence_locators
    )

    for evidence_index, evidence_locator in enumerate(
        evidence_locators
    ):
        assert isinstance(
            evidence_locator,
            dict,
        )

        source_locator = (
            evidence_locator.get(
                "source_locator"
            )
        )

        assert isinstance(
            source_locator,
            dict,
        )

        assert source_locator

        locator_key = (
            canonical_locator_key(
                source_locator
            )
        )

        matching_chunk_ids = set(
            locator_to_chunk_ids.get(
                locator_key,
                set(),
            )
        )

        if not matching_chunk_ids:
            unmapped_evidence.append(
                {
                    "candidate_id": (
                        candidate_id
                    ),
                    "evidence_index": (
                        evidence_index
                    ),
                    "source_locator": (
                        source_locator
                    ),
                }
            )
            continue

        assert matching_chunk_ids.issubset(
            chunk_id_set
        )

        matching_source_types = set(
            locator_to_source_types[
                locator_key
            ]
        )

        assert (
            matching_source_types
            & record_source_types
        )

        relevant_chunk_ids.update(
            matching_chunk_ids
        )

        mapped_evidence_count += 1

        evidence_chunk_multiplicity[
            len(matching_chunk_ids)
        ] += 1

    gold_mapping_by_candidate_id[
        candidate_id
    ] = relevant_chunk_ids

    query_relevant_chunk_multiplicity[
        len(relevant_chunk_ids)
    ] += 1

    for source_type in record_source_types:
        mapped_query_source_type_counts[
            source_type
        ] += 1

    gold_mapping_rows.append(
        {
            "candidate_id": candidate_id,
            "relevant_chunk_ids": sorted(
                relevant_chunk_ids
            ),
        }
    )


unmapped_query_ids = sorted(
    candidate_id
    for candidate_id, relevant_chunk_ids
    in gold_mapping_by_candidate_id.items()
    if not relevant_chunk_ids
)

mapped_relevant_chunk_id_set = {
    chunk_id
    for relevant_chunk_ids
    in gold_mapping_by_candidate_id.values()
    for chunk_id in relevant_chunk_ids
}

assert not unmapped_evidence, (
    unmapped_evidence[:20]
)

assert not unmapped_query_ids, (
    unmapped_query_ids[:20]
)

assert (
    mapped_evidence_count
    == evidence_locator_count
    == 3924
)

assert len(
    gold_mapping_by_candidate_id
) == 3813

assert mapped_relevant_chunk_id_set.issubset(
    chunk_id_set
)

assert len(
    mapped_relevant_chunk_id_set
) == 2483


gold_mapping_digest = hashlib.sha256()

for row in sorted(
    gold_mapping_rows,
    key=lambda item: item[
        "candidate_id"
    ],
):
    mapping_bytes = json.dumps(
        row,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    gold_mapping_digest.update(
        len(mapping_bytes).to_bytes(
            8,
            byteorder="big",
            signed=False,
        )
    )

    gold_mapping_digest.update(
        mapping_bytes
    )

gold_mapping_sha256 = (
    gold_mapping_digest.hexdigest()
)

assert (
    gold_mapping_sha256
    == EXPECTED_GOLD_MAPPING_SHA256
)


def evaluate_ranked_chunk_ids(
    ranked_chunk_ids,
    relevant_chunk_ids,
):
    ranked_chunk_ids = [
        str(chunk_id)
        for chunk_id
        in ranked_chunk_ids
    ]

    relevant_chunk_ids = {
        str(chunk_id)
        for chunk_id
        in relevant_chunk_ids
    }

    assert relevant_chunk_ids

    relevance = np.asarray(
        [
            1.0
            if chunk_id
            in relevant_chunk_ids
            else 0.0
            for chunk_id
            in ranked_chunk_ids
        ],
        dtype=np.float64,
    )

    def hit_at(k):
        return float(
            np.any(relevance[:k] > 0.0)
        )

    def precision_at(k):
        return float(
            relevance[:k].sum()
            / float(k)
        )

    def recall_at(k):
        return float(
            relevance[:k].sum()
            / len(relevant_chunk_ids)
        )

    def ndcg_at(k):
        observed = relevance[:k]

        if observed.size == 0:
            return 0.0

        discounts = 1.0 / np.log2(
            np.arange(
                2,
                observed.size + 2,
                dtype=np.float64,
            )
        )

        dcg = float(
            np.sum(
                observed * discounts
            )
        )

        ideal_count = min(
            len(relevant_chunk_ids),
            k,
        )

        ideal_discounts = (
            1.0
            / np.log2(
                np.arange(
                    2,
                    ideal_count + 2,
                    dtype=np.float64,
                )
            )
        )

        idcg = float(
            ideal_discounts.sum()
        )

        return (
            dcg / idcg
            if idcg > 0.0
            else 0.0
        )

    relevant_positions = np.flatnonzero(
        relevance > 0.0
    )

    reciprocal_rank = (
        1.0
        / float(
            relevant_positions[0] + 1
        )
        if relevant_positions.size
        else 0.0
    )

    return {
        "hit_at_10": hit_at(10),
        "hit_at_5": hit_at(5),
        "ndcg_at_10": ndcg_at(10),
        "ndcg_at_5": ndcg_at(5),
        "precision_at_10": (
            precision_at(10)
        ),
        "precision_at_5": (
            precision_at(5)
        ),
        "recall_at_10": recall_at(10),
        "recall_at_5": recall_at(5),
        "reciprocal_rank": float(
            reciprocal_rank
        ),
    }


METRIC_NAMES = (
    "hit_at_10",
    "hit_at_5",
    "ndcg_at_10",
    "ndcg_at_5",
    "precision_at_10",
    "precision_at_5",
    "recall_at_10",
    "recall_at_5",
    "reciprocal_rank",
)


development_baseline_rows = []

for candidate_id in (
    development_candidate_ids
):
    candidate_record = (
        candidate_record_by_id[
            candidate_id
        ]
    )

    ranked_chunk_ids = [
        str(candidate["chunk_id"])
        for candidate
        in candidate_record["candidates"]
    ]

    metrics = evaluate_ranked_chunk_ids(
        ranked_chunk_ids=ranked_chunk_ids,
        relevant_chunk_ids=(
            gold_mapping_by_candidate_id[
                candidate_id
            ]
        ),
    )

    development_baseline_rows.append(
        {
            "candidate_id": candidate_id,
            **metrics,
        }
    )


development_baseline_metrics = {
    metric_name: float(
        np.mean(
            [
                row[metric_name]
                for row
                in development_baseline_rows
            ]
        )
    )
    for metric_name in METRIC_NAMES
}


development_baseline_deltas = {
    metric_name: float(
        development_baseline_metrics[
            metric_name
        ]
        - expected_value
    )
    for metric_name, expected_value
    in EXPECTED_DEVELOPMENT_TOP_K_METRICS.items()
}

assert all(
    abs(delta) <= 1e-15
    for delta
    in development_baseline_deltas.values()
)

assert np.isclose(
    development_baseline_metrics[
        "reciprocal_rank"
    ],
    EXPECTED_DEVELOPMENT_TOP_100_MRR,
    rtol=0.0,
    atol=1e-15,
)


gold_mapping_and_baseline_result = {
    "mapping": {
        "method": (
            "EXACT_CANONICAL_SOURCE_LOCATOR_IDENTITY"
        ),
        "chunk_count": len(chunk_records),
        "chunk_member_locator_pair_count": (
            chunk_locator_pair_count
        ),
        "unique_chunk_locator_count": len(
            locator_to_chunk_ids
        ),
        "golden_query_count": len(
            golden_records
        ),
        "evidence_locator_count": (
            evidence_locator_count
        ),
        "mapped_evidence_count": (
            mapped_evidence_count
        ),
        "unmapped_evidence_count": len(
            unmapped_evidence
        ),
        "mapped_query_count": len(
            gold_mapping_by_candidate_id
        ),
        "unmapped_query_count": len(
            unmapped_query_ids
        ),
        "unique_relevant_chunk_count": len(
            mapped_relevant_chunk_id_set
        ),
        "evidence_to_chunk_multiplicity": {
            str(key): value
            for key, value
            in sorted(
                evidence_chunk_multiplicity.items()
            )
        },
        "query_relevant_chunk_multiplicity": {
            str(key): value
            for key, value
            in sorted(
                query_relevant_chunk_multiplicity.items()
            )
        },
        "mapped_query_source_type_counts": dict(
            sorted(
                mapped_query_source_type_counts.items()
            )
        ),
        "gold_mapping_sha256": (
            gold_mapping_sha256
        ),
        "status": "PASS",
    },
    "development_baseline": {
        "system": (
            "FROZEN_NB16_HYBRID_TOP_100"
        ),
        "query_count": len(
            development_candidate_ids
        ),
        "candidate_depth": 100,
        "metrics": (
            development_baseline_metrics
        ),
        "top_k_metric_parity": "PASS",
        "mrr_scope": (
            "PERSISTED_TOP_100"
        ),
        "status": "PASS",
    },
    "test_split_locked": (
        TEST_SPLIT_LOCKED
    ),
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "gold_mapping_and_baseline_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        gold_mapping_and_baseline_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "mapping": {
    "method": "EXACT_CANONICAL_SOURCE_LOCATOR_IDENTITY",
    "chunk_count": 3488,
    "chunk_member_locator_pair_count": 25120,
    "unique_chunk_locator_count": 19456,
    "golden_query_count": 3813,
    "evidence_locator_count": 3924,
    "mapped_evidence_count": 3924,
    "unmapped_evidence_count": 0,
    "mapped_query_count": 3813,
    "unmapped_query_count": 0,
    "unique_relevant_chunk_count": 2483,
    "evidence_to_chunk_multiplicity": {
      "1": 1543,
      "2": 1787,
      "3": 371,
      "4": 121,
      "5": 19,
      "6": 30,
      "7": 30,
      "8": 12,
      "10": 11
    },
    "query_relevant_chunk_multiplicity": {
      "1": 1447,
      "2": 1776,
      "3": 374,
      "4": 118,
      "5": 18,
      "6": 26,
      "7": 29,
      "8": 12,
      "9": 1,
      "10": 12
    },
    "mapped_query_source_type_counts": {
      "book": 395,
      "transcript": 2089,
      "website": 1329
    },
    "gold_mapping_sha256": "42216f67665cfc50697789dffe4eb550839ee

<div dir="rtl" align="right">

## ثبت نامزدهای Reranker و بررسی محیط اجرا

در این مرحله، نامزدهای آزمایشی Reranking بدون دسترسی به Test Split ثبت می‌شوند.

نامزدهای اصلی:

- خط مبنای بدون Reranking؛
- Cross-Encoder چندزبانه MiniLM با ۶ لایه؛
- Cross-Encoder چندزبانه MiniLM با ۱۲ لایه.

هدف این مرحله فقط تثبیت رجیستری آزمایش، بررسی منابع سخت‌افزاری و تشخیص مدل‌های موجود در کش محلی است. هیچ مدلی در این سل دانلود یا بارگذاری نمی‌شود.

مدل‌ها در مراحل بعد، پیش از ارزیابی، با Revision دقیق قفل خواهند شد. معیارهای انتخاب نهایی شامل کیفیت، پایداری، زمان پاسخ، مصرف حافظه و هزینه محاسباتی خواهند بود.

</div>

In [5]:
# Code Cell 05 — Register validated reranker candidates

RERANKER_CANDIDATE_REGISTRY_VERSION = (
    "nb17_reranker_candidates_v2"
)

RERANKER_CANDIDATES = (
    {
        "candidate_id": "identity_no_reranker_v1",
        "family": "NO_RERANKER",
        "model_id": None,
        "revision": None,
        "maximum_sequence_length": None,
        "pair_format": None,
        "score_type": "FROZEN_HYBRID_SCORE",
        "execution_policy": "BASELINE_ONLY",
    },
    {
        "candidate_id": (
            "mmarco_mminilmv2_l12_h384_v1"
        ),
        "family": "MULTILINGUAL_CROSS_ENCODER",
        "model_id": (
            "cross-encoder/"
            "mmarco-mMiniLMv2-L12-H384-v1"
        ),
        "revision": "RESOLVE_BEFORE_EXECUTION",
        "maximum_sequence_length": 512,
        "pair_format": "QUESTION_CHUNK_CONTENT",
        "score_type": "RAW_CROSS_ENCODER_LOGIT",
        "execution_policy": (
            "DEVELOPMENT_ONLY_UNTIL_FREEZE"
        ),
    },
)

RERANK_CANDIDATE_DEPTH_GRID = (
    20,
    50,
    100,
)

RERANK_FINAL_K_GRID = (
    5,
    10,
)

assert len({
    candidate["candidate_id"]
    for candidate in RERANKER_CANDIDATES
}) == len(RERANKER_CANDIDATES)

assert (
    RERANKER_CANDIDATES[0]["family"]
    == "NO_RERANKER"
)

assert all(
    depth <= 100
    for depth in RERANK_CANDIDATE_DEPTH_GRID
)

assert all(
    final_k <= depth
    for final_k in RERANK_FINAL_K_GRID
    for depth in RERANK_CANDIDATE_DEPTH_GRID
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN


def huggingface_cache_roots():
    roots = []

    hf_home = os.environ.get("HF_HOME")
    transformers_cache = os.environ.get(
        "TRANSFORMERS_CACHE"
    )
    xdg_cache_home = os.environ.get(
        "XDG_CACHE_HOME"
    )

    if hf_home:
        roots.append(
            Path(hf_home).expanduser() / "hub"
        )

    if transformers_cache:
        roots.append(
            Path(transformers_cache).expanduser()
        )

    if xdg_cache_home:
        roots.append(
            Path(xdg_cache_home).expanduser()
            / "huggingface"
            / "hub"
        )

    roots.append(
        Path.home()
        / ".cache"
        / "huggingface"
        / "hub"
    )

    unique_roots = []
    seen = set()

    for root in roots:
        resolved = root.resolve()

        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(resolved)

    return tuple(unique_roots)


def model_cache_slug(model_id):
    return "models--" + model_id.replace(
        "/",
        "--",
    )


def inspect_local_model_cache(model_id):
    directories = []

    for cache_root in huggingface_cache_roots():
        model_directory = (
            cache_root
            / model_cache_slug(model_id)
        )

        if model_directory.is_dir():
            directories.append(model_directory)

    revisions = set()

    for directory in directories:
        snapshots = directory / "snapshots"

        if snapshots.is_dir():
            for snapshot in snapshots.iterdir():
                if snapshot.is_dir():
                    revisions.add(snapshot.name)

    return {
        "model_id": model_id,
        "cache_present": bool(directories),
        "cache_directories": [
            str(path)
            for path in directories
        ],
        "snapshot_revisions": sorted(
            revisions
        ),
    }


local_model_cache_audit = {
    candidate["candidate_id"]: (
        inspect_local_model_cache(
            candidate["model_id"]
        )
    )
    for candidate in RERANKER_CANDIDATES
    if candidate["model_id"] is not None
}

runtime_device_audit = {
    "cuda_available": bool(
        torch.cuda.is_available()
    ),
    "cuda_device_count": int(
        torch.cuda.device_count()
    ),
    "mps_available": bool(
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ),
    "torch_thread_count": int(
        torch.get_num_threads()
    ),
    "recommended_initial_device": (
        "cuda"
        if torch.cuda.is_available()
        else (
            "mps"
            if (
                hasattr(torch.backends, "mps")
                and torch.backends.mps.is_available()
            )
            else "cpu"
        )
    ),
}

reranker_candidate_registration = {
    "registry_version": (
        RERANKER_CANDIDATE_REGISTRY_VERSION
    ),
    "scope": {
        "selection_split": "DEVELOPMENT_ONLY",
        "development_query_count": len(
            development_candidate_ids
        ),
        "test_split_locked": TEST_SPLIT_LOCKED,
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
    },
    "candidate_count": len(
        RERANKER_CANDIDATES
    ),
    "candidates": list(
        RERANKER_CANDIDATES
    ),
    "candidate_depth_grid": list(
        RERANK_CANDIDATE_DEPTH_GRID
    ),
    "final_k_grid": list(
        RERANK_FINAL_K_GRID
    ),
    "local_cache_audit": (
        local_model_cache_audit
    ),
    "runtime_device_audit": (
        runtime_device_audit
    ),
    "model_resolution_policy": (
        "PIN_EXACT_REVISION_BEFORE_EXECUTION"
    ),
    "download_performed": False,
    "model_loaded": False,
    "test_queries_accessed": False,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "candidate_registration_status": (
        "PASS_REGISTERED_NOT_LOADED"
    ),
}

print(
    json.dumps(
        reranker_candidate_registration,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "registry_version": "nb17_reranker_candidates_v2",
  "scope": {
    "selection_split": "DEVELOPMENT_ONLY",
    "development_query_count": 2664,
    "test_split_locked": true,
    "test_queries_accessed": false
  },
  "candidate_count": 2,
  "candidates": [
    {
      "candidate_id": "identity_no_reranker_v1",
      "family": "NO_RERANKER",
      "model_id": null,
      "revision": null,
      "maximum_sequence_length": null,
      "pair_format": null,
      "score_type": "FROZEN_HYBRID_SCORE",
      "execution_policy": "BASELINE_ONLY"
    },
    {
      "candidate_id": "mmarco_mminilmv2_l12_h384_v1",
      "family": "MULTILINGUAL_CROSS_ENCODER",
      "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
      "revision": "RESOLVE_BEFORE_EXECUTION",
      "maximum_sequence_length": 512,
      "pair_format": "QUESTION_CHUNK_CONTENT",
      "score_type": "RAW_CROSS_ENCODER_LOGIT",
      "execution_policy": "DEVELOPMENT_ONLY_UNTIL_FREEZE"
    }
  ],
  "candidate_depth_grid": [
  

<div dir="rtl" align="right">

## تثبیت Revision نامزدهای Cross-Encoder

در این مرحله شناسه هر مدل مستقیماً در Hugging Face بررسی و Commit SHA دقیق آن ثبت می‌شود.

این کار باعث می‌شود آزمایش‌های بعدی به نسخه متغیر `main` وابسته نباشند و کاملاً تکرارپذیر باقی بمانند.

در این سل:

- وجود مخزن هر مدل بررسی می‌شود؛
- Revision دقیق و تغییرناپذیر ثبت می‌شود؛
- وضعیت عمومی یا محدودبودن مدل بررسی می‌شود؛
- حجم تقریبی فایل‌های مدل گزارش می‌شود؛
- هیچ وزن مدلی دانلود یا بارگذاری نمی‌شود؛
- Test Split همچنان قفل باقی می‌ماند.

</div>

In [6]:
# Code Cell 06 — Resolve and pin exact reranker model revisions

huggingface_hub = importlib.import_module(
    "huggingface_hub"
)

HfApi = huggingface_hub.HfApi

hf_api = HfApi()

pinned_reranker_candidates = []
model_resolution_rows = []

for candidate in RERANKER_CANDIDATES:
    pinned_candidate = dict(candidate)

    if candidate["model_id"] is None:
        pinned_reranker_candidates.append(
            pinned_candidate
        )
        continue

    model_info = hf_api.model_info(
        repo_id=candidate["model_id"],
        revision="main",
        files_metadata=True,
    )

    resolved_revision = str(
        model_info.sha
    )

    assert re.fullmatch(
        r"[0-9a-f]{40}",
        resolved_revision,
    ), resolved_revision

    sibling_rows = []

    for sibling in model_info.siblings or []:
        sibling_size = getattr(
            sibling,
            "size",
            None,
        )

        sibling_rows.append(
            {
                "filename": sibling.rfilename,
                "size_bytes": (
                    int(sibling_size)
                    if sibling_size is not None
                    else None
                ),
            }
        )

    model_weight_extensions = (
        ".bin",
        ".safetensors",
    )

    model_weight_rows = [
        row
        for row in sibling_rows
        if row["filename"].endswith(
            model_weight_extensions
        )
    ]

    known_weight_bytes = sum(
        row["size_bytes"]
        for row in model_weight_rows
        if row["size_bytes"] is not None
    )

    pinned_candidate["revision"] = (
        resolved_revision
    )

    pinned_reranker_candidates.append(
        pinned_candidate
    )

    model_resolution_rows.append(
        {
            "candidate_id": (
                candidate["candidate_id"]
            ),
            "model_id": candidate["model_id"],
            "requested_revision": "main",
            "resolved_revision": (
                resolved_revision
            ),
            "private": bool(
                getattr(
                    model_info,
                    "private",
                    False,
                )
            ),
            "gated": bool(
                getattr(
                    model_info,
                    "gated",
                    False,
                )
            ),
            "pipeline_tag": getattr(
                model_info,
                "pipeline_tag",
                None,
            ),
            "library_name": getattr(
                model_info,
                "library_name",
                None,
            ),
            "repository_file_count": len(
                sibling_rows
            ),
            "model_weight_file_count": len(
                model_weight_rows
            ),
            "known_model_weight_bytes": int(
                known_weight_bytes
            ),
            "known_model_weight_megabytes": (
                float(
                    known_weight_bytes
                    / (1024 ** 2)
                )
            ),
            "resolution_status": "PASS",
        }
    )

PINNED_RERANKER_CANDIDATES = tuple(
    pinned_reranker_candidates
)

assert len(
    PINNED_RERANKER_CANDIDATES
) == len(
    RERANKER_CANDIDATES
)

assert all(
    candidate["revision"] is None
    or re.fullmatch(
        r"[0-9a-f]{40}",
        candidate["revision"],
    )
    for candidate
    in PINNED_RERANKER_CANDIDATES
)

assert len(model_resolution_rows) == 1
assert all(
    not row["private"]
    for row in model_resolution_rows
)
assert all(
    not row["gated"]
    for row in model_resolution_rows
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

reranker_revision_resolution = {
    "registry_version": (
        RERANKER_CANDIDATE_REGISTRY_VERSION
    ),
    "resolution_source": "HUGGING_FACE_HUB",
    "resolution_reference": "main",
    "resolved_model_count": len(
        model_resolution_rows
    ),
    "models": model_resolution_rows,
    "pinned_candidates": list(
        PINNED_RERANKER_CANDIDATES
    ),
    "immutability_contract": (
        "EXACT_40_CHARACTER_COMMIT_SHA"
    ),
    "download_performed": False,
    "model_loaded": False,
    "test_split_locked": (
        TEST_SPLIT_LOCKED
    ),
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "revision_resolution_status": "PASS",
}

print(
    json.dumps(
        reranker_revision_resolution,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "registry_version": "nb17_reranker_candidates_v2",
  "resolution_source": "HUGGING_FACE_HUB",
  "resolution_reference": "main",
  "resolved_model_count": 1,
  "models": [
    {
      "candidate_id": "mmarco_mminilmv2_l12_h384_v1",
      "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
      "requested_revision": "main",
      "resolved_revision": "1427fd652930e4ba29e8149678df786c240d8825",
      "private": false,
      "gated": false,
      "pipeline_tag": "text-ranking",
      "library_name": "sentence-transformers",
      "repository_file_count": 23,
      "model_weight_file_count": 4,
      "known_model_weight_bytes": 1530925239,
      "known_model_weight_megabytes": 1460.0040807724,
      "resolution_status": "PASS"
    }
  ],
  "pinned_candidates": [
    {
      "candidate_id": "identity_no_reranker_v1",
      "family": "NO_RERANKER",
      "model_id": null,
      "revision": null,
      "maximum_sequence_length": null,
      "pair_format": null,
      "score_type": "

<div dir="rtl" align="right">

## بارگذاری کنترل‌شده و Smoke Test مدل Reranker

در این مرحله Cross-Encoder منتخب با Revision دقیق و تغییرناپذیر بارگذاری می‌شود.

برای جلوگیری از اجرای پرهزینه یا خطای ساختاری، ابتدا تنها چهار جفت سؤال–Chunk از Development پردازش می‌شوند.

کنترل‌های این مرحله:

- استفاده انحصاری از Development Split؛
- عدم دسترسی به Test Split؛
- بارگذاری روی CPU؛
- محدودیت طول ورودی برابر با ۵۱۲ توکن؛
- استفاده از متن Canonical Chunk؛
- بررسی تعداد و متناهی‌بودن امتیازها؛
- ثبت زمان بارگذاری و زمان Smoke Test؛
- عدم ایجاد Artifact پروژه.

</div>

In [7]:
# Code Cell 07 — Load the pinned reranker locally and run a smoke test

PINNED_RERANKER_CANDIDATE = next(
    candidate
    for candidate in PINNED_RERANKER_CANDIDATES
    if candidate["family"]
    == "MULTILINGUAL_CROSS_ENCODER"
)

pinned_revision = (
    PINNED_RERANKER_CANDIDATE["revision"]
)

assert isinstance(pinned_revision, str)
assert re.fullmatch(
    r"[0-9a-f]{40}",
    pinned_revision,
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN


def extract_question_text(golden_record):
    evaluation_content = golden_record.get(
        "evaluation_content",
        {},
    )

    question = evaluation_content.get(
        "question"
    )

    assert isinstance(question, str)
    assert question.strip()

    return question.strip()


def extract_candidate_chunk_id(candidate):
    if isinstance(candidate, str):
        chunk_id = candidate

    elif isinstance(candidate, dict):
        chunk_id = candidate.get(
            "chunk_id"
        )

    else:
        raise TypeError(
            "Unsupported candidate representation: "
            f"{type(candidate).__name__}"
        )

    assert isinstance(chunk_id, str)
    assert chunk_id in chunk_record_by_id

    return chunk_id


smoke_candidate_id = (
    development_candidate_ids[0]
)

smoke_golden_record = (
    golden_record_by_candidate_id[
        smoke_candidate_id
    ]
)

smoke_candidate_record = (
    candidate_record_by_id[
        smoke_candidate_id
    ]
)

smoke_question = extract_question_text(
    smoke_golden_record
)

smoke_ranked_candidates = (
    smoke_candidate_record["candidates"][:4]
)

smoke_chunk_ids = [
    extract_candidate_chunk_id(candidate)
    for candidate in smoke_ranked_candidates
]

smoke_pairs = [
    (
        smoke_question,
        chunk_record_by_id[
            chunk_id
        ]["content"],
    )
    for chunk_id in smoke_chunk_ids
]

assert len(smoke_pairs) == 4

assert all(
    isinstance(question, str)
    and question.strip()
    and isinstance(content, str)
    and content.strip()
    for question, content in smoke_pairs
)

reranker_load_started = (
    time.perf_counter()
)

frozen_cross_encoder = CrossEncoder(
    PINNED_RERANKER_CANDIDATE[
        "model_id"
    ],
    revision=pinned_revision,
    max_length=(
        PINNED_RERANKER_CANDIDATE[
            "maximum_sequence_length"
        ]
    ),
    device="cpu",
    trust_remote_code=False,
    token=False,
    local_files_only=True,
)

reranker_load_seconds = float(
    time.perf_counter()
    - reranker_load_started
)

reranker_smoke_started = (
    time.perf_counter()
)

smoke_scores = (
    frozen_cross_encoder.predict(
        smoke_pairs,
        batch_size=4,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
)

reranker_smoke_seconds = float(
    time.perf_counter()
    - reranker_smoke_started
)

smoke_scores = np.asarray(
    smoke_scores,
    dtype=np.float64,
).reshape(-1)

assert smoke_scores.shape == (4,)
assert np.all(
    np.isfinite(smoke_scores)
)

smoke_order = sorted(
    range(len(smoke_chunk_ids)),
    key=lambda index: (
        -float(smoke_scores[index]),
        smoke_chunk_ids[index],
    ),
)

smoke_ranked_results = [
    {
        "rank": rank,
        "chunk_id": (
            smoke_chunk_ids[index]
        ),
        "score": float(
            smoke_scores[index]
        ),
    }
    for rank, index in enumerate(
        smoke_order,
        start=1,
    )
]

reranker_smoke_validation = {
    "candidate_id": (
        PINNED_RERANKER_CANDIDATE[
            "candidate_id"
        ]
    ),
    "model_id": (
        PINNED_RERANKER_CANDIDATE[
            "model_id"
        ]
    ),
    "revision": pinned_revision,
    "runtime": {
        "device": "cpu",
        "loading_policy": (
            "LOCAL_CACHE_ONLY"
        ),
        "network_access_allowed": False,
        "maximum_sequence_length": (
            PINNED_RERANKER_CANDIDATE[
                "maximum_sequence_length"
            ]
        ),
        "load_seconds": (
            reranker_load_seconds
        ),
    },
    "smoke_test": {
        "split": "development",
        "candidate_id": (
            smoke_candidate_id
        ),
        "pair_count": len(
            smoke_pairs
        ),
        "batch_size": 4,
        "elapsed_seconds": (
            reranker_smoke_seconds
        ),
        "pairs_per_second": float(
            len(smoke_pairs)
            / reranker_smoke_seconds
        ),
        "minimum_score": float(
            smoke_scores.min()
        ),
        "maximum_score": float(
            smoke_scores.max()
        ),
        "ranked_results": (
            smoke_ranked_results
        ),
    },
    "model_downloaded_or_cache_used": (
        True
    ),
    "project_artifacts_created": 0,
    "repository_mutated": False,
    "test_split_locked": (
        TEST_SPLIT_LOCKED
    ),
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "paid_api_calls": 0,
    "smoke_test_status": "PASS",
}

print(
    json.dumps(
        reranker_smoke_validation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "candidate_id": "mmarco_mminilmv2_l12_h384_v1",
  "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
  "revision": "1427fd652930e4ba29e8149678df786c240d8825",
  "runtime": {
    "device": "cpu",
    "loading_policy": "LOCAL_CACHE_ONLY",
    "network_access_allowed": false,
    "maximum_sequence_length": 512,
    "load_seconds": 2.1166089920002378
  },
  "smoke_test": {
    "split": "development",
    "candidate_id": "golden_v2_candidate_001b76f6baf33cd0899e5a26",
    "pair_count": 4,
    "batch_size": 4,
    "elapsed_seconds": 0.571969959999933,
    "pairs_per_second": 6.9933742674186385,
    "minimum_score": -7.700293064117432,
    "maximum_score": -1.192928433418274,
    "ranked_results": [
      {
        "rank": 1,
        "chunk_id": "chunk:baseline_phase1_char_safe:retrieval:39b52ea3ac39010ea4750703",
        "score": -1.192928433418274
      },
      {
        "rank": 2,
        "chunk_id": "chunk:baseline_phase1_char_safe:retrieval:51485b67547b986b1a23acd8",
        

<div dir="rtl" align="right">

## کالیبراسیون Batch Size و برآورد زمان اجرا

پیش از اجرای Pilot یا ارزیابی کامل، سرعت Cross-Encoder روی Batch Sizeهای مختلف اندازه‌گیری می‌شود.

این مرحله با ۳۲ جفت سؤال–Chunk از Development انجام می‌شود و اهداف زیر را دارد:

- انتخاب سریع‌ترین Batch Size سازگار با CPU؛
- بررسی یکسان‌بودن امتیازها میان Batch Sizeهای مختلف؛
- برآورد زمان Pilot؛
- برآورد زمان ارزیابی کامل در عمق‌های ۲۰، ۵۰ و ۱۰۰؛
- جلوگیری از اجرای چندساعته بدون برآورد قبلی.

هیچ Query از Test Split در این مرحله استفاده نمی‌شود.

</div>

In [8]:
# Code Cell 08 — Calibrate CPU batch size and estimate runtime

CALIBRATION_PAIR_COUNT = 32
CALIBRATION_BATCH_SIZES = (
    4,
    8,
    16,
)

calibration_candidate_id = (
    development_candidate_ids[0]
)

calibration_question = extract_question_text(
    golden_record_by_candidate_id[
        calibration_candidate_id
    ]
)

calibration_candidate_items = (
    candidate_record_by_id[
        calibration_candidate_id
    ]["candidates"][
        :CALIBRATION_PAIR_COUNT
    ]
)

calibration_chunk_ids = [
    extract_candidate_chunk_id(candidate)
    for candidate in calibration_candidate_items
]

calibration_pairs = [
    (
        calibration_question,
        chunk_record_by_id[chunk_id][
            "content"
        ],
    )
    for chunk_id in calibration_chunk_ids
]

assert len(calibration_pairs) == (
    CALIBRATION_PAIR_COUNT
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

batch_calibration_rows = []
scores_by_batch_size = {}

for batch_size in CALIBRATION_BATCH_SIZES:
    calibration_started = (
        time.perf_counter()
    )

    batch_scores = (
        frozen_cross_encoder.predict(
            calibration_pairs,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
    )

    elapsed_seconds = (
        time.perf_counter()
        - calibration_started
    )

    batch_scores = np.asarray(
        batch_scores,
        dtype=np.float64,
    ).reshape(-1)

    assert batch_scores.shape == (
        CALIBRATION_PAIR_COUNT,
    )
    assert np.all(np.isfinite(batch_scores))

    scores_by_batch_size[
        batch_size
    ] = batch_scores

    batch_calibration_rows.append(
        {
            "batch_size": batch_size,
            "pair_count": (
                CALIBRATION_PAIR_COUNT
            ),
            "elapsed_seconds": float(
                elapsed_seconds
            ),
            "pairs_per_second": float(
                CALIBRATION_PAIR_COUNT
                / elapsed_seconds
            ),
        }
    )

reference_scores = scores_by_batch_size[
    CALIBRATION_BATCH_SIZES[0]
]

maximum_score_delta = 0.0

for batch_size in (
    CALIBRATION_BATCH_SIZES[1:]
):
    score_delta = float(
        np.max(
            np.abs(
                scores_by_batch_size[
                    batch_size
                ]
                - reference_scores
            )
        )
    )

    maximum_score_delta = max(
        maximum_score_delta,
        score_delta,
    )

assert maximum_score_delta <= 1e-5

selected_batch_row = max(
    batch_calibration_rows,
    key=lambda row: (
        row["pairs_per_second"],
        -row["batch_size"],
    ),
)

SELECTED_RERANKER_BATCH_SIZE = int(
    selected_batch_row["batch_size"]
)

CALIBRATED_PAIRS_PER_SECOND = float(
    selected_batch_row[
        "pairs_per_second"
    ]
)


def projected_runtime(
    query_count,
    candidate_depth,
):
    pair_count = (
        int(query_count)
        * int(candidate_depth)
    )

    seconds = (
        pair_count
        / CALIBRATED_PAIRS_PER_SECOND
    )

    return {
        "query_count": int(query_count),
        "candidate_depth": int(
            candidate_depth
        ),
        "pair_count": int(pair_count),
        "projected_minutes": float(
            seconds / 60.0
        ),
        "projected_hours": float(
            seconds / 3600.0
        ),
    }


runtime_projections = {
    "pilot_50_queries_depth_100": (
        projected_runtime(
            query_count=50,
            candidate_depth=100,
        )
    ),
    "pilot_150_queries_depth_100": (
        projected_runtime(
            query_count=150,
            candidate_depth=100,
        )
    ),
    "development_depth_20": (
        projected_runtime(
            query_count=len(
                development_candidate_ids
            ),
            candidate_depth=20,
        )
    ),
    "development_depth_50": (
        projected_runtime(
            query_count=len(
                development_candidate_ids
            ),
            candidate_depth=50,
        )
    ),
    "development_depth_100": (
        projected_runtime(
            query_count=len(
                development_candidate_ids
            ),
            candidate_depth=100,
        )
    ),
    "test_depth_20": (
        projected_runtime(
            query_count=len(
                test_candidate_ids
            ),
            candidate_depth=20,
        )
    ),
}

reranker_runtime_calibration = {
    "model_id": (
        PINNED_RERANKER_CANDIDATE[
            "model_id"
        ]
    ),
    "revision": (
        PINNED_RERANKER_CANDIDATE[
            "revision"
        ]
    ),
    "device": "cpu",
    "calibration_pair_count": (
        CALIBRATION_PAIR_COUNT
    ),
    "batch_results": (
        batch_calibration_rows
    ),
    "maximum_score_delta": float(
        maximum_score_delta
    ),
    "score_parity": "PASS",
    "selected_batch_size": (
        SELECTED_RERANKER_BATCH_SIZE
    ),
    "calibrated_pairs_per_second": (
        CALIBRATED_PAIRS_PER_SECOND
    ),
    "runtime_projections": (
        runtime_projections
    ),
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "runtime_calibration_status": "PASS",
}

print(
    json.dumps(
        reranker_runtime_calibration,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
  "revision": "1427fd652930e4ba29e8149678df786c240d8825",
  "device": "cpu",
  "calibration_pair_count": 32,
  "batch_results": [
    {
      "batch_size": 4,
      "pair_count": 32,
      "elapsed_seconds": 4.626063282999894,
      "pairs_per_second": 6.917328631797001
    },
    {
      "batch_size": 8,
      "pair_count": 32,
      "elapsed_seconds": 5.911085262000142,
      "pairs_per_second": 5.413557507910504
    },
    {
      "batch_size": 16,
      "pair_count": 32,
      "elapsed_seconds": 6.3319376929998725,
      "pairs_per_second": 5.053745243794307
    }
  ],
  "maximum_score_delta": 2.384185791015625e-06,
  "score_parity": "PASS",
  "selected_batch_size": 4,
  "calibrated_pairs_per_second": 6.917328631797001,
  "runtime_projections": {
    "pilot_50_queries_depth_100": {
      "query_count": 50,
      "candidate_depth": 100,
      "pair_count": 5000,
      "projected_minutes": 12.04703979947889,
      "project

<div dir="rtl" align="right">

## ارزیابی نسخه ONNX INT8 روی CPU

در این مرحله نسخه رسمی و کم‌حجم `model_quint8_avx2.onnx` از همان Revision تثبیت‌شده بارگذاری می‌شود.

ارزیابی شامل موارد زیر است:

- استفاده از AVX2 و CPUExecutionProvider؛
- مقایسه امتیازها با خروجی مرجع PyTorch؛
- بررسی همبستگی رتبه‌ها و هم‌پوشانی Top-10؛
- آزمایش Batch Sizeهای مختلف؛
- محاسبه افزایش سرعت واقعی؛
- تعیین صلاحیت ONNX برای اجرای Pilot و Development.

Test Split در این مرحله همچنان قفل است.

</div>

In [9]:
# Code Cell 09 — Benchmark official ONNX INT8 reranker

onnxruntime = importlib.import_module(
    "onnxruntime"
)

hf_hub_download = (
    huggingface_hub.hf_hub_download
)

ONNX_MODEL_FILENAME = (
    "onnx/model_quint8_avx2.onnx"
)

onnx_download_started = time.perf_counter()

onnx_model_path = hf_hub_download(
    repo_id=(
        PINNED_RERANKER_CANDIDATE[
            "model_id"
        ]
    ),
    filename=ONNX_MODEL_FILENAME,
    revision=(
        PINNED_RERANKER_CANDIDATE[
            "revision"
        ]
    ),
    token=False,
)

onnx_download_seconds = (
    time.perf_counter()
    - onnx_download_started
)

onnx_model_path = Path(onnx_model_path)

assert onnx_model_path.is_file()
assert onnx_model_path.stat().st_size > (
    100 * 1024 * 1024
)

session_options = (
    onnxruntime.SessionOptions()
)

session_options.intra_op_num_threads = 4
session_options.inter_op_num_threads = 1
session_options.execution_mode = (
    onnxruntime.ExecutionMode.ORT_SEQUENTIAL
)
session_options.graph_optimization_level = (
    onnxruntime.GraphOptimizationLevel
    .ORT_ENABLE_ALL
)

onnx_session_started = time.perf_counter()

onnx_reranker_session = (
    onnxruntime.InferenceSession(
        str(onnx_model_path),
        sess_options=session_options,
        providers=["CPUExecutionProvider"],
    )
)

onnx_session_load_seconds = (
    time.perf_counter()
    - onnx_session_started
)

onnx_input_names = {
    model_input.name
    for model_input
    in onnx_reranker_session.get_inputs()
}

assert "input_ids" in onnx_input_names
assert "attention_mask" in onnx_input_names

reranker_tokenizer = (
    frozen_cross_encoder.tokenizer
)


def predict_onnx_pairs(
    pairs,
    batch_size,
):
    output_scores = []

    for start_index in range(
        0,
        len(pairs),
        batch_size,
    ):
        batch_pairs = pairs[
            start_index:
            start_index + batch_size
        ]

        questions = [
            pair[0]
            for pair in batch_pairs
        ]

        contents = [
            pair[1]
            for pair in batch_pairs
        ]

        encoded = reranker_tokenizer(
            questions,
            contents,
            padding=True,
            truncation=True,
            max_length=(
                PINNED_RERANKER_CANDIDATE[
                    "maximum_sequence_length"
                ]
            ),
            return_tensors="np",
        )

        model_inputs = {
            name: np.asarray(
                encoded[name],
                dtype=np.int64,
            )
            for name in onnx_input_names
            if name in encoded
        }

        assert set(model_inputs) == (
            onnx_input_names
        )

        batch_outputs = (
            onnx_reranker_session.run(
                None,
                model_inputs,
            )
        )

        batch_scores = np.asarray(
            batch_outputs[0],
            dtype=np.float64,
        ).reshape(-1)

        assert batch_scores.shape == (
            len(batch_pairs),
        )
        assert np.all(
            np.isfinite(batch_scores)
        )

        output_scores.extend(
            float(score)
            for score in batch_scores
        )

    return np.asarray(
        output_scores,
        dtype=np.float64,
    )


# Warm-up outside timed measurements.
_ = predict_onnx_pairs(
    calibration_pairs[:4],
    batch_size=4,
)

ONNX_BATCH_SIZES = (
    4,
    8,
    16,
    32,
)

onnx_batch_rows = []
onnx_scores_by_batch_size = {}

for batch_size in ONNX_BATCH_SIZES:
    benchmark_started = (
        time.perf_counter()
    )

    batch_scores = predict_onnx_pairs(
        calibration_pairs,
        batch_size=batch_size,
    )

    elapsed_seconds = (
        time.perf_counter()
        - benchmark_started
    )

    onnx_scores_by_batch_size[
        batch_size
    ] = batch_scores

    onnx_batch_rows.append(
        {
            "batch_size": batch_size,
            "pair_count": len(
                calibration_pairs
            ),
            "elapsed_seconds": float(
                elapsed_seconds
            ),
            "pairs_per_second": float(
                len(calibration_pairs)
                / elapsed_seconds
            ),
        }
    )

fastest_batch_row = max(
    onnx_batch_rows,
    key=lambda row: (
        row["pairs_per_second"],
        -row["batch_size"],
    ),
)

SELECTED_ONNX_BATCH_SIZE = int(
    fastest_batch_row["batch_size"]
)

assert SELECTED_ONNX_BATCH_SIZE == 4

ONNX_PAIRS_PER_SECOND = float(
    fastest_batch_row[
        "pairs_per_second"
    ]
)

onnx_reference_scores = (
    onnx_scores_by_batch_size[
        SELECTED_ONNX_BATCH_SIZE
    ]
)

onnx_batch_maximum_delta = max(
    float(
        np.max(
            np.abs(
                onnx_scores_by_batch_size[
                    batch_size
                ]
                - onnx_reference_scores
            )
        )
    )
    for batch_size
    in ONNX_BATCH_SIZES
    if batch_size
    != SELECTED_ONNX_BATCH_SIZE
)

pytorch_reference_scores = (
    scores_by_batch_size[4]
)

assert pytorch_reference_scores.shape == (
    onnx_reference_scores.shape
)

score_spearman_result = (
    scipy.stats.spearmanr(
        pytorch_reference_scores,
        onnx_reference_scores,
    )
)

score_spearman = float(
    score_spearman_result.statistic
)

assert np.isfinite(score_spearman)

pytorch_order = sorted(
    range(CALIBRATION_PAIR_COUNT),
    key=lambda index: (
        -float(
            pytorch_reference_scores[
                index
            ]
        ),
        calibration_chunk_ids[index],
    ),
)

onnx_order = sorted(
    range(CALIBRATION_PAIR_COUNT),
    key=lambda index: (
        -float(
            onnx_reference_scores[
                index
            ]
        ),
        calibration_chunk_ids[index],
    ),
)

pytorch_top_10 = {
    calibration_chunk_ids[index]
    for index in pytorch_order[:10]
}

onnx_top_10 = {
    calibration_chunk_ids[index]
    for index in onnx_order[:10]
}

top_10_overlap_count = len(
    pytorch_top_10 & onnx_top_10
)

top_10_overlap_ratio = float(
    top_10_overlap_count / 10.0
)

onnx_speedup_vs_pytorch = float(
    ONNX_PAIRS_PER_SECOND
    / CALIBRATED_PAIRS_PER_SECOND
)

ONNX_QUALITY_GATE_PASSED = bool(
    score_spearman >= 0.98
    and top_10_overlap_ratio >= 0.9
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

onnx_reranker_benchmark = {
    "model": {
        "model_id": (
            PINNED_RERANKER_CANDIDATE[
                "model_id"
            ]
        ),
        "revision": (
            PINNED_RERANKER_CANDIDATE[
                "revision"
            ]
        ),
        "onnx_filename": (
            ONNX_MODEL_FILENAME
        ),
        "onnx_path": str(
            onnx_model_path
        ),
        "onnx_size_megabytes": float(
            onnx_model_path.stat().st_size
            / (1024 ** 2)
        ),
    },
    "runtime": {
        "onnxruntime_version": (
            onnxruntime.__version__
        ),
        "providers": (
            onnx_reranker_session
            .get_providers()
        ),
        "download_or_cache_seconds": float(
            onnx_download_seconds
        ),
        "session_load_seconds": float(
            onnx_session_load_seconds
        ),
        "intra_op_threads": 4,
        "inter_op_threads": 1,
    },
    "benchmark": {
        "pair_count": (
            CALIBRATION_PAIR_COUNT
        ),
        "batch_results": (
            onnx_batch_rows
        ),
        "selected_batch_size": (
            SELECTED_ONNX_BATCH_SIZE
        ),
        "batch_size_policy": (
            "FIXED_FOR_REPRODUCIBILITY"
        ),
        "onnx_pairs_per_second": (
            ONNX_PAIRS_PER_SECOND
        ),
        "pytorch_pairs_per_second": (
            CALIBRATED_PAIRS_PER_SECOND
        ),
        "speedup_vs_pytorch": (
            onnx_speedup_vs_pytorch
        ),
        "cross_batch_maximum_score_delta": (
            onnx_batch_maximum_delta
        ),
        "cross_batch_score_parity": (
            "NOT_ASSUMED"
        ),
    },
    "pytorch_onnx_parity_at_batch_4": {
        "spearman_correlation": (
            score_spearman
        ),
        "top_10_overlap_count": (
            top_10_overlap_count
        ),
        "top_10_overlap_ratio": (
            top_10_overlap_ratio
        ),
        "quality_gate_passed": (
            ONNX_QUALITY_GATE_PASSED
        ),
    },
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "project_files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "onnx_benchmark_status": (
        "PASS"
        if ONNX_QUALITY_GATE_PASSED
        else "FAIL_PARITY_GATE"
    ),
}

print(
    json.dumps(
        onnx_reranker_benchmark,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "model": {
    "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "onnx_filename": "onnx/model_quint8_avx2.onnx",
    "onnx_path": "/Users/macbookpro/.cache/huggingface/hub/models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1/snapshots/1427fd652930e4ba29e8149678df786c240d8825/onnx/model_quint8_avx2.onnx",
    "onnx_size_megabytes": 113.12486267089844
  },
  "runtime": {
    "onnxruntime_version": "1.16.3",
    "providers": [
      "CPUExecutionProvider"
    ],
    "download_or_cache_seconds": 0.0004258760000084294,
    "session_load_seconds": 1.1041213050002625,
    "intra_op_threads": 4,
    "inter_op_threads": 1
  },
  "benchmark": {
    "pair_count": 32,
    "batch_results": [
      {
        "batch_size": 4,
        "pair_count": 32,
        "elapsed_seconds": 3.844981512999766,
        "pairs_per_second": 8.322536764301459
      },
      {
        "batch_size": 8,
        "pair_count": 32,
        "elapsed_s

<div dir="rtl" align="right">

## انتخاب قطعی Pilot متوازن Development

برای ارزیابی اولیه، از هر Source Type تعداد ۵۰ Query انتخاب می‌شود:

- ۵۰ Query از Book؛
- ۵۰ Query از Transcript؛
- ۵۰ Query از Website.

انتخاب با SHA256 و Seed ثابت انجام می‌شود تا کاملاً تکرارپذیر باشد.

در این مرحله فقط شناسه‌های Pilot تعیین می‌شوند و هیچ Pair توسط مدل امتیازدهی نمی‌شود. Test Split همچنان قفل باقی می‌ماند.

</div>

In [10]:
# Code Cell 10 — Select the deterministic Development pilot

PILOT_SELECTION_VERSION = (
    "nb17_development_pilot_v1"
)

PILOT_SELECTION_SEED = (
    "rag_finance_phase2_nb17_reranker_v1"
)

PILOT_PER_SOURCE_TYPE = 50
PILOT_CANDIDATE_DEPTH = 100

EXPECTED_SOURCE_TYPES = (
    "book",
    "transcript",
    "website",
)


def extract_query_source_type(
    candidate_id,
):
    golden_record = (
        golden_record_by_candidate_id[
            candidate_id
        ]
    )

    direct_candidates = [
        golden_record.get("source_type"),
        golden_record.get(
            "evaluation_content",
            {},
        ).get("source_type"),
        golden_record.get(
            "source",
            {},
        ).get("source_type"),
    ]

    for value in direct_candidates:
        if value in EXPECTED_SOURCE_TYPES:
            return value

    relevant_chunk_ids = (
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    relevant_source_types = {
        chunk_record_by_id[
            chunk_id
        ]["source_type"]
        for chunk_id
        in relevant_chunk_ids
    }

    assert len(relevant_source_types) == 1, {
        "candidate_id": candidate_id,
        "source_types": sorted(
            relevant_source_types
        ),
    }

    source_type = next(
        iter(relevant_source_types)
    )

    assert source_type in (
        EXPECTED_SOURCE_TYPES
    )

    return source_type


development_ids_by_source_type = {
    source_type: []
    for source_type
    in EXPECTED_SOURCE_TYPES
}

for candidate_id in development_candidate_ids:
    source_type = extract_query_source_type(
        candidate_id
    )

    development_ids_by_source_type[
        source_type
    ].append(candidate_id)

assert all(
    len(candidate_ids)
    >= PILOT_PER_SOURCE_TYPE
    for candidate_ids
    in development_ids_by_source_type.values()
)


def pilot_selection_hash(
    source_type,
    candidate_id,
):
    payload = (
        PILOT_SELECTION_SEED
        + "\x1f"
        + source_type
        + "\x1f"
        + candidate_id
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


pilot_candidate_ids_by_source_type = {}

for source_type in EXPECTED_SOURCE_TYPES:
    ordered_candidate_ids = sorted(
        development_ids_by_source_type[
            source_type
        ],
        key=lambda candidate_id: (
            pilot_selection_hash(
                source_type,
                candidate_id,
            ),
            candidate_id,
        ),
    )

    pilot_candidate_ids_by_source_type[
        source_type
    ] = ordered_candidate_ids[
        :PILOT_PER_SOURCE_TYPE
    ]

pilot_candidate_ids = tuple(
    candidate_id
    for source_type
    in EXPECTED_SOURCE_TYPES
    for candidate_id
    in pilot_candidate_ids_by_source_type[
        source_type
    ]
)

assert len(pilot_candidate_ids) == 150
assert len(set(pilot_candidate_ids)) == 150
assert set(pilot_candidate_ids).issubset(
    set(development_candidate_ids)
)
assert not (
    set(pilot_candidate_ids)
    & set(test_candidate_ids)
)

pilot_selection_payload = {
    "version": PILOT_SELECTION_VERSION,
    "seed": PILOT_SELECTION_SEED,
    "split": "development",
    "per_source_type": (
        PILOT_PER_SOURCE_TYPE
    ),
    "candidate_depth": (
        PILOT_CANDIDATE_DEPTH
    ),
    "source_types": {
        source_type: (
            pilot_candidate_ids_by_source_type[
                source_type
            ]
        )
        for source_type
        in EXPECTED_SOURCE_TYPES
    },
}

pilot_selection_sha256 = (
    hashlib.sha256(
        canonical_json_bytes(
            pilot_selection_payload
        )
    ).hexdigest()
)

pilot_pair_count = (
    len(pilot_candidate_ids)
    * PILOT_CANDIDATE_DEPTH
)

pilot_projected_seconds = (
    pilot_pair_count
    / ONNX_PAIRS_PER_SECOND
)

reranker_pilot_selection = {
    "selection_version": (
        PILOT_SELECTION_VERSION
    ),
    "selection_method": (
        "DETERMINISTIC_SHA256_BY_SOURCE_TYPE"
    ),
    "selection_seed": (
        PILOT_SELECTION_SEED
    ),
    "split": "development",
    "query_count": len(
        pilot_candidate_ids
    ),
    "per_source_type": {
        source_type: len(
            pilot_candidate_ids_by_source_type[
                source_type
            ]
        )
        for source_type
        in EXPECTED_SOURCE_TYPES
    },
    "candidate_depth": (
        PILOT_CANDIDATE_DEPTH
    ),
    "pair_count": pilot_pair_count,
    "selected_onnx_batch_size": (
        SELECTED_ONNX_BATCH_SIZE
    ),
    "calibrated_pairs_per_second": (
        ONNX_PAIRS_PER_SECOND
    ),
    "projected_minutes": float(
        pilot_projected_seconds / 60.0
    ),
    "pilot_selection_sha256": (
        pilot_selection_sha256
    ),
    "model_execution_performed": False,
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "pilot_selection_status": "PASS",
}

print(
    json.dumps(
        reranker_pilot_selection,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "selection_version": "nb17_development_pilot_v1",
  "selection_method": "DETERMINISTIC_SHA256_BY_SOURCE_TYPE",
  "selection_seed": "rag_finance_phase2_nb17_reranker_v1",
  "split": "development",
  "query_count": 150,
  "per_source_type": {
    "book": 50,
    "transcript": 50,
    "website": 50
  },
  "candidate_depth": 100,
  "pair_count": 15000,
  "selected_onnx_batch_size": 4,
  "calibrated_pairs_per_second": 8.322536764301459,
  "projected_minutes": 30.03891807031067,
  "pilot_selection_sha256": "07ddb004ea10a7fc44c20a25b65e866ab723f0c1b539adf4c3080a200a172c89",
  "model_execution_performed": false,
  "test_split_locked": true,
  "test_queries_accessed": false,
  "files_created": 0,
  "repository_mutated": false,
  "paid_api_calls": 0,
  "pilot_selection_status": "PASS"
}


<div dir="rtl" align="right">

## اجرای Pilot Reranking با قابلیت توقف و ادامه

در این مرحله ۱۵۰ Query منتخب Development تا عمق ۱۰۰ توسط مدل ONNX INT8 امتیازدهی می‌شوند.

برای جلوگیری از ازدست‌رفتن زمان:

- خروجی هر Query بلافاصله در یک Checkpoint مستقل ذخیره می‌شود؛
- نوشتن فایل به‌صورت Atomic انجام می‌شود؛
- اجرای مجدد، Queryهای تکمیل‌شده را تشخیص می‌دهد؛
- در صورت Interrupt، حداکثر پردازش Query جاری از دست می‌رود؛
- هر پنج Query وضعیت پیشرفت و زمان باقی‌مانده گزارش می‌شود؛
- Test Split همچنان قفل و دست‌نخورده باقی می‌ماند.

</div>

In [11]:
# Code Cell 11 — Execute resumable Development reranker pilot

PILOT_CACHE_DIRECTORY = (
    PROJECT_ROOT
    / "artifacts"
    / "phase2"
    / "reranking"
    / "pilot_cache_v1"
)

PILOT_CACHE_METADATA_PATH = (
    PILOT_CACHE_DIRECTORY
    / "pilot_cache_metadata.json"
)

PILOT_QUERY_CACHE_DIRECTORY = (
    PILOT_CACHE_DIRECTORY
    / "queries"
)

PILOT_CACHE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

PILOT_QUERY_CACHE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


def write_json_atomic(
    path,
    value,
):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def pilot_query_cache_path(
    candidate_id,
):
    candidate_hash = hashlib.sha256(
        candidate_id.encode("utf-8")
    ).hexdigest()

    return (
        PILOT_QUERY_CACHE_DIRECTORY
        / f"{candidate_hash}.json"
    )


pilot_cache_contract = {
    "cache_version": (
        "nb17_reranker_pilot_cache_v1"
    ),
    "pilot_selection_sha256": (
        pilot_selection_sha256
    ),
    "model_id": (
        PINNED_RERANKER_CANDIDATE[
            "model_id"
        ]
    ),
    "model_revision": (
        PINNED_RERANKER_CANDIDATE[
            "revision"
        ]
    ),
    "runtime": "ONNX_RUNTIME_CPU",
    "onnx_filename": (
        ONNX_MODEL_FILENAME
    ),
    "onnxruntime_version": (
        onnxruntime.__version__
    ),
    "batch_size": (
        SELECTED_ONNX_BATCH_SIZE
    ),
    "maximum_sequence_length": (
        PINNED_RERANKER_CANDIDATE[
            "maximum_sequence_length"
        ]
    ),
    "candidate_depth": (
        PILOT_CANDIDATE_DEPTH
    ),
    "score_dtype": "float64_json",
    "test_queries_accessed": False,
}

if PILOT_CACHE_METADATA_PATH.exists():
    existing_cache_contract = json.loads(
        PILOT_CACHE_METADATA_PATH.read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_cache_contract
        == pilot_cache_contract
    ), (
        "Existing pilot cache belongs to a "
        "different configuration."
    )
else:
    write_json_atomic(
        PILOT_CACHE_METADATA_PATH,
        pilot_cache_contract,
    )


def load_valid_pilot_cache():
    cached_rows = {}
    invalid_cache_paths = []

    for candidate_id in pilot_candidate_ids:
        cache_path = pilot_query_cache_path(
            candidate_id
        )

        if not cache_path.exists():
            continue

        try:
            row = json.loads(
                cache_path.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            invalid_cache_paths.append(
                str(cache_path)
            )
            continue

        valid = (
            row.get("candidate_id")
            == candidate_id
            and row.get("split")
            == "development"
            and row.get("candidate_depth")
            == PILOT_CANDIDATE_DEPTH
            and len(
                row.get("chunk_ids", [])
            )
            == PILOT_CANDIDATE_DEPTH
            and len(
                row.get("scores", [])
            )
            == PILOT_CANDIDATE_DEPTH
            and all(
                math.isfinite(float(score))
                for score
                in row.get("scores", [])
            )
        )

        if valid:
            cached_rows[candidate_id] = row
        else:
            invalid_cache_paths.append(
                str(cache_path)
            )

    assert not invalid_cache_paths, {
        "invalid_cache_paths": (
            invalid_cache_paths
        )
    }

    return cached_rows


pilot_score_rows_by_candidate_id = (
    load_valid_pilot_cache()
)

initial_cached_query_count = len(
    pilot_score_rows_by_candidate_id
)

pending_pilot_candidate_ids = [
    candidate_id
    for candidate_id
    in pilot_candidate_ids
    if candidate_id
    not in pilot_score_rows_by_candidate_id
]

pilot_execution_started = (
    time.perf_counter()
)

newly_scored_query_count = 0
newly_scored_pair_count = 0

for pending_index, candidate_id in enumerate(
    pending_pilot_candidate_ids,
    start=1,
):
    golden_record = (
        golden_record_by_candidate_id[
            candidate_id
        ]
    )

    candidate_record = (
        candidate_record_by_id[
            candidate_id
        ]
    )

    question = extract_question_text(
        golden_record
    )

    candidate_items = candidate_record[
        "candidates"
    ][:PILOT_CANDIDATE_DEPTH]

    chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate in candidate_items
    ]

    assert len(chunk_ids) == (
        PILOT_CANDIDATE_DEPTH
    )
    assert len(set(chunk_ids)) == (
        PILOT_CANDIDATE_DEPTH
    )

    query_pairs = [
        (
            question,
            chunk_record_by_id[
                chunk_id
            ]["content"],
        )
        for chunk_id in chunk_ids
    ]

    query_started = time.perf_counter()

    query_scores = predict_onnx_pairs(
        query_pairs,
        batch_size=(
            SELECTED_ONNX_BATCH_SIZE
        ),
    )

    query_elapsed_seconds = (
        time.perf_counter()
        - query_started
    )

    assert query_scores.shape == (
        PILOT_CANDIDATE_DEPTH,
    )
    assert np.all(
        np.isfinite(query_scores)
    )

    query_cache_row = {
        "candidate_id": candidate_id,
        "split": "development",
        "source_type": (
            extract_query_source_type(
                candidate_id
            )
        ),
        "candidate_depth": (
            PILOT_CANDIDATE_DEPTH
        ),
        "chunk_ids": chunk_ids,
        "scores": [
            float(score)
            for score in query_scores
        ],
        "elapsed_seconds": float(
            query_elapsed_seconds
        ),
    }

    write_json_atomic(
        pilot_query_cache_path(
            candidate_id
        ),
        query_cache_row,
    )

    pilot_score_rows_by_candidate_id[
        candidate_id
    ] = query_cache_row

    newly_scored_query_count += 1
    newly_scored_pair_count += len(
        query_pairs
    )

    completed_total = len(
        pilot_score_rows_by_candidate_id
    )

    if (
        pending_index == 1
        or pending_index % 5 == 0
        or pending_index
        == len(pending_pilot_candidate_ids)
    ):
        elapsed = (
            time.perf_counter()
            - pilot_execution_started
        )

        average_seconds_per_new_query = (
            elapsed
            / newly_scored_query_count
        )

        remaining_query_count = (
            len(pilot_candidate_ids)
            - completed_total
        )

        projected_remaining_minutes = (
            remaining_query_count
            * average_seconds_per_new_query
            / 60.0
        )

        print(
            json.dumps(
                {
                    "completed_queries": (
                        completed_total
                    ),
                    "total_queries": len(
                        pilot_candidate_ids
                    ),
                    "newly_scored_queries": (
                        newly_scored_query_count
                    ),
                    "remaining_queries": (
                        remaining_query_count
                    ),
                    "projected_remaining_minutes": (
                        projected_remaining_minutes
                    ),
                },
                ensure_ascii=False,
            )
        )

pilot_execution_seconds = (
    time.perf_counter()
    - pilot_execution_started
)

pilot_score_rows_by_candidate_id = (
    load_valid_pilot_cache()
)

assert len(
    pilot_score_rows_by_candidate_id
) == len(pilot_candidate_ids)

assert set(
    pilot_score_rows_by_candidate_id
) == set(pilot_candidate_ids)

pilot_total_scoring_seconds = sum(
    float(row["elapsed_seconds"])
    for row
    in pilot_score_rows_by_candidate_id.values()
)

pilot_total_pair_count = sum(
    len(row["scores"])
    for row
    in pilot_score_rows_by_candidate_id.values()
)

pilot_observed_pairs_per_second = float(
    pilot_total_pair_count
    / pilot_total_scoring_seconds
)

pilot_score_payload = [
    pilot_score_rows_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in pilot_candidate_ids
]

pilot_score_sha256 = hashlib.sha256(
    canonical_json_bytes(
        pilot_score_payload
    )
).hexdigest()

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

reranker_pilot_execution = {
    "pilot_selection_sha256": (
        pilot_selection_sha256
    ),
    "pilot_score_sha256": (
        pilot_score_sha256
    ),
    "split": "development",
    "query_count": len(
        pilot_score_rows_by_candidate_id
    ),
    "candidate_depth": (
        PILOT_CANDIDATE_DEPTH
    ),
    "pair_count": (
        pilot_total_pair_count
    ),
    "initial_cached_query_count": (
        initial_cached_query_count
    ),
    "newly_scored_query_count": (
        newly_scored_query_count
    ),
    "newly_scored_pair_count": (
        newly_scored_pair_count
    ),
    "current_run_seconds": float(
        pilot_execution_seconds
    ),
    "total_recorded_scoring_seconds": float(
        pilot_total_scoring_seconds
    ),
    "observed_pairs_per_second": (
        pilot_observed_pairs_per_second
    ),
    "cache": {
        "directory": str(
            PILOT_CACHE_DIRECTORY.relative_to(
                PROJECT_ROOT
            )
        ),
        "metadata_path": str(
            PILOT_CACHE_METADATA_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "query_checkpoint_count": len(
            pilot_score_rows_by_candidate_id
        ),
        "resume_supported": True,
    },
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "repository_mutated": True,
    "paid_api_calls": 0,
    "pilot_execution_status": "PASS",
}

print(
    json.dumps(
        reranker_pilot_execution,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "pilot_selection_sha256": "07ddb004ea10a7fc44c20a25b65e866ab723f0c1b539adf4c3080a200a172c89",
  "pilot_score_sha256": "8129e4ff2257e8902cd7f5ffc0e1b573050ed24e30f9a868255c650fc9455bc5",
  "split": "development",
  "query_count": 150,
  "candidate_depth": 100,
  "pair_count": 15000,
  "initial_cached_query_count": 150,
  "newly_scored_query_count": 0,
  "newly_scored_pair_count": 0,
  "current_run_seconds": 0.000516522999987501,
  "total_recorded_scoring_seconds": 2251.2226980950036,
  "observed_pairs_per_second": 6.663045825138969,
  "cache": {
    "directory": "artifacts/phase2/reranking/pilot_cache_v1",
    "metadata_path": "artifacts/phase2/reranking/pilot_cache_v1/pilot_cache_metadata.json",
    "query_checkpoint_count": 150,
    "resume_supported": true
  },
  "test_split_locked": true,
  "test_queries_accessed": false,
  "repository_mutated": true,
  "paid_api_calls": 0,
  "pilot_execution_status": "PASS"
}


<div dir="rtl" align="right">

## ارزیابی کیفیت Pilot و انتخاب Candidate Depth

امتیازهای ذخیره‌شده Pilot برای عمق‌های ۲۰، ۵۰ و ۱۰۰ بازاستفاده می‌شوند.

برای هر عمق:

- Candidateهای اولیه محدود می‌شوند؛
- رتبه‌بندی بر اساس امتیاز Cross-Encoder انجام می‌شود؛
- معیارهای بازیابی محاسبه می‌شوند؛
- نتیجه با Hybrid Baseline همان ۱۵۰ Query مقایسه می‌شود؛
- عملکرد جداگانه Book، Transcript و Website گزارش می‌شود.

در این مرحله مدل دوباره اجرا نمی‌شود و Test Split همچنان قفل باقی می‌ماند.

</div>

In [12]:
# Code Cell 12 — Evaluate pilot quality across candidate depths

def aggregate_metric_rows(
    metric_rows,
):
    assert metric_rows

    return {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row in metric_rows
                ]
            )
        )
        for metric_name in METRIC_NAMES
    }


def rank_reranker_candidates(
    chunk_ids,
    scores,
):
    assert len(chunk_ids) == len(scores)

    ranked_indexes = sorted(
        range(len(chunk_ids)),
        key=lambda index: (
            -float(scores[index]),
            chunk_ids[index],
        ),
    )

    return [
        chunk_ids[index]
        for index in ranked_indexes
    ]


pilot_baseline_metric_rows = []
pilot_reranker_rows_by_depth = {
    depth: []
    for depth in RERANK_CANDIDATE_DEPTH_GRID
}

for candidate_id in pilot_candidate_ids:
    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    candidate_record = (
        candidate_record_by_id[
            candidate_id
        ]
    )

    baseline_chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate
        in candidate_record["candidates"]
    ]

    baseline_metrics = (
        evaluate_ranked_chunk_ids(
            ranked_chunk_ids=(
                baseline_chunk_ids
            ),
            relevant_chunk_ids=(
                relevant_chunk_ids
            ),
        )
    )

    source_type = (
        extract_query_source_type(
            candidate_id
        )
    )

    pilot_baseline_metric_rows.append(
        {
            "candidate_id": candidate_id,
            "source_type": source_type,
            **baseline_metrics,
        }
    )

    cached_score_row = (
        pilot_score_rows_by_candidate_id[
            candidate_id
        ]
    )

    assert (
        cached_score_row["chunk_ids"]
        == baseline_chunk_ids[
            :PILOT_CANDIDATE_DEPTH
        ]
    )

    for depth in (
        RERANK_CANDIDATE_DEPTH_GRID
    ):
        depth_chunk_ids = (
            cached_score_row[
                "chunk_ids"
            ][:depth]
        )

        depth_scores = (
            cached_score_row[
                "scores"
            ][:depth]
        )

        reranked_chunk_ids = (
            rank_reranker_candidates(
                chunk_ids=depth_chunk_ids,
                scores=depth_scores,
            )
        )

        reranker_metrics = (
            evaluate_ranked_chunk_ids(
                ranked_chunk_ids=(
                    reranked_chunk_ids
                ),
                relevant_chunk_ids=(
                    relevant_chunk_ids
                ),
            )
        )

        pilot_reranker_rows_by_depth[
            depth
        ].append(
            {
                "candidate_id": (
                    candidate_id
                ),
                "source_type": (
                    source_type
                ),
                "candidate_depth": (
                    depth
                ),
                "ranked_chunk_ids": (
                    reranked_chunk_ids
                ),
                **reranker_metrics,
            }
        )

pilot_baseline_metrics = (
    aggregate_metric_rows(
        pilot_baseline_metric_rows
    )
)

pilot_baseline_metrics_by_source_type = {
    source_type: aggregate_metric_rows(
        [
            row
            for row
            in pilot_baseline_metric_rows
            if row["source_type"]
            == source_type
        ]
    )
    for source_type
    in EXPECTED_SOURCE_TYPES
}

pilot_depth_evaluations = []

for depth in RERANK_CANDIDATE_DEPTH_GRID:
    depth_rows = (
        pilot_reranker_rows_by_depth[
            depth
        ]
    )

    depth_metrics = aggregate_metric_rows(
        depth_rows
    )

    metrics_delta_vs_baseline = {
        metric_name: float(
            depth_metrics[metric_name]
            - pilot_baseline_metrics[
                metric_name
            ]
        )
        for metric_name in METRIC_NAMES
    }

    source_type_metrics = {}
    source_type_deltas = {}

    for source_type in (
        EXPECTED_SOURCE_TYPES
    ):
        source_rows = [
            row
            for row in depth_rows
            if row["source_type"]
            == source_type
        ]

        assert len(source_rows) == (
            PILOT_PER_SOURCE_TYPE
        )

        source_metrics = (
            aggregate_metric_rows(
                source_rows
            )
        )

        source_type_metrics[
            source_type
        ] = source_metrics

        source_type_deltas[
            source_type
        ] = {
            metric_name: float(
                source_metrics[
                    metric_name
                ]
                - (
                    pilot_baseline_metrics_by_source_type[
                        source_type
                    ][metric_name]
                )
            )
            for metric_name in METRIC_NAMES
        }

    pilot_depth_evaluations.append(
        {
            "candidate_depth": depth,
            "query_count": len(
                depth_rows
            ),
            "metrics": depth_metrics,
            "delta_vs_hybrid_baseline": (
                metrics_delta_vs_baseline
            ),
            "source_type_metrics": (
                source_type_metrics
            ),
            "source_type_deltas": (
                source_type_deltas
            ),
            "status": "PASS",
        }
    )

selected_pilot_depth_evaluation = max(
    pilot_depth_evaluations,
    key=lambda row: (
        row["metrics"]["ndcg_at_10"],
        row["metrics"][
            "reciprocal_rank"
        ],
        row["metrics"]["hit_at_10"],
        -row["candidate_depth"],
    ),
)

PILOT_SELECTED_CANDIDATE_DEPTH = int(
    selected_pilot_depth_evaluation[
        "candidate_depth"
    ]
)

pilot_primary_delta = float(
    selected_pilot_depth_evaluation[
        "delta_vs_hybrid_baseline"
    ]["ndcg_at_10"]
)

pilot_minimum_source_delta = min(
    source_delta["ndcg_at_10"]
    for source_delta
    in selected_pilot_depth_evaluation[
        "source_type_deltas"
    ].values()
)

PILOT_RERANKER_PROMOTION_ELIGIBLE = bool(
    pilot_primary_delta > 0.0
    and pilot_minimum_source_delta
    >= -0.02
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

reranker_pilot_evaluation = {
    "scope": {
        "split": "development",
        "query_count": len(
            pilot_candidate_ids
        ),
        "per_source_type": (
            PILOT_PER_SOURCE_TYPE
        ),
        "model_execution_performed": (
            False
        ),
    },
    "hybrid_baseline": {
        "metrics": (
            pilot_baseline_metrics
        ),
        "source_type_metrics": (
            pilot_baseline_metrics_by_source_type
        ),
    },
    "depth_evaluations": (
        pilot_depth_evaluations
    ),
    "selected_pilot_depth": (
        PILOT_SELECTED_CANDIDATE_DEPTH
    ),
    "selected_pilot_metrics": (
        selected_pilot_depth_evaluation[
            "metrics"
        ]
    ),
    "selected_delta_vs_baseline": (
        selected_pilot_depth_evaluation[
            "delta_vs_hybrid_baseline"
        ]
    ),
    "minimum_source_ndcg_at_10_delta": (
        pilot_minimum_source_delta
    ),
    "promotion_policy": {
        "overall_ndcg_at_10_delta": (
            "MUST_BE_POSITIVE"
        ),
        "minimum_source_ndcg_at_10_delta": (
            "MUST_BE_AT_LEAST_-0.02"
        ),
    },
    "promotion_eligible": (
        PILOT_RERANKER_PROMOTION_ELIGIBLE
    ),
    "test_split_locked": (
        TEST_SPLIT_LOCKED
    ),
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "pilot_evaluation_status": "PASS",
}

print(
    json.dumps(
        reranker_pilot_evaluation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 150,
    "per_source_type": 50,
    "model_execution_performed": false
  },
  "hybrid_baseline": {
    "metrics": {
      "hit_at_10": 0.8066666666666666,
      "hit_at_5": 0.72,
      "ndcg_at_10": 0.5559781978235591,
      "ndcg_at_5": 0.525363588618517,
      "precision_at_10": 0.104,
      "precision_at_5": 0.18133333333333332,
      "recall_at_10": 0.6663333333333333,
      "recall_at_5": 0.5774444444444445,
      "reciprocal_rank": 0.6012630862058107
    },
    "source_type_metrics": {
      "book": {
        "hit_at_10": 0.76,
        "hit_at_5": 0.68,
        "ndcg_at_10": 0.5595580580671567,
        "ndcg_at_5": 0.5345620237365661,
        "precision_at_10": 0.07800000000000001,
        "precision_at_5": 0.14,
        "recall_at_10": 0.72,
        "recall_at_5": 0.64,
        "reciprocal_rank": 0.5362963349019076
      },
      "transcript": {
        "hit_at_10": 0.72,
        "hit_at_5": 0.58,
        "ndcg_at_10"

<div dir="rtl" align="right">

## تحلیل Bootstrap و تصمیم کیفیت–هزینه Candidate Depth

در این مرحله با Bootstrap زوجی بررسی می‌شود:

- آیا عمق ۲۰ واقعاً از Hybrid Baseline بهتر است؛
- آیا عمق ۵۰ واقعاً از Hybrid Baseline بهتر است؛
- آیا برتری عمق ۵۰ نسبت به ۲۰ پایدار است؛
- آیا عمق ۱۰۰ مزیتی نسبت به ۵۰ دارد؛
- هزینه زمانی اجرای کامل هر عمق چقدر خواهد بود.

این تحلیل فقط از نتایج ذخیره‌شده Pilot استفاده می‌کند و هیچ اجرای مدل انجام نمی‌دهد.

</div>

In [13]:
# Code Cell 13 — Bootstrap depth stability and quality-cost decision

BOOTSTRAP_SAMPLE_COUNT = 5000
BOOTSTRAP_SEED = 17013

bootstrap_rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

pilot_baseline_ndcg_by_candidate_id = {
    row["candidate_id"]: float(
        row["ndcg_at_10"]
    )
    for row in pilot_baseline_metric_rows
}

pilot_depth_ndcg_by_candidate_id = {
    depth: {
        row["candidate_id"]: float(
            row["ndcg_at_10"]
        )
        for row
        in pilot_reranker_rows_by_depth[
            depth
        ]
    }
    for depth
    in RERANK_CANDIDATE_DEPTH_GRID
}

assert all(
    set(depth_rows)
    == set(pilot_candidate_ids)
    for depth_rows
    in pilot_depth_ndcg_by_candidate_id.values()
)


def paired_bootstrap_delta(
    left_values,
    right_values,
):
    left_array = np.asarray(
        left_values,
        dtype=np.float64,
    )

    right_array = np.asarray(
        right_values,
        dtype=np.float64,
    )

    assert left_array.shape == (
        right_array.shape
    )
    assert left_array.ndim == 1

    observed_differences = (
        left_array - right_array
    )

    bootstrap_indexes = (
        bootstrap_rng.integers(
            low=0,
            high=len(left_array),
            size=(
                BOOTSTRAP_SAMPLE_COUNT,
                len(left_array),
            ),
        )
    )

    bootstrap_deltas = np.mean(
        observed_differences[
            bootstrap_indexes
        ],
        axis=1,
    )

    return {
        "query_count": int(
            len(left_array)
        ),
        "mean_delta": float(
            observed_differences.mean()
        ),
        "confidence_interval_95": {
            "lower": float(
                np.percentile(
                    bootstrap_deltas,
                    2.5,
                )
            ),
            "upper": float(
                np.percentile(
                    bootstrap_deltas,
                    97.5,
                )
            ),
        },
        "probability_delta_positive": float(
            np.mean(
                bootstrap_deltas > 0.0
            )
        ),
        "sample_count": (
            BOOTSTRAP_SAMPLE_COUNT
        ),
        "seed": BOOTSTRAP_SEED,
    }


ordered_pilot_ids = list(
    pilot_candidate_ids
)

baseline_ndcg_values = [
    pilot_baseline_ndcg_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in ordered_pilot_ids
]

depth_ndcg_values = {
    depth: [
        pilot_depth_ndcg_by_candidate_id[
            depth
        ][candidate_id]
        for candidate_id
        in ordered_pilot_ids
    ]
    for depth
    in RERANK_CANDIDATE_DEPTH_GRID
}

bootstrap_comparisons = {
    "depth_20_vs_baseline": (
        paired_bootstrap_delta(
            depth_ndcg_values[20],
            baseline_ndcg_values,
        )
    ),
    "depth_50_vs_baseline": (
        paired_bootstrap_delta(
            depth_ndcg_values[50],
            baseline_ndcg_values,
        )
    ),
    "depth_50_vs_depth_20": (
        paired_bootstrap_delta(
            depth_ndcg_values[50],
            depth_ndcg_values[20],
        )
    ),
    "depth_100_vs_depth_50": (
        paired_bootstrap_delta(
            depth_ndcg_values[100],
            depth_ndcg_values[50],
        )
    ),
}


def runtime_projection_for_depth(
    query_count,
    candidate_depth,
):
    pair_count = (
        int(query_count)
        * int(candidate_depth)
    )

    projected_seconds = (
        pair_count
        / pilot_observed_pairs_per_second
    )

    return {
        "query_count": int(query_count),
        "candidate_depth": int(
            candidate_depth
        ),
        "pair_count": int(pair_count),
        "projected_hours": float(
            projected_seconds / 3600.0
        ),
    }


runtime_by_depth = {
    str(depth): {
        "development": (
            runtime_projection_for_depth(
                len(
                    development_candidate_ids
                ),
                depth,
            )
        ),
        "test": (
            runtime_projection_for_depth(
                len(test_candidate_ids),
                depth,
            )
        ),
        "development_and_test_hours": float(
            (
                len(
                    development_candidate_ids
                )
                + len(test_candidate_ids)
            )
            * depth
            / pilot_observed_pairs_per_second
            / 3600.0
        ),
    }
    for depth in (
        20,
        50,
        100,
    )
}

depth_20_gate = (
    bootstrap_comparisons[
        "depth_20_vs_baseline"
    ]["confidence_interval_95"]["lower"]
    > 0.0
)

depth_50_gate = (
    bootstrap_comparisons[
        "depth_50_vs_baseline"
    ]["confidence_interval_95"]["lower"]
    > 0.0
)

depth_50_incremental_gate = (
    bootstrap_comparisons[
        "depth_50_vs_depth_20"
    ]["mean_delta"]
    >= 0.005
    and bootstrap_comparisons[
        "depth_50_vs_depth_20"
    ]["probability_delta_positive"]
    >= 0.80
)

depth_100_incremental_gate = (
    bootstrap_comparisons[
        "depth_100_vs_depth_50"
    ]["mean_delta"]
    >= 0.005
    and bootstrap_comparisons[
        "depth_100_vs_depth_50"
    ]["probability_delta_positive"]
    >= 0.80
)

if (
    depth_50_gate
    and depth_50_incremental_gate
):
    COST_ADJUSTED_SELECTED_DEPTH = 50
elif depth_20_gate:
    COST_ADJUSTED_SELECTED_DEPTH = 20
else:
    COST_ADJUSTED_SELECTED_DEPTH = None

assert not depth_100_incremental_gate

FULL_DEVELOPMENT_EXECUTION_ELIGIBLE = bool(
    COST_ADJUSTED_SELECTED_DEPTH
    is not None
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

reranker_depth_decision = {
    "scope": {
        "split": "development_pilot",
        "query_count": len(
            pilot_candidate_ids
        ),
        "model_execution_performed": False,
    },
    "bootstrap": {
        "sample_count": (
            BOOTSTRAP_SAMPLE_COUNT
        ),
        "seed": BOOTSTRAP_SEED,
        "comparisons": (
            bootstrap_comparisons
        ),
    },
    "runtime_projection": (
        runtime_by_depth
    ),
    "gates": {
        "depth_20_vs_baseline": (
            depth_20_gate
        ),
        "depth_50_vs_baseline": (
            depth_50_gate
        ),
        "depth_50_incremental_value": (
            depth_50_incremental_gate
        ),
        "depth_100_incremental_value": (
            depth_100_incremental_gate
        ),
    },
    "selected_candidate_depth": (
        COST_ADJUSTED_SELECTED_DEPTH
    ),
    "full_development_execution_eligible": (
        FULL_DEVELOPMENT_EXECUTION_ELIGIBLE
    ),
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "depth_decision_status": "PASS",
}

print(
    json.dumps(
        reranker_depth_decision,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development_pilot",
    "query_count": 150,
    "model_execution_performed": false
  },
  "bootstrap": {
    "sample_count": 5000,
    "seed": 17013,
    "comparisons": {
      "depth_20_vs_baseline": {
        "query_count": 150,
        "mean_delta": 0.05572042730911917,
        "confidence_interval_95": {
          "lower": 0.01870191500852918,
          "upper": 0.09530428610310765
        },
        "probability_delta_positive": 0.9976,
        "sample_count": 5000,
        "seed": 17013
      },
      "depth_50_vs_baseline": {
        "query_count": 150,
        "mean_delta": 0.07298570648290936,
        "confidence_interval_95": {
          "lower": 0.02966705035583852,
          "upper": 0.11775471156276078
        },
        "probability_delta_positive": 0.9992,
        "sample_count": 5000,
        "seed": 17013
      },
      "depth_50_vs_depth_20": {
        "query_count": 150,
        "mean_delta": 0.01726527917379019,
        "confidence_inter

<div dir="rtl" align="right">

## اجرای کامل Reranker روی Development Split

این مرحله نخستین اجرای کامل Reranker در NB17 است.

تنظیمات اجرا:

- Split: Development Only؛
- تعداد Query: ۲۶۶۴؛
- Candidate Depth: ۵۰؛
- Final Metrics Depth: ۵ و ۱۰؛
- Runtime: ONNX Runtime INT8 AVX2؛
- Batch Size ثابت: ۴؛
- Test Split: قفل؛
- Checkpoint: یک فایل Atomic برای هر Query؛
- قابلیت Resume: فعال.

امتیازهای ۱۵۰ Query محاسبه‌شده در Pilot بازاستفاده می‌شوند. در صورت توقف، اجرای مجدد فقط Queryهای فاقد Checkpoint را پردازش خواهد کرد.

</div>

In [14]:
# Code Cell 14 — Execute resumable full Development reranking

FULL_DEVELOPMENT_CANDIDATE_DEPTH = (
    COST_ADJUSTED_SELECTED_DEPTH
)

assert (
    FULL_DEVELOPMENT_CANDIDATE_DEPTH
    == 50
)

DEVELOPMENT_CACHE_VERSION = (
    "nb17_development_reranker_cache_v1"
)

DEVELOPMENT_CACHE_DIRECTORY = (
    PROJECT_ROOT
    / "artifacts"
    / "phase2"
    / "reranking"
    / "development_cache_v1"
)

DEVELOPMENT_CACHE_METADATA_PATH = (
    DEVELOPMENT_CACHE_DIRECTORY
    / "development_cache_metadata.json"
)

DEVELOPMENT_QUERY_CACHE_DIRECTORY = (
    DEVELOPMENT_CACHE_DIRECTORY
    / "queries"
)

DEVELOPMENT_CACHE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

DEVELOPMENT_QUERY_CACHE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

development_candidate_ids_ordered = tuple(
    development_candidate_ids
)

development_query_set_sha256 = (
    hashlib.sha256(
        canonical_json_bytes(
            list(
                development_candidate_ids_ordered
            )
        )
    ).hexdigest()
)

development_cache_contract = {
    "cache_version": (
        DEVELOPMENT_CACHE_VERSION
    ),
    "split": "development",
    "query_count": len(
        development_candidate_ids_ordered
    ),
    "query_set_sha256": (
        development_query_set_sha256
    ),
    "candidate_depth": (
        FULL_DEVELOPMENT_CANDIDATE_DEPTH
    ),
    "model_id": (
        PINNED_RERANKER_CANDIDATE[
            "model_id"
        ]
    ),
    "model_revision": (
        PINNED_RERANKER_CANDIDATE[
            "revision"
        ]
    ),
    "runtime": "ONNX_RUNTIME_CPU",
    "onnx_filename": (
        ONNX_MODEL_FILENAME
    ),
    "onnxruntime_version": (
        onnxruntime.__version__
    ),
    "provider": "CPUExecutionProvider",
    "batch_size": (
        SELECTED_ONNX_BATCH_SIZE
    ),
    "maximum_sequence_length": (
        PINNED_RERANKER_CANDIDATE[
            "maximum_sequence_length"
        ]
    ),
    "score_dtype": "float64_json",
    "test_queries_accessed": False,
}

if DEVELOPMENT_CACHE_METADATA_PATH.exists():
    existing_development_contract = (
        json.loads(
            DEVELOPMENT_CACHE_METADATA_PATH
            .read_text(
                encoding="utf-8"
            )
        )
    )

    assert (
        existing_development_contract
        == development_cache_contract
    ), (
        "Existing Development cache belongs "
        "to a different configuration."
    )
else:
    write_json_atomic(
        DEVELOPMENT_CACHE_METADATA_PATH,
        development_cache_contract,
    )


def development_query_cache_path(
    candidate_id,
):
    candidate_hash = hashlib.sha256(
        candidate_id.encode("utf-8")
    ).hexdigest()

    return (
        DEVELOPMENT_QUERY_CACHE_DIRECTORY
        / f"{candidate_hash}.json"
    )


def validate_development_cache_row(
    row,
    candidate_id,
):
    if not isinstance(row, dict):
        return False

    if (
        row.get("candidate_id")
        != candidate_id
    ):
        return False

    if row.get("split") != "development":
        return False

    if (
        row.get("candidate_depth")
        != FULL_DEVELOPMENT_CANDIDATE_DEPTH
    ):
        return False

    chunk_ids = row.get("chunk_ids")
    scores = row.get("scores")

    if not isinstance(chunk_ids, list):
        return False

    if not isinstance(scores, list):
        return False

    if len(chunk_ids) != (
        FULL_DEVELOPMENT_CANDIDATE_DEPTH
    ):
        return False

    if len(scores) != (
        FULL_DEVELOPMENT_CANDIDATE_DEPTH
    ):
        return False

    if len(set(chunk_ids)) != len(
        chunk_ids
    ):
        return False

    if not all(
        math.isfinite(float(score))
        for score in scores
    ):
        return False

    expected_chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate
        in candidate_record_by_id[
            candidate_id
        ]["candidates"][
            :FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ]
    ]

    if chunk_ids != expected_chunk_ids:
        return False

    return True


def load_valid_development_cache():
    cached_rows = {}
    invalid_paths = []

    for candidate_id in (
        development_candidate_ids_ordered
    ):
        cache_path = (
            development_query_cache_path(
                candidate_id
            )
        )

        if not cache_path.exists():
            continue

        try:
            row = json.loads(
                cache_path.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            invalid_paths.append(
                str(cache_path)
            )
            continue

        if validate_development_cache_row(
            row=row,
            candidate_id=candidate_id,
        ):
            cached_rows[candidate_id] = row
        else:
            invalid_paths.append(
                str(cache_path)
            )

    assert not invalid_paths, {
        "invalid_cache_paths": (
            invalid_paths
        )
    }

    return cached_rows


# Reuse completed Pilot scores for the selected depth.
pilot_reused_query_count = 0

for candidate_id in pilot_candidate_ids:
    destination_path = (
        development_query_cache_path(
            candidate_id
        )
    )

    if destination_path.exists():
        continue

    pilot_row = (
        pilot_score_rows_by_candidate_id[
            candidate_id
        ]
    )

    reused_row = {
        "candidate_id": candidate_id,
        "split": "development",
        "source_type": (
            pilot_row["source_type"]
        ),
        "candidate_depth": (
            FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ),
        "chunk_ids": (
            pilot_row["chunk_ids"][
                :FULL_DEVELOPMENT_CANDIDATE_DEPTH
            ]
        ),
        "scores": (
            pilot_row["scores"][
                :FULL_DEVELOPMENT_CANDIDATE_DEPTH
            ]
        ),
        "elapsed_seconds": None,
        "score_origin": (
            "REUSED_FROM_DEVELOPMENT_PILOT"
        ),
    }

    assert validate_development_cache_row(
        row=reused_row,
        candidate_id=candidate_id,
    )

    write_json_atomic(
        destination_path,
        reused_row,
    )

    pilot_reused_query_count += 1

development_score_rows_by_candidate_id = (
    load_valid_development_cache()
)

initial_cached_query_count = len(
    development_score_rows_by_candidate_id
)

pending_development_candidate_ids = [
    candidate_id
    for candidate_id
    in development_candidate_ids_ordered
    if candidate_id
    not in development_score_rows_by_candidate_id
]

print(
    json.dumps(
        {
            "execution": (
                "FULL_DEVELOPMENT_RERANKING"
            ),
            "total_query_count": len(
                development_candidate_ids_ordered
            ),
            "initial_cached_query_count": (
                initial_cached_query_count
            ),
            "pilot_reused_this_run": (
                pilot_reused_query_count
            ),
            "pending_query_count": len(
                pending_development_candidate_ids
            ),
            "candidate_depth": (
                FULL_DEVELOPMENT_CANDIDATE_DEPTH
            ),
            "resume_supported": True,
            "test_split_locked": (
                TEST_SPLIT_LOCKED
            ),
            "status": "STARTING",
        },
        ensure_ascii=False,
        indent=2,
    )
)

development_execution_started = (
    time.perf_counter()
)

newly_scored_query_count = 0
newly_scored_pair_count = 0
current_run_scoring_seconds = 0.0

for pending_index, candidate_id in enumerate(
    pending_development_candidate_ids,
    start=1,
):
    question = extract_question_text(
        golden_record_by_candidate_id[
            candidate_id
        ]
    )

    candidate_items = (
        candidate_record_by_id[
            candidate_id
        ]["candidates"][
            :FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ]
    )

    chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate in candidate_items
    ]

    assert len(chunk_ids) == (
        FULL_DEVELOPMENT_CANDIDATE_DEPTH
    )

    assert len(set(chunk_ids)) == (
        FULL_DEVELOPMENT_CANDIDATE_DEPTH
    )

    query_pairs = [
        (
            question,
            chunk_record_by_id[
                chunk_id
            ]["content"],
        )
        for chunk_id in chunk_ids
    ]

    query_started = time.perf_counter()

    query_scores = predict_onnx_pairs(
        query_pairs,
        batch_size=(
            SELECTED_ONNX_BATCH_SIZE
        ),
    )

    query_elapsed_seconds = (
        time.perf_counter()
        - query_started
    )

    assert query_scores.shape == (
        FULL_DEVELOPMENT_CANDIDATE_DEPTH,
    )

    assert np.all(
        np.isfinite(query_scores)
    )

    cache_row = {
        "candidate_id": candidate_id,
        "split": "development",
        "source_type": (
            extract_query_source_type(
                candidate_id
            )
        ),
        "candidate_depth": (
            FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ),
        "chunk_ids": chunk_ids,
        "scores": [
            float(score)
            for score in query_scores
        ],
        "elapsed_seconds": float(
            query_elapsed_seconds
        ),
        "score_origin": (
            "FULL_DEVELOPMENT_EXECUTION"
        ),
    }

    assert validate_development_cache_row(
        row=cache_row,
        candidate_id=candidate_id,
    )

    write_json_atomic(
        development_query_cache_path(
            candidate_id
        ),
        cache_row,
    )

    development_score_rows_by_candidate_id[
        candidate_id
    ] = cache_row

    newly_scored_query_count += 1
    newly_scored_pair_count += len(
        query_pairs
    )

    current_run_scoring_seconds += (
        query_elapsed_seconds
    )

    completed_total = len(
        development_score_rows_by_candidate_id
    )

    if (
        pending_index == 1
        or pending_index % 10 == 0
        or pending_index
        == len(
            pending_development_candidate_ids
        )
    ):
        average_seconds_per_query = (
            current_run_scoring_seconds
            / newly_scored_query_count
        )

        remaining_query_count = (
            len(
                development_candidate_ids_ordered
            )
            - completed_total
        )

        projected_remaining_hours = (
            remaining_query_count
            * average_seconds_per_query
            / 3600.0
        )

        print(
            json.dumps(
                {
                    "completed_queries": (
                        completed_total
                    ),
                    "total_queries": len(
                        development_candidate_ids_ordered
                    ),
                    "newly_scored_queries": (
                        newly_scored_query_count
                    ),
                    "remaining_queries": (
                        remaining_query_count
                    ),
                    "projected_remaining_hours": (
                        projected_remaining_hours
                    ),
                },
                ensure_ascii=False,
            )
        )

development_execution_seconds = (
    time.perf_counter()
    - development_execution_started
)

development_score_rows_by_candidate_id = (
    load_valid_development_cache()
)

assert len(
    development_score_rows_by_candidate_id
) == len(
    development_candidate_ids_ordered
)

assert set(
    development_score_rows_by_candidate_id
) == set(
    development_candidate_ids_ordered
)

development_score_payload = [
    development_score_rows_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in development_candidate_ids_ordered
]

development_score_sha256 = (
    hashlib.sha256(
        canonical_json_bytes(
            development_score_payload
        )
    ).hexdigest()
)

recorded_scoring_seconds = sum(
    float(row["elapsed_seconds"])
    for row
    in development_score_rows_by_candidate_id.values()
    if row["elapsed_seconds"] is not None
)

recorded_scoring_pair_count = sum(
    len(row["scores"])
    for row
    in development_score_rows_by_candidate_id.values()
    if row["elapsed_seconds"] is not None
)

observed_development_pairs_per_second = (
    float(
        recorded_scoring_pair_count
        / recorded_scoring_seconds
    )
    if recorded_scoring_seconds > 0.0
    else None
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

full_development_execution = {
    "split": "development",
    "query_count": len(
        development_score_rows_by_candidate_id
    ),
    "candidate_depth": (
        FULL_DEVELOPMENT_CANDIDATE_DEPTH
    ),
    "total_pair_count": (
        len(
            development_candidate_ids_ordered
        )
        * FULL_DEVELOPMENT_CANDIDATE_DEPTH
    ),
    "initial_cached_query_count": (
        initial_cached_query_count
    ),
    "pilot_reused_this_run": (
        pilot_reused_query_count
    ),
    "newly_scored_query_count": (
        newly_scored_query_count
    ),
    "newly_scored_pair_count": (
        newly_scored_pair_count
    ),
    "current_run_seconds": float(
        development_execution_seconds
    ),
    "recorded_model_scoring_seconds": float(
        recorded_scoring_seconds
    ),
    "observed_pairs_per_second": (
        observed_development_pairs_per_second
    ),
    "development_query_set_sha256": (
        development_query_set_sha256
    ),
    "development_score_sha256": (
        development_score_sha256
    ),
    "cache": {
        "directory": str(
            DEVELOPMENT_CACHE_DIRECTORY
            .relative_to(PROJECT_ROOT)
        ),
        "metadata_path": str(
            DEVELOPMENT_CACHE_METADATA_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "query_checkpoint_count": len(
            development_score_rows_by_candidate_id
        ),
        "resume_supported": True,
        "atomic_per_query": True,
    },
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "repository_mutated": True,
    "paid_api_calls": 0,
    "full_development_execution_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        full_development_execution,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "execution": "FULL_DEVELOPMENT_RERANKING",
  "total_query_count": 2664,
  "initial_cached_query_count": 2664,
  "pilot_reused_this_run": 0,
  "pending_query_count": 0,
  "candidate_depth": 50,
  "resume_supported": true,
  "test_split_locked": true,
  "status": "STARTING"
}
{
  "split": "development",
  "query_count": 2664,
  "candidate_depth": 50,
  "total_pair_count": 133200,
  "initial_cached_query_count": 2664,
  "pilot_reused_this_run": 0,
  "newly_scored_query_count": 0,
  "newly_scored_pair_count": 0,
  "current_run_seconds": 0.0005128959996909543,
  "recorded_model_scoring_seconds": 16583.184938295002,
  "observed_pairs_per_second": 7.57996732640454,
  "development_query_set_sha256": "589f440d93f1cfd7ca75b552642bc496a1fbb5e428524697bb86c85ad9a9822d",
  "development_score_sha256": "f81976fffdc04c7b65e89e7b22d0c717d3d8ed3dc80cc1907687368f0b721760",
  "cache": {
    "directory": "artifacts/phase2/reranking/development_cache_v1",
    "metadata_path": "artifacts/phase2/reranking

<div dir="rtl" align="right">

## ارزیابی کامل Reranker روی Development Split

در این مرحله خروجی کامل Development با Hybrid Baseline مقایسه می‌شود.

کنترل‌های اصلی:

- معیارهای کلی Retrieval؛
- معیارهای جداگانه Book، Transcript و Website؛
- Bootstrap زوجی برای `nDCG@10`؛
- شمار Queryهای بهبودیافته، بدون تغییر و پسرفت‌کرده؛
- کنترل Parity خط مبنای NB16؛
- تحلیل Latency اجرای کامل؛
- تعیین صلاحیت Reranker برای Freeze.

این مرحله فقط از Checkpointهای ذخیره‌شده استفاده می‌کند و مدل دوباره اجرا نمی‌شود.

</div>

In [15]:
# Code Cell 15 — Evaluate full Development reranking

FULL_BOOTSTRAP_SAMPLE_COUNT = 5000
FULL_BOOTSTRAP_SEED = 17015

full_development_baseline_rows = []
full_development_reranker_rows = []

for candidate_id in (
    development_candidate_ids_ordered
):
    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    source_type = extract_query_source_type(
        candidate_id
    )

    baseline_chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate
        in candidate_record_by_id[
            candidate_id
        ]["candidates"]
    ]

    baseline_metrics = (
        evaluate_ranked_chunk_ids(
            ranked_chunk_ids=(
                baseline_chunk_ids
            ),
            relevant_chunk_ids=(
                relevant_chunk_ids
            ),
        )
    )

    full_development_baseline_rows.append(
        {
            "candidate_id": candidate_id,
            "source_type": source_type,
            **baseline_metrics,
        }
    )

    score_row = (
        development_score_rows_by_candidate_id[
            candidate_id
        ]
    )

    reranked_chunk_ids = (
        rank_reranker_candidates(
            chunk_ids=(
                score_row["chunk_ids"]
            ),
            scores=score_row["scores"],
        )
    )

    reranker_metrics = (
        evaluate_ranked_chunk_ids(
            ranked_chunk_ids=(
                reranked_chunk_ids
            ),
            relevant_chunk_ids=(
                relevant_chunk_ids
            ),
        )
    )

    full_development_reranker_rows.append(
        {
            "candidate_id": candidate_id,
            "source_type": source_type,
            "ranked_chunk_ids": (
                reranked_chunk_ids
            ),
            **reranker_metrics,
        }
    )

assert len(
    full_development_baseline_rows
) == len(
    development_candidate_ids_ordered
)

assert len(
    full_development_reranker_rows
) == len(
    development_candidate_ids_ordered
)

full_development_baseline_metrics = (
    aggregate_metric_rows(
        full_development_baseline_rows
    )
)

full_development_reranker_metrics = (
    aggregate_metric_rows(
        full_development_reranker_rows
    )
)

TOP_K_METRIC_NAMES = tuple(
    metric_name
    for metric_name in METRIC_NAMES
    if metric_name != "reciprocal_rank"
)

baseline_parity_deltas = {
    metric_name: float(
        full_development_baseline_metrics[
            metric_name
        ]
        - development_baseline_metrics[
            metric_name
        ]
    )
    for metric_name
    in TOP_K_METRIC_NAMES
}

assert all(
    abs(delta) <= 1e-15
    for delta
    in baseline_parity_deltas.values()
)

overall_metric_deltas = {
    metric_name: float(
        full_development_reranker_metrics[
            metric_name
        ]
        - full_development_baseline_metrics[
            metric_name
        ]
    )
    for metric_name in METRIC_NAMES
}

baseline_metrics_by_source_type = {}
reranker_metrics_by_source_type = {}
metric_deltas_by_source_type = {}

for source_type in EXPECTED_SOURCE_TYPES:
    source_baseline_rows = [
        row
        for row
        in full_development_baseline_rows
        if row["source_type"]
        == source_type
    ]

    source_reranker_rows = [
        row
        for row
        in full_development_reranker_rows
        if row["source_type"]
        == source_type
    ]

    assert len(source_baseline_rows) == len(
        source_reranker_rows
    )

    baseline_source_metrics = (
        aggregate_metric_rows(
            source_baseline_rows
        )
    )

    reranker_source_metrics = (
        aggregate_metric_rows(
            source_reranker_rows
        )
    )

    baseline_metrics_by_source_type[
        source_type
    ] = baseline_source_metrics

    reranker_metrics_by_source_type[
        source_type
    ] = reranker_source_metrics

    metric_deltas_by_source_type[
        source_type
    ] = {
        metric_name: float(
            reranker_source_metrics[
                metric_name
            ]
            - baseline_source_metrics[
                metric_name
            ]
        )
        for metric_name in METRIC_NAMES
    }

baseline_row_by_candidate_id = {
    row["candidate_id"]: row
    for row
    in full_development_baseline_rows
}

reranker_row_by_candidate_id = {
    row["candidate_id"]: row
    for row
    in full_development_reranker_rows
}

per_query_ndcg_deltas = np.asarray(
    [
        (
            reranker_row_by_candidate_id[
                candidate_id
            ]["ndcg_at_10"]
            - baseline_row_by_candidate_id[
                candidate_id
            ]["ndcg_at_10"]
        )
        for candidate_id
        in development_candidate_ids_ordered
    ],
    dtype=np.float64,
)

improved_query_count = int(
    np.sum(per_query_ndcg_deltas > 0.0)
)

unchanged_query_count = int(
    np.sum(per_query_ndcg_deltas == 0.0)
)

regressed_query_count = int(
    np.sum(per_query_ndcg_deltas < 0.0)
)

full_bootstrap_rng = (
    np.random.default_rng(
        FULL_BOOTSTRAP_SEED
    )
)

bootstrap_mean_deltas = np.empty(
    FULL_BOOTSTRAP_SAMPLE_COUNT,
    dtype=np.float64,
)

for bootstrap_index in range(
    FULL_BOOTSTRAP_SAMPLE_COUNT
):
    sampled_indexes = (
        full_bootstrap_rng.integers(
            low=0,
            high=len(
                per_query_ndcg_deltas
            ),
            size=len(
                per_query_ndcg_deltas
            ),
        )
    )

    bootstrap_mean_deltas[
        bootstrap_index
    ] = float(
        np.mean(
            per_query_ndcg_deltas[
                sampled_indexes
            ]
        )
    )

full_bootstrap_summary = {
    "sample_count": (
        FULL_BOOTSTRAP_SAMPLE_COUNT
    ),
    "seed": FULL_BOOTSTRAP_SEED,
    "mean_delta": float(
        per_query_ndcg_deltas.mean()
    ),
    "confidence_interval_95": {
        "lower": float(
            np.percentile(
                bootstrap_mean_deltas,
                2.5,
            )
        ),
        "upper": float(
            np.percentile(
                bootstrap_mean_deltas,
                97.5,
            )
        ),
    },
    "probability_delta_positive": float(
        np.mean(
            bootstrap_mean_deltas > 0.0
        )
    ),
}

scored_latency_values = np.asarray(
    [
        float(row["elapsed_seconds"])
        for row
        in development_score_rows_by_candidate_id
        .values()
        if row["elapsed_seconds"]
        is not None
    ],
    dtype=np.float64,
)

assert scored_latency_values.size == 2514

development_latency_summary = {
    "measured_query_count": int(
        scored_latency_values.size
    ),
    "minimum_seconds": float(
        scored_latency_values.min()
    ),
    "median_p50_seconds": float(
        np.median(
            scored_latency_values
        )
    ),
    "p95_seconds": float(
        np.percentile(
            scored_latency_values,
            95,
        )
    ),
    "maximum_seconds": float(
        scored_latency_values.max()
    ),
    "scope": (
        "LOCAL_ONNX_CPU_DEPTH_50"
    ),
}

minimum_source_ndcg_delta = min(
    metric_deltas_by_source_type[
        source_type
    ]["ndcg_at_10"]
    for source_type
    in EXPECTED_SOURCE_TYPES
)

RERANKER_DEVELOPMENT_GATE_PASSED = bool(
    overall_metric_deltas[
        "ndcg_at_10"
    ] > 0.0
    and full_bootstrap_summary[
        "confidence_interval_95"
    ]["lower"] > 0.0
    and minimum_source_ndcg_delta
    >= -0.01
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

full_development_evaluation = {
    "scope": {
        "split": "development",
        "query_count": len(
            development_candidate_ids_ordered
        ),
        "candidate_depth": (
            FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ),
        "test_queries_accessed": False,
    },
    "hybrid_baseline": {
        "metrics": (
            full_development_baseline_metrics
        ),
        "nb16_top_k_metric_parity": (
            "PASS"
        ),
        "maximum_top_k_metric_delta": (
            max(
                abs(delta)
                for delta
                in baseline_parity_deltas.values()
            )
        ),
    },
    "reranker": {
        "metrics": (
            full_development_reranker_metrics
        ),
        "delta_vs_hybrid_baseline": (
            overall_metric_deltas
        ),
    },
    "source_type_analysis": {
        source_type: {
            "query_count": sum(
                1
                for row
                in full_development_reranker_rows
                if row["source_type"]
                == source_type
            ),
            "baseline_metrics": (
                baseline_metrics_by_source_type[
                    source_type
                ]
            ),
            "reranker_metrics": (
                reranker_metrics_by_source_type[
                    source_type
                ]
            ),
            "delta": (
                metric_deltas_by_source_type[
                    source_type
                ]
            ),
        }
        for source_type
        in EXPECTED_SOURCE_TYPES
    },
    "query_outcomes": {
        "improved_ndcg_at_10_count": (
            improved_query_count
        ),
        "unchanged_ndcg_at_10_count": (
            unchanged_query_count
        ),
        "regressed_ndcg_at_10_count": (
            regressed_query_count
        ),
    },
    "bootstrap_ndcg_at_10": (
        full_bootstrap_summary
    ),
    "latency": (
        development_latency_summary
    ),
    "minimum_source_ndcg_at_10_delta": (
        minimum_source_ndcg_delta
    ),
    "promotion_gate_passed": (
        RERANKER_DEVELOPMENT_GATE_PASSED
    ),
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "development_evaluation_status": (
        "PASS"
        if RERANKER_DEVELOPMENT_GATE_PASSED
        else "FAIL_PROMOTION_GATE"
    ),
}

print(
    json.dumps(
        full_development_evaluation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 2664,
    "candidate_depth": 50,
    "test_queries_accessed": false
  },
  "hybrid_baseline": {
    "metrics": {
      "hit_at_10": 0.7856606606606606,
      "hit_at_5": 0.7117117117117117,
      "ndcg_at_10": 0.5381727779077033,
      "ndcg_at_5": 0.5115595588576384,
      "precision_at_10": 0.10762012012012014,
      "precision_at_5": 0.18641141141141143,
      "recall_at_10": 0.621242522284189,
      "recall_at_5": 0.5469027062777063,
      "reciprocal_rank": 0.6082169141249909
    },
    "nb16_top_k_metric_parity": "PASS",
    "maximum_top_k_metric_delta": 0.0
  },
  "reranker": {
    "metrics": {
      "hit_at_10": 0.8498498498498499,
      "hit_at_5": 0.7916666666666666,
      "ndcg_at_10": 0.5988342300716867,
      "ndcg_at_5": 0.5732765152763587,
      "precision_at_10": 0.115015015015015,
      "precision_at_5": 0.20157657657657657,
      "recall_at_10": 0.6676743707993709,
      "recall_at_5": 0.5977872217455551,
 

<div dir="rtl" align="right">

## تحلیل خطا و پسرفت‌های Development

با وجود بهبود کلی و پایدار، برخی Queryها پسرفت داشته‌اند. در این مرحله دلایل آن‌ها طبقه‌بندی می‌شود:

- نبودن سند مرتبط در Candidate Pool عمق ۵۰؛
- حذف نتیجه مرتبط از Top-10 توسط Reranker؛
- کاهش تعداد اسناد مرتبط در Top-10؛
- جابه‌جایی نامطلوب اسناد مرتبط درون Top-10؛
- سایر تغییرات رتبه‌بندی.

تحلیل بر اساس Source Type، طول سؤال و تعداد اسناد مرتبط نیز گزارش می‌شود. این مرحله هیچ اجرای مدل ندارد.

</div>

In [16]:
# Code Cell 16 — Development failure taxonomy and error slices

def first_relevant_rank(
    ranked_chunk_ids,
    relevant_chunk_ids,
):
    for rank, chunk_id in enumerate(
        ranked_chunk_ids,
        start=1,
    ):
        if chunk_id in relevant_chunk_ids:
            return rank

    return None


def relevant_count_at_10(
    ranked_chunk_ids,
    relevant_chunk_ids,
):
    return sum(
        chunk_id in relevant_chunk_ids
        for chunk_id
        in ranked_chunk_ids[:10]
    )


def question_length_bucket(
    question,
):
    character_count = len(question)

    if character_count <= 50:
        return "CHARACTERS_0_TO_50"

    if character_count <= 100:
        return "CHARACTERS_51_TO_100"

    return "CHARACTERS_OVER_100"


def relevance_multiplicity_bucket(
    relevant_count,
):
    if relevant_count == 1:
        return "ONE_RELEVANT_CHUNK"

    if relevant_count == 2:
        return "TWO_RELEVANT_CHUNKS"

    return "THREE_OR_MORE_RELEVANT_CHUNKS"


failure_taxonomy_counter = (
    collections.Counter()
)

outcome_counter_by_source_type = {
    source_type: collections.Counter()
    for source_type
    in EXPECTED_SOURCE_TYPES
}

failure_rows = []

slice_delta_values = {
    "source_type": (
        collections.defaultdict(list)
    ),
    "question_length": (
        collections.defaultdict(list)
    ),
    "relevance_multiplicity": (
        collections.defaultdict(list)
    ),
}

for candidate_id in (
    development_candidate_ids_ordered
):
    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    source_type = extract_query_source_type(
        candidate_id
    )

    question = extract_question_text(
        golden_record_by_candidate_id[
            candidate_id
        ]
    )

    baseline_ranked_chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate
        in candidate_record_by_id[
            candidate_id
        ]["candidates"]
    ]

    reranker_ranked_chunk_ids = (
        reranker_row_by_candidate_id[
            candidate_id
        ]["ranked_chunk_ids"]
    )

    baseline_ndcg = float(
        baseline_row_by_candidate_id[
            candidate_id
        ]["ndcg_at_10"]
    )

    reranker_ndcg = float(
        reranker_row_by_candidate_id[
            candidate_id
        ]["ndcg_at_10"]
    )

    ndcg_delta = (
        reranker_ndcg - baseline_ndcg
    )

    if ndcg_delta > 0.0:
        outcome = "IMPROVED"
    elif ndcg_delta < 0.0:
        outcome = "REGRESSED"
    else:
        outcome = "UNCHANGED"

    outcome_counter_by_source_type[
        source_type
    ][outcome] += 1

    question_bucket = (
        question_length_bucket(
            question
        )
    )

    multiplicity_bucket = (
        relevance_multiplicity_bucket(
            len(relevant_chunk_ids)
        )
    )

    slice_delta_values[
        "source_type"
    ][source_type].append(ndcg_delta)

    slice_delta_values[
        "question_length"
    ][question_bucket].append(ndcg_delta)

    slice_delta_values[
        "relevance_multiplicity"
    ][multiplicity_bucket].append(
        ndcg_delta
    )

    if outcome != "REGRESSED":
        continue

    candidate_pool_chunk_ids = (
        baseline_ranked_chunk_ids[
            :FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ]
    )

    relevant_in_candidate_pool = (
        relevant_chunk_ids
        & set(candidate_pool_chunk_ids)
    )

    baseline_rank = first_relevant_rank(
        baseline_ranked_chunk_ids,
        relevant_chunk_ids,
    )

    reranker_rank = first_relevant_rank(
        reranker_ranked_chunk_ids,
        relevant_chunk_ids,
    )

    baseline_relevant_at_10 = (
        relevant_count_at_10(
            baseline_ranked_chunk_ids,
            relevant_chunk_ids,
        )
    )

    reranker_relevant_at_10 = (
        relevant_count_at_10(
            reranker_ranked_chunk_ids,
            relevant_chunk_ids,
        )
    )

    if not relevant_in_candidate_pool:
        failure_type = (
            "NO_RELEVANT_CHUNK_IN_TOP_50_POOL"
        )
    elif (
        baseline_relevant_at_10 > 0
        and reranker_relevant_at_10 == 0
    ):
        failure_type = (
            "RERANKER_REMOVED_ALL_RELEVANT_FROM_TOP_10"
        )
    elif (
        reranker_relevant_at_10
        < baseline_relevant_at_10
    ):
        failure_type = (
            "RERANKER_REDUCED_TOP_10_RECALL"
        )
    elif (
        baseline_rank is not None
        and reranker_rank is not None
        and reranker_rank > baseline_rank
    ):
        failure_type = (
            "RERANKER_DEMOTED_BEST_RELEVANT_CHUNK"
        )
    else:
        failure_type = (
            "OTHER_TOP_10_ORDERING_REGRESSION"
        )

    failure_taxonomy_counter[
        failure_type
    ] += 1

    failure_rows.append(
        {
            "candidate_id": candidate_id,
            "source_type": source_type,
            "failure_type": failure_type,
            "ndcg_at_10_delta": float(
                ndcg_delta
            ),
            "baseline_first_relevant_rank": (
                baseline_rank
            ),
            "reranker_first_relevant_rank": (
                reranker_rank
            ),
            "baseline_relevant_at_10": (
                baseline_relevant_at_10
            ),
            "reranker_relevant_at_10": (
                reranker_relevant_at_10
            ),
            "relevant_chunk_count": len(
                relevant_chunk_ids
            ),
            "question_character_count": len(
                question
            ),
        }
    )

assert len(failure_rows) == (
    regressed_query_count
)

assert sum(
    failure_taxonomy_counter.values()
) == regressed_query_count


def summarize_delta_slices(
    bucket_values,
):
    return {
        bucket_name: {
            "query_count": len(values),
            "mean_ndcg_at_10_delta": float(
                np.mean(values)
            ),
            "negative_delta_count": int(
                np.sum(
                    np.asarray(
                        values,
                        dtype=np.float64,
                    )
                    < 0.0
                )
            ),
        }
        for bucket_name, values
        in sorted(bucket_values.items())
    }


source_outcomes = {
    source_type: {
        "query_count": int(
            sum(
                outcome_counter_by_source_type[
                    source_type
                ].values()
            )
        ),
        "improved_count": int(
            outcome_counter_by_source_type[
                source_type
            ]["IMPROVED"]
        ),
        "unchanged_count": int(
            outcome_counter_by_source_type[
                source_type
            ]["UNCHANGED"]
        ),
        "regressed_count": int(
            outcome_counter_by_source_type[
                source_type
            ]["REGRESSED"]
        ),
    }
    for source_type
    in EXPECTED_SOURCE_TYPES
}

worst_regression_rows = sorted(
    failure_rows,
    key=lambda row: (
        row["ndcg_at_10_delta"],
        row["candidate_id"],
    ),
)[:25]

RERANKER_FAILURE_ANALYSIS_ACCEPTED = bool(
    RERANKER_DEVELOPMENT_GATE_PASSED
    and minimum_source_ndcg_delta
    > 0.0
)

assert TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED
assert not RERANKER_CONFIGURATION_FROZEN

reranker_failure_analysis = {
    "scope": {
        "split": "development",
        "query_count": len(
            development_candidate_ids_ordered
        ),
        "model_execution_performed": False,
    },
    "overall_outcomes": {
        "improved_count": (
            improved_query_count
        ),
        "unchanged_count": (
            unchanged_query_count
        ),
        "regressed_count": (
            regressed_query_count
        ),
        "regression_ratio": float(
            regressed_query_count
            / len(
                development_candidate_ids_ordered
            )
        ),
    },
    "failure_taxonomy": dict(
        sorted(
            failure_taxonomy_counter.items()
        )
    ),
    "source_type_outcomes": (
        source_outcomes
    ),
    "slice_analysis": {
        slice_name: summarize_delta_slices(
            bucket_values
        )
        for slice_name, bucket_values
        in slice_delta_values.items()
    },
    "worst_regressions": (
        worst_regression_rows
    ),
    "interpretation": (
        "Query-level regressions are retained "
        "as explicit evidence; promotion remains "
        "eligible because aggregate, bootstrap, "
        "and every source-type nDCG@10 delta are "
        "positive."
    ),
    "failure_analysis_accepted": (
        RERANKER_FAILURE_ANALYSIS_ACCEPTED
    ),
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "failure_analysis_status": "PASS",
}

print(
    json.dumps(
        reranker_failure_analysis,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 2664,
    "model_execution_performed": false
  },
  "overall_outcomes": {
    "improved_count": 893,
    "unchanged_count": 1287,
    "regressed_count": 484,
    "regression_ratio": 0.1816816816816817
  },
  "failure_taxonomy": {
    "OTHER_TOP_10_ORDERING_REGRESSION": 104,
    "RERANKER_DEMOTED_BEST_RELEVANT_CHUNK": 222,
    "RERANKER_REDUCED_TOP_10_RECALL": 110,
    "RERANKER_REMOVED_ALL_RELEVANT_FROM_TOP_10": 48
  },
  "source_type_outcomes": {
    "book": {
      "query_count": 277,
      "improved_count": 74,
      "unchanged_count": 163,
      "regressed_count": 40
    },
    "transcript": {
      "query_count": 1459,
      "improved_count": 572,
      "unchanged_count": 625,
      "regressed_count": 262
    },
    "website": {
      "query_count": 928,
      "improved_count": 247,
      "unchanged_count": 499,
      "regressed_count": 182
    }
  },
  "slice_analysis": {
    "source_type": {
      "book": {
        "q

<div dir="rtl" align="right">

## تثبیت تنظیمات Reranker و ثبت ADR

پس از تکمیل انتخاب فقط روی Development، تنظیمات نهایی Reranker تثبیت می‌شوند.

تنظیم منتخب:

- مدل: `mmarco-mMiniLMv2-L12-H384-v1`؛
- Revision دقیق: `1427fd652930e4ba29e8149678df786c240d8825`؛
- Runtime: ONNX Runtime INT8 AVX2؛
- Candidate Depth: ۵۰؛
- Batch Size: ۴؛
- Maximum Sequence Length: ۵۱۲؛
- Final Top-K: ۱۰؛
- Tie-break: `CHUNK_ID_ASCENDING`.

پس از ثبت Configuration و ADR:

- هیچ تغییر دیگری براساس Test مجاز نیست؛
- Test فقط برای ارزیابی مستقل نهایی استفاده می‌شود؛
- نتیجه Test نمی‌تواند مدل، عمق یا تنظیمات Runtime را تغییر دهد.

</div>

In [17]:
# Code Cell 17 — Freeze reranker configuration and ADR

RERANKER_RELEASE_ID = (
    "rag_finance_phase2_reranker_v1"
)

RERANKER_CONFIGURATION_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "phase2"
    / "reranking"
    / "reranker_configuration_v1.json"
)

RERANKER_ADR_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase2"
    / "reranker_adr_v1.json"
)

RERANKER_CONFIGURATION_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

RERANKER_ADR_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def sha256_path_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            block = file_handle.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


onnx_model_sha256 = sha256_path_file(
    onnx_model_path
)

assert len(onnx_model_sha256) == 64
assert RERANKER_DEVELOPMENT_GATE_PASSED
assert RERANKER_FAILURE_ANALYSIS_ACCEPTED
assert FULL_DEVELOPMENT_EXECUTION_ELIGIBLE
assert not TEST_QUERIES_ACCESSED
assert TEST_SPLIT_LOCKED

reranker_configuration_payload = {
    "artifact_type": (
        "NB17_RERANKER_CONFIGURATION"
    ),
    "schema_version": "1.0",
    "release_id": RERANKER_RELEASE_ID,
    "selection_scope": {
        "corpus": "ALL_CANONICAL_CHUNKS",
        "selection_queries": (
            "DEVELOPMENT_ONLY"
        ),
        "development_query_count": len(
            development_candidate_ids_ordered
        ),
        "test_query_count": len(
            test_candidate_ids
        ),
        "test_queries_accessed": False,
        "test_access_policy": (
            "LOCKED_DURING_CONFIGURATION_SELECTION"
        ),
    },
    "canonical_inputs": {
        "nb16_configuration_sha256": (
            "e7dc28836b0e6b2f72cf19aa61507af4"
            "1a76df4980b056f39af015ed91ba26d5"
        ),
        "nb16_manifest_sha256": (
            "362b9cbfd33e2f8e307ffc6cc7dcffc4"
            "8d2cc308fe74f9bb0ba896d1e080032c"
        ),
        "nb16_release_sha256": (
            "8953c5d4d20a0f0cb2cf9fb35d86c42"
            "1c06cf02b193aa2b19acba46640d0216e"
        ),
        "nb17_handoff_sha256": (
            "88e8c325f54f89c4aeed94787bc36048"
            "f9cc1d6c963e781f7254b6c8c04f7113"
        ),
        "hybrid_candidate_sha256": (
            "f54277680f3d20681b1585915f54bb44"
            "85ecb1fcccb15c27df0061ff0725d19a"
        ),
        "canonical_chunk_sha256": (
            "3891c4ecad9332ba753cd605bbccd99da"
            "9d063f64934a30ee39e495349cffb50"
        ),
        "golden_dataset_sha256": (
            "b9ca18e710686ddca48a25c637840b7"
            "16adb306717cafd8ec7c96ce9e182e6f2"
        ),
        "gold_mapping_sha256": (
            "42216f67665cfc50697789dffe4eb550"
            "839ee22bcbfc0489accc407519dad4d1"
        ),
        "split_sha256": (
            "68df1196f683b6ab9a9df08c015f001b"
            "0caca056331eb80d0048aec783736d8f"
        ),
        "development_score_sha256": (
            development_score_sha256
        ),
    },
    "reranker": {
        "candidate_id": (
            PINNED_RERANKER_CANDIDATE[
                "candidate_id"
            ]
        ),
        "family": (
            "MULTILINGUAL_CROSS_ENCODER"
        ),
        "model_id": (
            PINNED_RERANKER_CANDIDATE[
                "model_id"
            ]
        ),
        "revision": (
            PINNED_RERANKER_CANDIDATE[
                "revision"
            ]
        ),
        "pair_format": (
            "QUESTION_CHUNK_CONTENT"
        ),
        "score_type": (
            "RAW_CROSS_ENCODER_LOGIT"
        ),
        "maximum_sequence_length": 512,
    },
    "runtime": {
        "backend": "ONNX_RUNTIME",
        "model_variant": (
            "INT8_AVX2"
        ),
        "onnx_filename": (
            ONNX_MODEL_FILENAME
        ),
        "onnx_model_sha256": (
            onnx_model_sha256
        ),
        "onnxruntime_version": (
            onnxruntime.__version__
        ),
        "execution_provider": (
            "CPUExecutionProvider"
        ),
        "intra_op_threads": 4,
        "inter_op_threads": 1,
        "batch_size": (
            SELECTED_ONNX_BATCH_SIZE
        ),
        "batch_size_policy": (
            "FIXED_FOR_REPRODUCIBILITY"
        ),
    },
    "ranking": {
        "candidate_depth": (
            FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ),
        "final_top_k": 10,
        "deterministic_tie_break": (
            "CHUNK_ID_ASCENDING"
        ),
        "input_candidate_order": (
            "FROZEN_NB16_HYBRID_ORDER"
        ),
    },
    "development_selection": {
        "primary_metric": "ndcg_at_10",
        "baseline_metrics": (
            full_development_baseline_metrics
        ),
        "reranker_metrics": (
            full_development_reranker_metrics
        ),
        "delta_vs_baseline": (
            overall_metric_deltas
        ),
        "bootstrap_ndcg_at_10": (
            full_bootstrap_summary
        ),
        "source_type_deltas": (
            metric_deltas_by_source_type
        ),
        "minimum_source_ndcg_at_10_delta": (
            minimum_source_ndcg_delta
        ),
        "query_outcomes": {
            "improved_count": (
                improved_query_count
            ),
            "unchanged_count": (
                unchanged_query_count
            ),
            "regressed_count": (
                regressed_query_count
            ),
        },
        "failure_taxonomy": dict(
            sorted(
                failure_taxonomy_counter.items()
            )
        ),
        "promotion_gate_passed": (
            RERANKER_DEVELOPMENT_GATE_PASSED
        ),
    },
    "latency": (
        development_latency_summary
    ),
    "freeze_rules": {
        "model_mutation_after_test": False,
        "revision_mutation_after_test": False,
        "runtime_mutation_after_test": False,
        "candidate_depth_mutation_after_test": False,
        "batch_size_mutation_after_test": False,
        "maximum_sequence_length_mutation_after_test": False,
        "test_result_may_change_configuration": False,
        "test_result_role": (
            "INDEPENDENT_FINAL_EVALUATION_ONLY"
        ),
    },
    "paid_api_calls": 0,
    "configuration_status": (
        "FROZEN_BEFORE_TEST_ACCESS"
    ),
}

reranker_configuration_sha256 = (
    hashlib.sha256(
        canonical_json_bytes(
            reranker_configuration_payload
        )
    ).hexdigest()
)

reranker_configuration = {
    **reranker_configuration_payload,
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
}

write_json_atomic(
    RERANKER_CONFIGURATION_PATH,
    reranker_configuration,
)

reranker_configuration_file_sha256 = (
    sha256_path_file(
        RERANKER_CONFIGURATION_PATH
    )
)

reranker_adr_payload = {
    "artifact_type": (
        "NB17_RERANKER_ADR"
    ),
    "schema_version": "1.0",
    "adr_id": "NB17-RERANKER-ADR-001",
    "release_id": RERANKER_RELEASE_ID,
    "status": "ACCEPTED_AND_FROZEN",
    "decision": {
        "selected_system": (
            "MULTILINGUAL_CROSS_ENCODER_ONNX_INT8"
        ),
        "selected_candidate_depth": 50,
        "selected_final_top_k": 10,
        "configuration_sha256": (
            reranker_configuration_sha256
        ),
    },
    "decision_drivers": [
        (
            "Full Development nDCG@10 improved "
            "by more than 0.06."
        ),
        (
            "The paired bootstrap 95% confidence "
            "interval remained entirely positive."
        ),
        (
            "Book, transcript, and website slices "
            "all improved."
        ),
        (
            "Depth 50 improved over depth 20 in "
            "the deterministic pilot with positive "
            "incremental probability."
        ),
        (
            "Depth 100 was statistically worse "
            "than depth 50 in the pilot."
        ),
        (
            "ONNX INT8 passed the PyTorch ranking "
            "parity gate and reduced CPU latency."
        ),
    ],
    "alternatives": {
        "no_reranker": (
            "REJECTED_LOWER_DEVELOPMENT_QUALITY"
        ),
        "candidate_depth_20": (
            "REJECTED_LOWER_PILOT_NDCG_AT_10"
        ),
        "candidate_depth_100": (
            "REJECTED_LOWER_QUALITY_AND_HIGHER_COST"
        ),
        "pytorch_runtime": (
            "REJECTED_HIGHER_CPU_LATENCY"
        ),
    },
    "known_limitations": [
        (
            "18.17 percent of Development queries "
            "showed negative nDCG@10 deltas."
        ),
        (
            "Questions longer than 100 characters "
            "showed smaller average gains."
        ),
        (
            "The runtime evidence is local CPU "
            "benchmarking and not a production SLO."
        ),
    ],
    "test_policy": {
        "test_queries_accessed_before_freeze": (
            False
        ),
        "configuration_changes_after_test": (
            False
        ),
        "test_role": (
            "INDEPENDENT_FINAL_EVALUATION_ONLY"
        ),
    },
}

reranker_adr_sha256 = hashlib.sha256(
    canonical_json_bytes(
        reranker_adr_payload
    )
).hexdigest()

reranker_adr = {
    **reranker_adr_payload,
    "adr_sha256": reranker_adr_sha256,
}

write_json_atomic(
    RERANKER_ADR_PATH,
    reranker_adr,
)

reranker_adr_file_sha256 = (
    sha256_path_file(
        RERANKER_ADR_PATH
    )
)

reloaded_configuration = json.loads(
    RERANKER_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_adr = json.loads(
    RERANKER_ADR_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    reloaded_configuration
    == reranker_configuration
)

assert reloaded_adr == reranker_adr

assert (
    reloaded_configuration[
        "configuration_sha256"
    ]
    == reranker_configuration_sha256
)

assert (
    reloaded_adr["decision"][
        "configuration_sha256"
    ]
    == reranker_configuration_sha256
)

RERANKER_CONFIGURATION_FROZEN = True
TEST_SPLIT_LOCKED = False

assert RERANKER_CONFIGURATION_FROZEN
assert not TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED

reranker_configuration_freeze = {
    "release_id": RERANKER_RELEASE_ID,
    "configuration": {
        "path": str(
            RERANKER_CONFIGURATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "file_sha256": (
            reranker_configuration_file_sha256
        ),
        "payload_sha256": (
            reranker_configuration_sha256
        ),
        "status": (
            "FROZEN_BEFORE_TEST_ACCESS"
        ),
    },
    "adr": {
        "path": str(
            RERANKER_ADR_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": (
            reranker_adr_file_sha256
        ),
        "payload_sha256": (
            reranker_adr_sha256
        ),
        "status": "ACCEPTED_AND_FROZEN",
    },
    "selected_configuration": {
        "model_id": (
            PINNED_RERANKER_CANDIDATE[
                "model_id"
            ]
        ),
        "revision": (
            PINNED_RERANKER_CANDIDATE[
                "revision"
            ]
        ),
        "runtime": "ONNX_INT8_AVX2",
        "candidate_depth": 50,
        "batch_size": 4,
        "final_top_k": 10,
    },
    "development_metrics": (
        full_development_reranker_metrics
    ),
    "development_delta_vs_baseline": (
        overall_metric_deltas
    ),
    "test_queries_accessed": False,
    "configuration_frozen": True,
    "test_split_locked": False,
    "files_created": 2,
    "repository_mutated": True,
    "paid_api_calls": 0,
    "configuration_freeze_status": "PASS",
}

print(
    json.dumps(
        reranker_configuration_freeze,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_reranker_v1",
  "configuration": {
    "path": "artifacts/phase2/reranking/reranker_configuration_v1.json",
    "file_sha256": "dd2290188c4e99be1dada40b167b1041c69f7f077ebe6dea1fddcb58b4834a86",
    "payload_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
    "status": "FROZEN_BEFORE_TEST_ACCESS"
  },
  "adr": {
    "path": "reports/phase2/reranker_adr_v1.json",
    "file_sha256": "a86099fefcbfe15d322e81609a33a5693e7df380b99047d456220e21eef14db2",
    "payload_sha256": "52d6089ea97047fceaa836e2b5234f40372f15fe80a481315f39d9725dffb581",
    "status": "ACCEPTED_AND_FROZEN"
  },
  "selected_configuration": {
    "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "runtime": "ONNX_INT8_AVX2",
    "candidate_depth": 50,
    "batch_size": 4,
    "final_top_k": 10
  },
  "development_metrics": {
    "hit_at_10": 0.8498498498498499,
    "hit_at_5": 0.79166666

<div dir="rtl" align="right">

## ارزیابی مستقل Reranker روی Test Split

Configuration و ADR پیش از دسترسی به Test تثبیت شده‌اند. اکنون Test Split فقط برای ارزیابی مستقل نهایی باز می‌شود.

قواعد این مرحله:

- هیچ مدل یا پارامتری تغییر نمی‌کند؛
- Candidate Depth برابر ۵۰ باقی می‌ماند؛
- Batch Size برابر ۴ باقی می‌ماند؛
- هر Query در Checkpoint مستقل و Atomic ذخیره می‌شود؛
- اجرای قطع‌شده قابل ادامه است؛
- نتیجه Test صرفاً گزارش می‌شود و نمی‌تواند Configuration را تغییر دهد.

</div>

In [18]:
# Code Cell 18 — Execute resumable independent Test reranking

assert RERANKER_CONFIGURATION_FROZEN
assert not TEST_SPLIT_LOCKED
assert not TEST_QUERIES_ACCESSED

FROZEN_TEST_CANDIDATE_DEPTH = 50

assert (
    FROZEN_TEST_CANDIDATE_DEPTH
    == reranker_configuration[
        "ranking"
    ]["candidate_depth"]
)

TEST_CACHE_VERSION = (
    "nb17_test_reranker_cache_v1"
)

TEST_CACHE_DIRECTORY = (
    PROJECT_ROOT
    / "artifacts"
    / "phase2"
    / "reranking"
    / "test_cache_v1"
)

TEST_CACHE_METADATA_PATH = (
    TEST_CACHE_DIRECTORY
    / "test_cache_metadata.json"
)

TEST_QUERY_CACHE_DIRECTORY = (
    TEST_CACHE_DIRECTORY
    / "queries"
)

TEST_CACHE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

TEST_QUERY_CACHE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

test_candidate_ids_ordered = tuple(
    test_candidate_ids
)

test_query_set_sha256 = hashlib.sha256(
    canonical_json_bytes(
        list(test_candidate_ids_ordered)
    )
).hexdigest()

configuration_file_sha256_before_test = (
    sha256_path_file(
        RERANKER_CONFIGURATION_PATH
    )
)

configuration_payload_sha256_before_test = (
    reranker_configuration[
        "configuration_sha256"
    ]
)

assert (
    configuration_file_sha256_before_test
    == reranker_configuration_file_sha256
)

assert (
    configuration_payload_sha256_before_test
    == reranker_configuration_sha256
)

test_cache_contract = {
    "cache_version": TEST_CACHE_VERSION,
    "split": "test",
    "query_count": len(
        test_candidate_ids_ordered
    ),
    "query_set_sha256": (
        test_query_set_sha256
    ),
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "candidate_depth": (
        FROZEN_TEST_CANDIDATE_DEPTH
    ),
    "model_id": (
        reranker_configuration[
            "reranker"
        ]["model_id"]
    ),
    "model_revision": (
        reranker_configuration[
            "reranker"
        ]["revision"]
    ),
    "runtime": "ONNX_RUNTIME_CPU",
    "onnx_model_sha256": (
        onnx_model_sha256
    ),
    "batch_size": (
        SELECTED_ONNX_BATCH_SIZE
    ),
    "test_result_may_change_configuration": (
        False
    ),
}

if TEST_CACHE_METADATA_PATH.exists():
    existing_test_contract = json.loads(
        TEST_CACHE_METADATA_PATH.read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_test_contract
        == test_cache_contract
    ), (
        "Existing Test cache belongs to a "
        "different frozen configuration."
    )
else:
    write_json_atomic(
        TEST_CACHE_METADATA_PATH,
        test_cache_contract,
    )


def test_query_cache_path(
    candidate_id,
):
    candidate_hash = hashlib.sha256(
        candidate_id.encode("utf-8")
    ).hexdigest()

    return (
        TEST_QUERY_CACHE_DIRECTORY
        / f"{candidate_hash}.json"
    )


def validate_test_cache_row(
    row,
    candidate_id,
):
    if not isinstance(row, dict):
        return False

    if (
        row.get("candidate_id")
        != candidate_id
    ):
        return False

    if row.get("split") != "test":
        return False

    if (
        row.get("candidate_depth")
        != FROZEN_TEST_CANDIDATE_DEPTH
    ):
        return False

    if (
        row.get("configuration_sha256")
        != reranker_configuration_sha256
    ):
        return False

    chunk_ids = row.get("chunk_ids")
    scores = row.get("scores")

    if not isinstance(chunk_ids, list):
        return False

    if not isinstance(scores, list):
        return False

    if len(chunk_ids) != (
        FROZEN_TEST_CANDIDATE_DEPTH
    ):
        return False

    if len(scores) != (
        FROZEN_TEST_CANDIDATE_DEPTH
    ):
        return False

    if len(set(chunk_ids)) != len(
        chunk_ids
    ):
        return False

    if not all(
        math.isfinite(float(score))
        for score in scores
    ):
        return False

    expected_chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate
        in candidate_record_by_id[
            candidate_id
        ]["candidates"][
            :FROZEN_TEST_CANDIDATE_DEPTH
        ]
    ]

    return chunk_ids == expected_chunk_ids


def load_valid_test_cache():
    cached_rows = {}
    invalid_paths = []

    for candidate_id in (
        test_candidate_ids_ordered
    ):
        cache_path = test_query_cache_path(
            candidate_id
        )

        if not cache_path.exists():
            continue

        try:
            row = json.loads(
                cache_path.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            invalid_paths.append(
                str(cache_path)
            )
            continue

        if validate_test_cache_row(
            row=row,
            candidate_id=candidate_id,
        ):
            cached_rows[candidate_id] = row
        else:
            invalid_paths.append(
                str(cache_path)
            )

    assert not invalid_paths, {
        "invalid_test_cache_paths": (
            invalid_paths
        )
    }

    return cached_rows


test_score_rows_by_candidate_id = (
    load_valid_test_cache()
)

initial_test_cached_query_count = len(
    test_score_rows_by_candidate_id
)

pending_test_candidate_ids = [
    candidate_id
    for candidate_id
    in test_candidate_ids_ordered
    if candidate_id
    not in test_score_rows_by_candidate_id
]

# Test access begins only after configuration freeze.
TEST_QUERIES_ACCESSED = True

assert RERANKER_CONFIGURATION_FROZEN
assert TEST_QUERIES_ACCESSED

print(
    json.dumps(
        {
            "execution": (
                "INDEPENDENT_FROZEN_TEST_RERANKING"
            ),
            "total_query_count": len(
                test_candidate_ids_ordered
            ),
            "initial_cached_query_count": (
                initial_test_cached_query_count
            ),
            "pending_query_count": len(
                pending_test_candidate_ids
            ),
            "candidate_depth": (
                FROZEN_TEST_CANDIDATE_DEPTH
            ),
            "configuration_sha256": (
                reranker_configuration_sha256
            ),
            "resume_supported": True,
            "status": "STARTING",
        },
        ensure_ascii=False,
        indent=2,
    )
)

test_execution_started = (
    time.perf_counter()
)

newly_scored_test_query_count = 0
newly_scored_test_pair_count = 0
current_test_scoring_seconds = 0.0

for pending_index, candidate_id in enumerate(
    pending_test_candidate_ids,
    start=1,
):
    question = extract_question_text(
        golden_record_by_candidate_id[
            candidate_id
        ]
    )

    candidate_items = (
        candidate_record_by_id[
            candidate_id
        ]["candidates"][
            :FROZEN_TEST_CANDIDATE_DEPTH
        ]
    )

    chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate in candidate_items
    ]

    assert len(chunk_ids) == (
        FROZEN_TEST_CANDIDATE_DEPTH
    )

    assert len(set(chunk_ids)) == (
        FROZEN_TEST_CANDIDATE_DEPTH
    )

    query_pairs = [
        (
            question,
            chunk_record_by_id[
                chunk_id
            ]["content"],
        )
        for chunk_id in chunk_ids
    ]

    query_started = time.perf_counter()

    query_scores = predict_onnx_pairs(
        query_pairs,
        batch_size=(
            SELECTED_ONNX_BATCH_SIZE
        ),
    )

    query_elapsed_seconds = (
        time.perf_counter()
        - query_started
    )

    assert query_scores.shape == (
        FROZEN_TEST_CANDIDATE_DEPTH,
    )

    assert np.all(
        np.isfinite(query_scores)
    )

    test_cache_row = {
        "candidate_id": candidate_id,
        "split": "test",
        "source_type": (
            extract_query_source_type(
                candidate_id
            )
        ),
        "configuration_sha256": (
            reranker_configuration_sha256
        ),
        "candidate_depth": (
            FROZEN_TEST_CANDIDATE_DEPTH
        ),
        "chunk_ids": chunk_ids,
        "scores": [
            float(score)
            for score in query_scores
        ],
        "elapsed_seconds": float(
            query_elapsed_seconds
        ),
    }

    assert validate_test_cache_row(
        row=test_cache_row,
        candidate_id=candidate_id,
    )

    write_json_atomic(
        test_query_cache_path(
            candidate_id
        ),
        test_cache_row,
    )

    test_score_rows_by_candidate_id[
        candidate_id
    ] = test_cache_row

    newly_scored_test_query_count += 1
    newly_scored_test_pair_count += len(
        query_pairs
    )

    current_test_scoring_seconds += (
        query_elapsed_seconds
    )

    completed_total = len(
        test_score_rows_by_candidate_id
    )

    if (
        pending_index == 1
        or pending_index % 10 == 0
        or pending_index
        == len(pending_test_candidate_ids)
    ):
        average_seconds_per_query = (
            current_test_scoring_seconds
            / newly_scored_test_query_count
        )

        remaining_query_count = (
            len(test_candidate_ids_ordered)
            - completed_total
        )

        projected_remaining_hours = (
            remaining_query_count
            * average_seconds_per_query
            / 3600.0
        )

        print(
            json.dumps(
                {
                    "completed_queries": (
                        completed_total
                    ),
                    "total_queries": len(
                        test_candidate_ids_ordered
                    ),
                    "newly_scored_queries": (
                        newly_scored_test_query_count
                    ),
                    "remaining_queries": (
                        remaining_query_count
                    ),
                    "projected_remaining_hours": (
                        projected_remaining_hours
                    ),
                },
                ensure_ascii=False,
            )
        )

test_execution_seconds = (
    time.perf_counter()
    - test_execution_started
)

test_score_rows_by_candidate_id = (
    load_valid_test_cache()
)

assert len(
    test_score_rows_by_candidate_id
) == len(test_candidate_ids_ordered)

test_score_payload = [
    test_score_rows_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in test_candidate_ids_ordered
]

test_score_sha256 = hashlib.sha256(
    canonical_json_bytes(
        test_score_payload
    )
).hexdigest()

configuration_file_sha256_after_test = (
    sha256_path_file(
        RERANKER_CONFIGURATION_PATH
    )
)

assert (
    configuration_file_sha256_after_test
    == configuration_file_sha256_before_test
)

assert (
    reranker_configuration[
        "configuration_sha256"
    ]
    == configuration_payload_sha256_before_test
)

test_recorded_scoring_seconds = sum(
    float(row["elapsed_seconds"])
    for row
    in test_score_rows_by_candidate_id.values()
)

test_pair_count = sum(
    len(row["scores"])
    for row
    in test_score_rows_by_candidate_id.values()
)

test_observed_pairs_per_second = float(
    test_pair_count
    / test_recorded_scoring_seconds
)

independent_test_execution = {
    "split": "test",
    "query_count": len(
        test_score_rows_by_candidate_id
    ),
    "candidate_depth": (
        FROZEN_TEST_CANDIDATE_DEPTH
    ),
    "pair_count": test_pair_count,
    "initial_cached_query_count": (
        initial_test_cached_query_count
    ),
    "newly_scored_query_count": (
        newly_scored_test_query_count
    ),
    "newly_scored_pair_count": (
        newly_scored_test_pair_count
    ),
    "current_run_seconds": float(
        test_execution_seconds
    ),
    "recorded_model_scoring_seconds": float(
        test_recorded_scoring_seconds
    ),
    "observed_pairs_per_second": (
        test_observed_pairs_per_second
    ),
    "test_query_set_sha256": (
        test_query_set_sha256
    ),
    "test_score_sha256": (
        test_score_sha256
    ),
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "configuration_file_changed": False,
    "configuration_changed_after_test": False,
    "cache": {
        "directory": str(
            TEST_CACHE_DIRECTORY.relative_to(
                PROJECT_ROOT
            )
        ),
        "query_checkpoint_count": len(
            test_score_rows_by_candidate_id
        ),
        "resume_supported": True,
        "atomic_per_query": True,
    },
    "test_queries_accessed": (
        TEST_QUERIES_ACCESSED
    ),
    "repository_mutated": True,
    "paid_api_calls": 0,
    "independent_test_execution_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        independent_test_execution,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "execution": "INDEPENDENT_FROZEN_TEST_RERANKING",
  "total_query_count": 1149,
  "initial_cached_query_count": 1149,
  "pending_query_count": 0,
  "candidate_depth": 50,
  "configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
  "resume_supported": true,
  "status": "STARTING"
}
{
  "split": "test",
  "query_count": 1149,
  "candidate_depth": 50,
  "pair_count": 57450,
  "initial_cached_query_count": 1149,
  "newly_scored_query_count": 0,
  "newly_scored_pair_count": 0,
  "current_run_seconds": 0.0004993669999748818,
  "recorded_model_scoring_seconds": 8113.274928389066,
  "observed_pairs_per_second": 7.080987703125574,
  "test_query_set_sha256": "6ccafc24b7d662a8513f27d860950f4d06a7e0b67cd527e859afbfa396fc3bcf",
  "test_score_sha256": "06c5f7d46863e91bf3da6a51159ed66ee0e69f6675b90c468d4901d03306eb0a",
  "configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
  "configuration_file_changed": false,
  "configura

<div dir="rtl" align="right">

## ارزیابی نهایی کیفیت Reranker روی Test Split

در این مرحله امتیازهای ذخیره‌شده Test به رتبه‌بندی نهایی تبدیل می‌شوند و با Hybrid Baseline مقایسه می‌شوند.

گزارش شامل موارد زیر است:

- معیارهای کلی Test؛
- تغییر معیارها نسبت به Hybrid Baseline؛
- تحلیل جداگانه Book، Transcript و Website؛
- Bootstrap زوجی `nDCG@10`؛
- شمار Queryهای بهبودیافته، بدون تغییر و پسرفت‌کرده؛
- Latency واقعی Test؛
- کنترل عدم تغییر Configuration پس از Test.

نتیجه این مرحله صرفاً ارزیابی مستقل است و تنظیمات Frozen را تغییر نمی‌دهد.

</div>

In [19]:
# Code Cell 19 — Evaluate frozen reranker on Test split

EXPECTED_NB16_TEST_TOP_K_METRICS = {
    "hit_at_10": 0.7554395126196692,
    "hit_at_5": 0.6962576153176675,
    "ndcg_at_10": 0.5216652301206817,
    "ndcg_at_5": 0.4983407459398288,
    "precision_at_10": 0.10208877284595298,
    "precision_at_5": 0.1805047867711053,
    "recall_at_10": 0.5998580546230676,
    "recall_at_5": 0.5336607401881552,
}

TEST_BOOTSTRAP_SAMPLE_COUNT = 5000
TEST_BOOTSTRAP_SEED = 17019

test_baseline_rows = []
test_reranker_rows = []

for candidate_id in test_candidate_ids_ordered:
    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    source_type = extract_query_source_type(
        candidate_id
    )

    baseline_chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate
        in candidate_record_by_id[
            candidate_id
        ]["candidates"]
    ]

    baseline_metrics = (
        evaluate_ranked_chunk_ids(
            ranked_chunk_ids=(
                baseline_chunk_ids
            ),
            relevant_chunk_ids=(
                relevant_chunk_ids
            ),
        )
    )

    test_baseline_rows.append(
        {
            "candidate_id": candidate_id,
            "source_type": source_type,
            **baseline_metrics,
        }
    )

    score_row = (
        test_score_rows_by_candidate_id[
            candidate_id
        ]
    )

    reranked_chunk_ids = (
        rank_reranker_candidates(
            chunk_ids=score_row[
                "chunk_ids"
            ],
            scores=score_row["scores"],
        )
    )

    reranker_metrics = (
        evaluate_ranked_chunk_ids(
            ranked_chunk_ids=(
                reranked_chunk_ids
            ),
            relevant_chunk_ids=(
                relevant_chunk_ids
            ),
        )
    )

    test_reranker_rows.append(
        {
            "candidate_id": candidate_id,
            "source_type": source_type,
            "ranked_chunk_ids": (
                reranked_chunk_ids
            ),
            **reranker_metrics,
        }
    )

assert len(test_baseline_rows) == 1149
assert len(test_reranker_rows) == 1149

test_baseline_metrics = (
    aggregate_metric_rows(
        test_baseline_rows
    )
)

test_reranker_metrics = (
    aggregate_metric_rows(
        test_reranker_rows
    )
)

nb16_test_parity_deltas = {
    metric_name: float(
        test_baseline_metrics[
            metric_name
        ]
        - expected_value
    )
    for metric_name, expected_value
    in EXPECTED_NB16_TEST_TOP_K_METRICS.items()
}

assert all(
    abs(delta) <= 1e-15
    for delta
    in nb16_test_parity_deltas.values()
)

test_metric_deltas = {
    metric_name: float(
        test_reranker_metrics[
            metric_name
        ]
        - test_baseline_metrics[
            metric_name
        ]
    )
    for metric_name in METRIC_NAMES
}

test_baseline_by_source_type = {}
test_reranker_by_source_type = {}
test_delta_by_source_type = {}

for source_type in EXPECTED_SOURCE_TYPES:
    source_baseline_rows = [
        row
        for row in test_baseline_rows
        if row["source_type"]
        == source_type
    ]

    source_reranker_rows = [
        row
        for row in test_reranker_rows
        if row["source_type"]
        == source_type
    ]

    assert len(source_baseline_rows) == len(
        source_reranker_rows
    )

    baseline_source_metrics = (
        aggregate_metric_rows(
            source_baseline_rows
        )
    )

    reranker_source_metrics = (
        aggregate_metric_rows(
            source_reranker_rows
        )
    )

    test_baseline_by_source_type[
        source_type
    ] = baseline_source_metrics

    test_reranker_by_source_type[
        source_type
    ] = reranker_source_metrics

    test_delta_by_source_type[
        source_type
    ] = {
        metric_name: float(
            reranker_source_metrics[
                metric_name
            ]
            - baseline_source_metrics[
                metric_name
            ]
        )
        for metric_name in METRIC_NAMES
    }

test_baseline_row_by_id = {
    row["candidate_id"]: row
    for row in test_baseline_rows
}

test_reranker_row_by_id = {
    row["candidate_id"]: row
    for row in test_reranker_rows
}

test_ndcg_deltas = np.asarray(
    [
        (
            test_reranker_row_by_id[
                candidate_id
            ]["ndcg_at_10"]
            - test_baseline_row_by_id[
                candidate_id
            ]["ndcg_at_10"]
        )
        for candidate_id
        in test_candidate_ids_ordered
    ],
    dtype=np.float64,
)

test_improved_query_count = int(
    np.sum(test_ndcg_deltas > 0.0)
)

test_unchanged_query_count = int(
    np.sum(test_ndcg_deltas == 0.0)
)

test_regressed_query_count = int(
    np.sum(test_ndcg_deltas < 0.0)
)

test_bootstrap_rng = (
    np.random.default_rng(
        TEST_BOOTSTRAP_SEED
    )
)

test_bootstrap_mean_deltas = np.empty(
    TEST_BOOTSTRAP_SAMPLE_COUNT,
    dtype=np.float64,
)

for bootstrap_index in range(
    TEST_BOOTSTRAP_SAMPLE_COUNT
):
    sampled_indexes = (
        test_bootstrap_rng.integers(
            low=0,
            high=len(test_ndcg_deltas),
            size=len(test_ndcg_deltas),
        )
    )

    test_bootstrap_mean_deltas[
        bootstrap_index
    ] = float(
        np.mean(
            test_ndcg_deltas[
                sampled_indexes
            ]
        )
    )

test_bootstrap_summary = {
    "sample_count": (
        TEST_BOOTSTRAP_SAMPLE_COUNT
    ),
    "seed": TEST_BOOTSTRAP_SEED,
    "mean_delta": float(
        test_ndcg_deltas.mean()
    ),
    "confidence_interval_95": {
        "lower": float(
            np.percentile(
                test_bootstrap_mean_deltas,
                2.5,
            )
        ),
        "upper": float(
            np.percentile(
                test_bootstrap_mean_deltas,
                97.5,
            )
        ),
    },
    "probability_delta_positive": float(
        np.mean(
            test_bootstrap_mean_deltas
            > 0.0
        )
    ),
}

test_latency_values = np.asarray(
    [
        float(row["elapsed_seconds"])
        for row
        in test_score_rows_by_candidate_id
        .values()
    ],
    dtype=np.float64,
)

test_latency_summary = {
    "query_count": int(
        test_latency_values.size
    ),
    "minimum_seconds": float(
        test_latency_values.min()
    ),
    "median_p50_seconds": float(
        np.median(test_latency_values)
    ),
    "p95_seconds": float(
        np.percentile(
            test_latency_values,
            95,
        )
    ),
    "maximum_seconds": float(
        test_latency_values.max()
    ),
    "scope": (
        "LOCAL_ONNX_CPU_DEPTH_50"
    ),
}

configuration_file_sha256_final = (
    sha256_path_file(
        RERANKER_CONFIGURATION_PATH
    )
)

assert (
    configuration_file_sha256_final
    == reranker_configuration_file_sha256
)

assert (
    reranker_configuration[
        "configuration_sha256"
    ]
    == reranker_configuration_sha256
)

minimum_test_source_ndcg_delta = min(
    test_delta_by_source_type[
        source_type
    ]["ndcg_at_10"]
    for source_type
    in EXPECTED_SOURCE_TYPES
)

test_quality_observation = (
    "POSITIVE_GENERALIZATION"
    if (
        test_metric_deltas[
            "ndcg_at_10"
        ] > 0.0
        and test_bootstrap_summary[
            "confidence_interval_95"
        ]["lower"] > 0.0
    )
    else "NO_POSITIVE_GENERALIZATION"
)

independent_test_evaluation = {
    "scope": {
        "split": "test",
        "query_count": len(
            test_candidate_ids_ordered
        ),
        "candidate_depth": (
            FROZEN_TEST_CANDIDATE_DEPTH
        ),
        "configuration_sha256": (
            reranker_configuration_sha256
        ),
    },
    "hybrid_baseline": {
        "metrics": (
            test_baseline_metrics
        ),
        "nb16_top_k_metric_parity": (
            "PASS"
        ),
        "maximum_top_k_metric_delta": (
            max(
                abs(delta)
                for delta
                in nb16_test_parity_deltas.values()
            )
        ),
    },
    "reranker": {
        "metrics": test_reranker_metrics,
        "delta_vs_hybrid_baseline": (
            test_metric_deltas
        ),
    },
    "source_type_analysis": {
        source_type: {
            "query_count": sum(
                1
                for row
                in test_reranker_rows
                if row["source_type"]
                == source_type
            ),
            "baseline_metrics": (
                test_baseline_by_source_type[
                    source_type
                ]
            ),
            "reranker_metrics": (
                test_reranker_by_source_type[
                    source_type
                ]
            ),
            "delta": (
                test_delta_by_source_type[
                    source_type
                ]
            ),
        }
        for source_type
        in EXPECTED_SOURCE_TYPES
    },
    "query_outcomes": {
        "improved_count": (
            test_improved_query_count
        ),
        "unchanged_count": (
            test_unchanged_query_count
        ),
        "regressed_count": (
            test_regressed_query_count
        ),
    },
    "bootstrap_ndcg_at_10": (
        test_bootstrap_summary
    ),
    "latency": test_latency_summary,
    "minimum_source_ndcg_at_10_delta": (
        minimum_test_source_ndcg_delta
    ),
    "quality_observation": (
        test_quality_observation
    ),
    "configuration_changed_after_test": (
        False
    ),
    "test_queries_accessed": True,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "independent_test_evaluation_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        independent_test_evaluation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "test",
    "query_count": 1149,
    "candidate_depth": 50,
    "configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738"
  },
  "hybrid_baseline": {
    "metrics": {
      "hit_at_10": 0.7554395126196692,
      "hit_at_5": 0.6962576153176675,
      "ndcg_at_10": 0.5216652301206817,
      "ndcg_at_5": 0.4983407459398288,
      "precision_at_10": 0.10208877284595298,
      "precision_at_5": 0.1805047867711053,
      "recall_at_10": 0.5998580546230676,
      "recall_at_5": 0.5336607401881552,
      "reciprocal_rank": 0.5899166127502804
    },
    "nb16_top_k_metric_parity": "PASS",
    "maximum_top_k_metric_delta": 0.0
  },
  "reranker": {
    "metrics": {
      "hit_at_10": 0.803307223672759,
      "hit_at_5": 0.7493472584856397,
      "ndcg_at_10": 0.5735362065701753,
      "ndcg_at_5": 0.5496140572542837,
      "precision_at_10": 0.10809399477806789,
      "precision_at_5": 0.18920800696257617,
      "recall_at_10": 0.634317

<div dir="rtl" align="right">

## ساخت Artifactهای نسخه‌دار Reranking

در این مرحله خروجی‌های نهایی NB17 ساخته می‌شوند:

- Candidateهای Rerank‌شده برای تمام Development و Test؛
- گزارش مستقل Test؛
- Evidence اجرای Reranker؛
- Manifest نسخه‌دار؛
- Release اولیه؛
- Acceptance Report؛
- قرارداد تحویل به `18_context_builder.ipynb`.

وضعیت Release تا زمان Reload Validation برابر `PENDING_RELOAD_VALIDATION` باقی می‌ماند.

</div>

In [20]:
# Code Cell 20 — Materialize versioned reranking artifacts

RERANKER_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "phase2"
    / "retrieval"
    / "reranker_v1"
)

RERANKER_ARTIFACT_DIRECTORY = (
    PROJECT_ROOT
    / "artifacts"
    / "phase2"
    / "reranking"
)

RERANKER_REPORT_DIRECTORY = (
    PROJECT_ROOT
    / "reports"
    / "phase2"
)

RERANKED_CANDIDATES_PATH = (
    RERANKER_DATA_DIRECTORY
    / "reranked_candidates_v1.jsonl"
)

RERANKER_EVIDENCE_PATH = (
    RERANKER_DATA_DIRECTORY
    / "reranker_evidence_v1.json"
)

RERANKER_MANIFEST_PATH = (
    RERANKER_DATA_DIRECTORY
    / "reranker_manifest.json"
)

RERANKER_TEST_EVALUATION_PATH = (
    RERANKER_REPORT_DIRECTORY
    / "reranker_test_evaluation_v1.json"
)

RERANKER_ACCEPTANCE_PATH = (
    RERANKER_REPORT_DIRECTORY
    / "reranker_acceptance_v1.json"
)

RERANKER_RELEASE_PATH = (
    RERANKER_ARTIFACT_DIRECTORY
    / "reranker_release_v1.json"
)

NB18_RERANKER_HANDOFF_PATH = (
    RERANKER_ARTIFACT_DIRECTORY
    / "nb18_reranker_handoff_v1.json"
)

for directory in (
    RERANKER_DATA_DIRECTORY,
    RERANKER_ARTIFACT_DIRECTORY,
    RERANKER_REPORT_DIRECTORY,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def write_jsonl_atomic(
    path,
    records,
):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        for record in records:
            file_handle.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                )
            )
            file_handle.write("\n")

        file_handle.flush()
        os.fsync(file_handle.fileno())

    temporary_path.replace(path)


all_score_rows_by_candidate_id = {
    **development_score_rows_by_candidate_id,
    **test_score_rows_by_candidate_id,
}

assert len(
    all_score_rows_by_candidate_id
) == 3813

reranked_candidate_records = []

for hybrid_record in hybrid_candidate_records:
    candidate_id = hybrid_record[
        "candidate_id"
    ]

    split_name = hybrid_record["split"]

    score_row = (
        all_score_rows_by_candidate_id[
            candidate_id
        ]
    )

    assert score_row["split"] == split_name
    assert (
        score_row["configuration_sha256"]
        == reranker_configuration_sha256
    ) if split_name == "test" else True

    original_candidates = (
        hybrid_record["candidates"][
            :FULL_DEVELOPMENT_CANDIDATE_DEPTH
        ]
    )

    original_chunk_ids = [
        extract_candidate_chunk_id(
            candidate
        )
        for candidate in original_candidates
    ]

    assert (
        original_chunk_ids
        == score_row["chunk_ids"]
    )

    original_by_chunk_id = {
        chunk_id: {
            "hybrid_rank": rank,
            "hybrid_candidate": candidate,
        }
        for rank, (
            chunk_id,
            candidate,
        )
        in enumerate(
            zip(
                original_chunk_ids,
                original_candidates,
            ),
            start=1,
        )
    }

    reranked_chunk_ids = (
        rank_reranker_candidates(
            chunk_ids=score_row[
                "chunk_ids"
            ],
            scores=score_row["scores"],
        )
    )

    score_by_chunk_id = {
        chunk_id: float(score)
        for chunk_id, score
        in zip(
            score_row["chunk_ids"],
            score_row["scores"],
        )
    }

    reranked_candidates = [
        {
            "rank": reranker_rank,
            "chunk_id": chunk_id,
            "reranker_score": (
                score_by_chunk_id[
                    chunk_id
                ]
            ),
            "hybrid_rank": (
                original_by_chunk_id[
                    chunk_id
                ]["hybrid_rank"]
            ),
            "hybrid_candidate": (
                original_by_chunk_id[
                    chunk_id
                ]["hybrid_candidate"]
            ),
        }
        for reranker_rank, chunk_id
        in enumerate(
            reranked_chunk_ids,
            start=1,
        )
    ]

    reranked_candidate_records.append(
        {
            "candidate_id": candidate_id,
            "split": split_name,
            "source_type": (
                extract_query_source_type(
                    candidate_id
                )
            ),
            "release_id": (
                RERANKER_RELEASE_ID
            ),
            "configuration_sha256": (
                reranker_configuration_sha256
            ),
            "candidate_depth": (
                FULL_DEVELOPMENT_CANDIDATE_DEPTH
            ),
            "final_top_k": 10,
            "candidates": (
                reranked_candidates
            ),
        }
    )

assert len(
    reranked_candidate_records
) == 3813

assert sum(
    record["split"] == "development"
    for record
    in reranked_candidate_records
) == 2664

assert sum(
    record["split"] == "test"
    for record
    in reranked_candidate_records
) == 1149

assert all(
    len(record["candidates"]) == 50
    for record
    in reranked_candidate_records
)

write_jsonl_atomic(
    RERANKED_CANDIDATES_PATH,
    reranked_candidate_records,
)

reranked_candidates_file_sha256 = (
    sha256_path_file(
        RERANKED_CANDIDATES_PATH
    )
)

reranker_test_evaluation_artifact = {
    "artifact_type": (
        "NB17_INDEPENDENT_TEST_EVALUATION"
    ),
    "schema_version": "1.0",
    "release_id": RERANKER_RELEASE_ID,
    **independent_test_evaluation,
}

write_json_atomic(
    RERANKER_TEST_EVALUATION_PATH,
    reranker_test_evaluation_artifact,
)

reranker_test_evaluation_sha256 = (
    sha256_path_file(
        RERANKER_TEST_EVALUATION_PATH
    )
)

reranker_evidence = {
    "artifact_type": (
        "NB17_RERANKER_EVIDENCE"
    ),
    "schema_version": "1.0",
    "release_id": RERANKER_RELEASE_ID,
    "configuration": {
        "path": str(
            RERANKER_CONFIGURATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "file_sha256": (
            reranker_configuration_file_sha256
        ),
        "payload_sha256": (
            reranker_configuration_sha256
        ),
    },
    "adr": {
        "path": str(
            RERANKER_ADR_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": (
            reranker_adr_file_sha256
        ),
        "payload_sha256": (
            reranker_adr_sha256
        ),
    },
    "model": {
        "model_id": (
            PINNED_RERANKER_CANDIDATE[
                "model_id"
            ]
        ),
        "revision": (
            PINNED_RERANKER_CANDIDATE[
                "revision"
            ]
        ),
        "onnx_model_sha256": (
            onnx_model_sha256
        ),
    },
    "development": {
        "query_count": 2664,
        "candidate_depth": 50,
        "score_sha256": (
            development_score_sha256
        ),
        "baseline_metrics": (
            full_development_baseline_metrics
        ),
        "reranker_metrics": (
            full_development_reranker_metrics
        ),
        "delta": overall_metric_deltas,
        "bootstrap": (
            full_bootstrap_summary
        ),
        "failure_taxonomy": dict(
            sorted(
                failure_taxonomy_counter.items()
            )
        ),
    },
    "test": {
        "query_count": 1149,
        "candidate_depth": 50,
        "score_sha256": (
            test_score_sha256
        ),
        "baseline_metrics": (
            test_baseline_metrics
        ),
        "reranker_metrics": (
            test_reranker_metrics
        ),
        "delta": test_metric_deltas,
        "bootstrap": (
            test_bootstrap_summary
        ),
        "quality_observation": (
            test_quality_observation
        ),
    },
    "configuration_changed_after_test": (
        False
    ),
    "paid_api_calls": 0,
    "evidence_status": "PASS",
}

write_json_atomic(
    RERANKER_EVIDENCE_PATH,
    reranker_evidence,
)

reranker_evidence_sha256 = (
    sha256_path_file(
        RERANKER_EVIDENCE_PATH
    )
)

reranker_manifest = {
    "artifact_type": (
        "NB17_RERANKER_MANIFEST"
    ),
    "schema_version": "1.0",
    "release_id": RERANKER_RELEASE_ID,
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "input": {
        "nb16_hybrid_candidate_path": (
            "data/phase2/retrieval/"
            "hybrid_retrieval_v1/"
            "hybrid_candidates_v1.jsonl"
        ),
        "nb16_hybrid_candidate_sha256": (
            "f54277680f3d20681b1585915f54bb44"
            "85ecb1fcccb15c27df0061ff0725d19a"
        ),
        "candidate_record_count": 3813,
        "input_candidate_depth": 100,
    },
    "output": {
        "path": str(
            RERANKED_CANDIDATES_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranked_candidates_file_sha256
        ),
        "record_count": 3813,
        "development_record_count": 2664,
        "test_record_count": 1149,
        "candidate_depth": 50,
        "final_top_k": 10,
    },
    "evidence": {
        "path": str(
            RERANKER_EVIDENCE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_evidence_sha256
        ),
    },
    "test_evaluation": {
        "path": str(
            RERANKER_TEST_EVALUATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_test_evaluation_sha256
        ),
    },
    "configuration_changed_after_test": (
        False
    ),
    "manifest_status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
}

write_json_atomic(
    RERANKER_MANIFEST_PATH,
    reranker_manifest,
)

reranker_manifest_sha256 = (
    sha256_path_file(
        RERANKER_MANIFEST_PATH
    )
)

reranker_release = {
    "artifact_type": (
        "NB17_RERANKER_RELEASE"
    ),
    "schema_version": "1.0",
    "release_id": RERANKER_RELEASE_ID,
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "logical_reranker_output_sha256": (
        reranked_candidates_file_sha256
    ),
    "manifest": {
        "path": str(
            RERANKER_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_manifest_sha256
        ),
    },
    "candidate_artifact": {
        "path": str(
            RERANKED_CANDIDATES_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranked_candidates_file_sha256
        ),
        "record_count": 3813,
        "candidate_depth": 50,
    },
    "development_metrics": (
        full_development_reranker_metrics
    ),
    "test_metrics": (
        test_reranker_metrics
    ),
    "configuration_changed_after_test": (
        False
    ),
    "status": (
        "FROZEN_FOR_CONTEXT_BUILDING_PENDING_RELOAD"
    ),
}

write_json_atomic(
    RERANKER_RELEASE_PATH,
    reranker_release,
)

reranker_release_sha256 = (
    sha256_path_file(
        RERANKER_RELEASE_PATH
    )
)

nb18_reranker_handoff = {
    "artifact_type": (
        "NB18_RERANKER_HANDOFF"
    ),
    "schema_version": "1.0",
    "source_notebook": (
        "17_reranking.ipynb"
    ),
    "target_notebook": (
        "18_context_builder.ipynb"
    ),
    "release_id": RERANKER_RELEASE_ID,
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "release": {
        "path": str(
            RERANKER_RELEASE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_release_sha256
        ),
    },
    "manifest": {
        "path": str(
            RERANKER_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_manifest_sha256
        ),
    },
    "candidate_artifact": {
        "path": str(
            RERANKED_CANDIDATES_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranked_candidates_file_sha256
        ),
        "record_count": 3813,
        "candidate_depth": 50,
        "consumer_top_k": 10,
    },
    "constraints": {
        "do_not_change_reranker_configuration": (
            True
        ),
        "do_not_recompute_reranker_scores": (
            True
        ),
        "preserve_candidate_order": True,
    },
    "status": (
        "READY_FOR_NB18_PENDING_RELOAD"
    ),
}

write_json_atomic(
    NB18_RERANKER_HANDOFF_PATH,
    nb18_reranker_handoff,
)

nb18_reranker_handoff_sha256 = (
    sha256_path_file(
        NB18_RERANKER_HANDOFF_PATH
    )
)

reranker_acceptance = {
    "artifact_type": (
        "NB17_RERANKER_ACCEPTANCE"
    ),
    "schema_version": "1.0",
    "release_id": RERANKER_RELEASE_ID,
    "gates": {
        "canonical_input_gate": "PASS",
        "pilot_gate": "PASS",
        "development_gate": "PASS",
        "source_slice_gate": "PASS",
        "bootstrap_gate": "PASS",
        "failure_analysis_gate": "PASS",
        "configuration_freeze_gate": "PASS",
        "independent_test_gate": "PASS",
        "configuration_immutability_gate": (
            "PASS"
        ),
        "reload_validation_gate": (
            "PENDING"
        ),
    },
    "status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
}

write_json_atomic(
    RERANKER_ACCEPTANCE_PATH,
    reranker_acceptance,
)

reranker_acceptance_sha256 = (
    sha256_path_file(
        RERANKER_ACCEPTANCE_PATH
    )
)

artifact_materialization = {
    "release_id": RERANKER_RELEASE_ID,
    "candidate_artifact": {
        "path": str(
            RERANKED_CANDIDATES_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranked_candidates_file_sha256
        ),
        "record_count": 3813,
        "candidate_depth": 50,
    },
    "configuration": {
        "path": str(
            RERANKER_CONFIGURATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_configuration_file_sha256
        ),
    },
    "adr": {
        "path": str(
            RERANKER_ADR_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": (
            reranker_adr_file_sha256
        ),
    },
    "evidence": {
        "path": str(
            RERANKER_EVIDENCE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_evidence_sha256
        ),
    },
    "manifest": {
        "path": str(
            RERANKER_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_manifest_sha256
        ),
    },
    "release": {
        "path": str(
            RERANKER_RELEASE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_release_sha256
        ),
        "status": (
            "FROZEN_FOR_CONTEXT_BUILDING_PENDING_RELOAD"
        ),
    },
    "nb18_handoff": {
        "path": str(
            NB18_RERANKER_HANDOFF_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            nb18_reranker_handoff_sha256
        ),
        "status": (
            "READY_FOR_NB18_PENDING_RELOAD"
        ),
    },
    "acceptance": {
        "path": str(
            RERANKER_ACCEPTANCE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_acceptance_sha256
        ),
        "status": (
            "PASS_PENDING_RELOAD_VALIDATION"
        ),
    },
    "configuration_changed_after_test": (
        False
    ),
    "model_execution_performed": False,
    "files_created": 7,
    "paid_api_calls": 0,
    "artifact_write_status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
}

print(
    json.dumps(
        artifact_materialization,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_reranker_v1",
  "candidate_artifact": {
    "path": "data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl",
    "sha256": "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a",
    "record_count": 3813,
    "candidate_depth": 50
  },
  "configuration": {
    "path": "artifacts/phase2/reranking/reranker_configuration_v1.json",
    "sha256": "dd2290188c4e99be1dada40b167b1041c69f7f077ebe6dea1fddcb58b4834a86"
  },
  "adr": {
    "path": "reports/phase2/reranker_adr_v1.json",
    "sha256": "a86099fefcbfe15d322e81609a33a5693e7df380b99047d456220e21eef14db2"
  },
  "evidence": {
    "path": "data/phase2/retrieval/reranker_v1/reranker_evidence_v1.json",
    "sha256": "7d1b9947b48d1f8deb338ecc5860b961f4ce4b60980c58d4f3bd7080f1e7650f"
  },
  "manifest": {
    "path": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
    "sha256": "ed10199fe3aaa44e4fb7a94f346314aaa768f6d39c2a7c1c172707e1ea3ebeee"
  },
  "release": {
    "path":

<div dir="rtl" align="right">

## Reload Validation و نهایی‌سازی Release

در این مرحله Artifactهای نوشته‌شده مستقل از وضعیت محاسباتی حافظه دوباره بارگذاری می‌شوند.

کنترل‌ها:

- Hash تمام ورودی‌های اصلی؛
- تعداد و ترتیب ۳۸۱۳ رکورد؛
- تعداد ۵۰ Candidate برای هر Query؛
- ترتیب رتبه‌ها و Tie-break؛
- برابری کامل امتیازهای ذخیره‌شده با Checkpointها؛
- بازتولید معیارهای Development و Test؛
- عدم تغییر Configuration و Candidate Artifact؛
- نهایی‌سازی Manifest، Release، Acceptance و NB18 Handoff.

</div>

In [21]:
# Code Cell 21 — Reload validation and finalization

RERANKER_RELOAD_VALIDATION_PATH = (
    RERANKER_DATA_DIRECTORY
    / "reranker_reload_validation_v1.json"
)

assert (
    sha256_path_file(
        RERANKED_CANDIDATES_PATH
    )
    == reranked_candidates_file_sha256
)

assert (
    sha256_path_file(
        RERANKER_CONFIGURATION_PATH
    )
    == reranker_configuration_file_sha256
)

assert (
    sha256_path_file(
        RERANKER_ADR_PATH
    )
    == reranker_adr_file_sha256
)

assert (
    sha256_path_file(
        RERANKER_EVIDENCE_PATH
    )
    == reranker_evidence_sha256
)

assert (
    sha256_path_file(
        RERANKER_TEST_EVALUATION_PATH
    )
    == reranker_test_evaluation_sha256
)

reloaded_configuration = json.loads(
    RERANKER_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_candidate_records = load_jsonl(
    RERANKED_CANDIDATES_PATH
)

assert (
    reloaded_configuration
    == reranker_configuration
)

assert len(
    reloaded_candidate_records
) == 3813

assert [
    record["candidate_id"]
    for record
    in reloaded_candidate_records
] == [
    record["candidate_id"]
    for record
    in hybrid_candidate_records
]

assert all(
    len(record["candidates"]) == 50
    for record
    in reloaded_candidate_records
)

assert all(
    [
        candidate["rank"]
        for candidate
        in record["candidates"]
    ]
    == list(range(1, 51))
    for record
    in reloaded_candidate_records
)

maximum_absolute_score_delta = 0.0
score_mismatch_candidate_ids = []
ranking_mismatch_candidate_ids = []

reloaded_development_metric_rows = []
reloaded_test_metric_rows = []

for record in reloaded_candidate_records:
    candidate_id = record["candidate_id"]

    cached_score_row = (
        all_score_rows_by_candidate_id[
            candidate_id
        ]
    )

    cached_score_by_chunk_id = {
        chunk_id: float(score)
        for chunk_id, score
        in zip(
            cached_score_row["chunk_ids"],
            cached_score_row["scores"],
        )
    }

    expected_ranked_chunk_ids = (
        rank_reranker_candidates(
            chunk_ids=(
                cached_score_row[
                    "chunk_ids"
                ]
            ),
            scores=(
                cached_score_row[
                    "scores"
                ]
            ),
        )
    )

    reloaded_ranked_chunk_ids = [
        candidate["chunk_id"]
        for candidate
        in record["candidates"]
    ]

    if (
        reloaded_ranked_chunk_ids
        != expected_ranked_chunk_ids
    ):
        ranking_mismatch_candidate_ids.append(
            candidate_id
        )

    for candidate in record["candidates"]:
        chunk_id = candidate["chunk_id"]

        score_delta = abs(
            float(
                candidate[
                    "reranker_score"
                ]
            )
            - cached_score_by_chunk_id[
                chunk_id
            ]
        )

        maximum_absolute_score_delta = max(
            maximum_absolute_score_delta,
            score_delta,
        )

        if score_delta != 0.0:
            score_mismatch_candidate_ids.append(
                candidate_id
            )
            break

    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    metrics = evaluate_ranked_chunk_ids(
        ranked_chunk_ids=(
            reloaded_ranked_chunk_ids
        ),
        relevant_chunk_ids=(
            relevant_chunk_ids
        ),
    )

    metric_row = {
        "candidate_id": candidate_id,
        **metrics,
    }

    if record["split"] == "development":
        reloaded_development_metric_rows.append(
            metric_row
        )
    elif record["split"] == "test":
        reloaded_test_metric_rows.append(
            metric_row
        )
    else:
        raise AssertionError(
            record["split"]
        )

assert not ranking_mismatch_candidate_ids
assert not score_mismatch_candidate_ids
assert maximum_absolute_score_delta == 0.0

assert len(
    reloaded_development_metric_rows
) == 2664

assert len(
    reloaded_test_metric_rows
) == 1149

reloaded_development_metrics = (
    aggregate_metric_rows(
        reloaded_development_metric_rows
    )
)

reloaded_test_metrics = (
    aggregate_metric_rows(
        reloaded_test_metric_rows
    )
)

development_reload_metric_deltas = {
    metric_name: float(
        reloaded_development_metrics[
            metric_name
        ]
        - full_development_reranker_metrics[
            metric_name
        ]
    )
    for metric_name in METRIC_NAMES
}

test_reload_metric_deltas = {
    metric_name: float(
        reloaded_test_metrics[
            metric_name
        ]
        - test_reranker_metrics[
            metric_name
        ]
    )
    for metric_name in METRIC_NAMES
}

assert all(
    abs(delta) <= 1e-15
    for delta
    in development_reload_metric_deltas.values()
)

assert all(
    abs(delta) <= 1e-15
    for delta
    in test_reload_metric_deltas.values()
)

assert (
    sha256_path_file(
        RERANKED_CANDIDATES_PATH
    )
    == reranked_candidates_file_sha256
)

assert (
    sha256_path_file(
        RERANKER_CONFIGURATION_PATH
    )
    == reranker_configuration_file_sha256
)

reranker_reload_validation = {
    "artifact_type": (
        "NB17_RERANKER_RELOAD_VALIDATION"
    ),
    "schema_version": "1.0",
    "release_id": RERANKER_RELEASE_ID,
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "logical_reranker_output_sha256": (
        reranked_candidates_file_sha256
    ),
    "candidate_validation": {
        "record_count": len(
            reloaded_candidate_records
        ),
        "development_record_count": len(
            reloaded_development_metric_rows
        ),
        "test_record_count": len(
            reloaded_test_metric_rows
        ),
        "candidate_depth": 50,
        "id_order_parity": "PASS",
        "ranking_parity": "PASS",
        "score_parity": "PASS",
        "maximum_absolute_score_delta": (
            maximum_absolute_score_delta
        ),
    },
    "metric_validation": {
        "development_metrics": (
            reloaded_development_metrics
        ),
        "test_metrics": (
            reloaded_test_metrics
        ),
        "development_metric_parity": (
            "PASS"
        ),
        "test_metric_parity": "PASS",
        "maximum_development_metric_delta": (
            max(
                abs(delta)
                for delta
                in development_reload_metric_deltas.values()
            )
        ),
        "maximum_test_metric_delta": (
            max(
                abs(delta)
                for delta
                in test_reload_metric_deltas.values()
            )
        ),
    },
    "configuration_changed": False,
    "candidate_artifact_changed": False,
    "kernel_state_dependency": False,
    "paid_api_calls": 0,
    "reload_validation_status": "PASS",
}

write_json_atomic(
    RERANKER_RELOAD_VALIDATION_PATH,
    reranker_reload_validation,
)

reranker_reload_validation_sha256 = (
    sha256_path_file(
        RERANKER_RELOAD_VALIDATION_PATH
    )
)

final_manifest = {
    **reranker_manifest,
    "reload_validation": {
        "path": str(
            RERANKER_RELOAD_VALIDATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_reload_validation_sha256
        ),
        "status": "PASS",
    },
    "manifest_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

write_json_atomic(
    RERANKER_MANIFEST_PATH,
    final_manifest,
)

final_manifest_sha256 = sha256_path_file(
    RERANKER_MANIFEST_PATH
)

final_release = {
    **reranker_release,
    "manifest": {
        "path": str(
            RERANKER_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_manifest_sha256
        ),
    },
    "reload_validation": {
        "path": str(
            RERANKER_RELOAD_VALIDATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_reload_validation_sha256
        ),
    },
    "status": (
        "FROZEN_FOR_CONTEXT_BUILDING"
    ),
}

write_json_atomic(
    RERANKER_RELEASE_PATH,
    final_release,
)

final_release_sha256 = sha256_path_file(
    RERANKER_RELEASE_PATH
)

final_nb18_handoff = {
    **nb18_reranker_handoff,
    "release": {
        "path": str(
            RERANKER_RELEASE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_release_sha256
        ),
    },
    "manifest": {
        "path": str(
            RERANKER_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_manifest_sha256
        ),
    },
    "reload_validation": {
        "path": str(
            RERANKER_RELOAD_VALIDATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_reload_validation_sha256
        ),
    },
    "status": "READY_FOR_NB18",
}

write_json_atomic(
    NB18_RERANKER_HANDOFF_PATH,
    final_nb18_handoff,
)

final_nb18_handoff_sha256 = (
    sha256_path_file(
        NB18_RERANKER_HANDOFF_PATH
    )
)

final_acceptance = {
    **reranker_acceptance,
    "gates": {
        **reranker_acceptance["gates"],
        "reload_validation_gate": "PASS",
    },
    "reload_validation": {
        "path": str(
            RERANKER_RELOAD_VALIDATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_reload_validation_sha256
        ),
    },
    "status": "PASS_COMPLETE_FROZEN",
}

write_json_atomic(
    RERANKER_ACCEPTANCE_PATH,
    final_acceptance,
)

final_acceptance_sha256 = (
    sha256_path_file(
        RERANKER_ACCEPTANCE_PATH
    )
)

assert (
    sha256_path_file(
        RERANKED_CANDIDATES_PATH
    )
    == reranked_candidates_file_sha256
)

assert (
    sha256_path_file(
        RERANKER_CONFIGURATION_PATH
    )
    == reranker_configuration_file_sha256
)

assert (
    json.loads(
        RERANKER_RELEASE_PATH.read_text(
            encoding="utf-8"
        )
    )["status"]
    == "FROZEN_FOR_CONTEXT_BUILDING"
)

assert (
    json.loads(
        NB18_RERANKER_HANDOFF_PATH
        .read_text(
            encoding="utf-8"
        )
    )["status"]
    == "READY_FOR_NB18"
)

assert (
    json.loads(
        RERANKER_ACCEPTANCE_PATH.read_text(
            encoding="utf-8"
        )
    )["status"]
    == "PASS_COMPLETE_FROZEN"
)

reranker_finalization = {
    "release_id": RERANKER_RELEASE_ID,
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "logical_reranker_output_sha256": (
        reranked_candidates_file_sha256
    ),
    "candidate_record_count": 3813,
    "candidate_depth": 50,
    "reload_validation": {
        "path": str(
            RERANKER_RELOAD_VALIDATION_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            reranker_reload_validation_sha256
        ),
        "status": "PASS",
    },
    "manifest": {
        "path": str(
            RERANKER_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_manifest_sha256
        ),
        "status": "PASS_COMPLETE_FROZEN",
    },
    "release": {
        "path": str(
            RERANKER_RELEASE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_release_sha256
        ),
        "status": (
            "FROZEN_FOR_CONTEXT_BUILDING"
        ),
    },
    "nb18_handoff": {
        "path": str(
            NB18_RERANKER_HANDOFF_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_nb18_handoff_sha256
        ),
        "status": "READY_FOR_NB18",
    },
    "acceptance": {
        "path": str(
            RERANKER_ACCEPTANCE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_acceptance_sha256
        ),
        "status": "PASS_COMPLETE_FROZEN",
    },
    "configuration_changed": False,
    "candidate_artifact_changed": False,
    "paid_api_calls": 0,
    "finalization_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

print(
    json.dumps(
        reranker_finalization,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_reranker_v1",
  "configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
  "logical_reranker_output_sha256": "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a",
  "candidate_record_count": 3813,
  "candidate_depth": 50,
  "reload_validation": {
    "path": "data/phase2/retrieval/reranker_v1/reranker_reload_validation_v1.json",
    "sha256": "d575550c0d0a3feadfa27025c4f5b359eed5ddac4f91c1e720e303e49c188e4a",
    "status": "PASS"
  },
  "manifest": {
    "path": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
    "sha256": "230a720eab3fac9ef9f4fea8ab03152660f045fbda8926e22abcb37f809e6be5",
    "status": "PASS_COMPLETE_FROZEN"
  },
  "release": {
    "path": "artifacts/phase2/reranking/reranker_release_v1.json",
    "sha256": "8f9006a2cbc425aaa68459928af1c15ab31e88e881d8fbd76a5b312dd601a9b4",
    "status": "FROZEN_FOR_CONTEXT_BUILDING"
  },
  "nb18_handoff": {
    "path": "artifacts/phase2

<div dir="rtl" align="right">

## کنترل نهایی سلامت NB17

این مرحله سلامت کامل نوت‌بوک Reranking را بررسی می‌کند:

- وجود دقیق Code Cellهای ۰۱ تا ۲۲؛
- نبود Code Cell موقت یا بدون شماره؛
- سلامت Syntax؛
- نبود Error باقی‌مانده در خروجی سل‌های قبلی؛
- رعایت قرارداد RTL در Markdownها؛
- صحت Hash تمام Artifactهای نهایی؛
- صحت Release، Manifest، Acceptance و NB18 Handoff؛
- تثبیت Configuration پیش از Test؛
- عدم تغییر Configuration و Candidate Artifact؛
- آمادگی برای `18_context_builder.ipynb`.

پس از PASS این سل، نوت‌بوک باید ذخیره و سپس با `Restart Kernel → Run All` از ابتدا اجرا شود.

</div>

In [23]:
# Code Cell 22 — NB17 final health check

NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "phase2"
    / "17_reranking.ipynb"
)

assert NOTEBOOK_PATH.is_file()

notebook_payload = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)

notebook_cells = notebook_payload[
    "cells"
]

code_cells = [
    cell
    for cell in notebook_cells
    if cell.get("cell_type") == "code"
]

markdown_cells = [
    cell
    for cell in notebook_cells
    if cell.get("cell_type") == "markdown"
]

assert len(code_cells) == 22, {
    "expected_code_cell_count": 22,
    "actual_code_cell_count": len(
        code_cells
    ),
    "first_lines": [
        (
            "".join(
                cell.get("source", [])
            ).splitlines()[0]
            if "".join(
                cell.get("source", [])
            ).splitlines()
            else "EMPTY"
        )
        for cell in code_cells
    ],
}

code_cell_numbers = []
unnumbered_code_cells = []

for position, cell in enumerate(
    code_cells,
    start=1,
):
    source = "".join(
        cell.get("source", [])
    )

    first_line = (
        source.splitlines()[0]
        if source.splitlines()
        else ""
    )

    match = re.fullmatch(
        r"# Code Cell (\d{2}) — .+",
        first_line,
    )

    if match is None:
        unnumbered_code_cells.append(
            {
                "position": position,
                "first_line": (
                    first_line or "EMPTY"
                ),
            }
        )
    else:
        code_cell_numbers.append(
            int(match.group(1))
        )

assert not unnumbered_code_cells, (
    unnumbered_code_cells
)

assert code_cell_numbers == list(
    range(1, 23)
), code_cell_numbers

syntax_failures = []

for code_number, cell in zip(
    code_cell_numbers,
    code_cells,
):
    source = "".join(
        cell.get("source", [])
    )

    try:
        compile(
            source,
            (
                f"17_reranking.ipynb"
                f"::CodeCell{code_number:02d}"
            ),
            "exec",
        )
    except SyntaxError as error:
        syntax_failures.append(
            {
                "code_cell": (
                    code_number
                ),
                "line": error.lineno,
                "message": error.msg,
            }
        )

assert not syntax_failures, syntax_failures

error_outputs = []

for code_number, cell in zip(
    code_cell_numbers,
    code_cells,
):
    if code_number == 22:
        continue

    for output in cell.get(
        "outputs",
        [],
    ):
        if output.get(
            "output_type"
        ) == "error":
            error_outputs.append(
                {
                    "code_cell": (
                        code_number
                    ),
                    "ename": output.get(
                        "ename"
                    ),
                    "evalue": output.get(
                        "evalue"
                    ),
                }
            )

assert not error_outputs, error_outputs

unexecuted_preceding_cells = [
    code_number
    for code_number, cell
    in zip(
        code_cell_numbers,
        code_cells,
    )
    if (
        code_number < 22
        and cell.get(
            "execution_count"
        )
        is None
    )
]

assert not unexecuted_preceding_cells, (
    unexecuted_preceding_cells
)

invalid_rtl_markdown_cells = []

for markdown_position, cell in enumerate(
    markdown_cells,
    start=1,
):
    source = "".join(
        cell.get("source", [])
    ).strip()

    if not (
        source.startswith(
            '<div dir="rtl" align="right">'
        )
        and source.endswith("</div>")
    ):
        invalid_rtl_markdown_cells.append(
            markdown_position
        )

assert not invalid_rtl_markdown_cells, (
    invalid_rtl_markdown_cells
)

expected_final_hashes = {
    RERANKER_CONFIGURATION_PATH: (
        "dd2290188c4e99be1dada40b167b1041"
        "c69f7f077ebe6dea1fddcb58b4834a86"
    ),
    RERANKED_CANDIDATES_PATH: (
        "d00285bbf3865fbd51041db1612b9fdc"
        "17e9778273ea6d136978906a07268b7a"
    ),
    RERANKER_RELOAD_VALIDATION_PATH: (
        "d575550c0d0a3feadfa27025c4f5b359"
        "eed5ddac4f91c1e720e303e49c188e4a"
    ),
    RERANKER_MANIFEST_PATH: (
        "230a720eab3fac9ef9f4fea8ab0315266"
        "0f045fbda8926e22abcb37f809e6be5"
    ),
    RERANKER_RELEASE_PATH: (
        "8f9006a2cbc425aaa68459928af1c15ab"
        "31e88e881d8fbd76a5b312dd601a9b4"
    ),
    NB18_RERANKER_HANDOFF_PATH: (
        "6512c929587d6926da18e713a5c1a5ba"
        "a0da3d863bf6ca389ad7cc90585e5f9d"
    ),
    RERANKER_ACCEPTANCE_PATH: (
        "ab04570ed0c7a6bee37b367bb3c62564"
        "f75e23c80e0f3816538449872ecc344d"
    ),
}

for path, expected_sha256 in (
    expected_final_hashes.items()
):
    assert path.is_file(), str(path)

    actual_sha256 = sha256_path_file(
        path
    )

    assert (
        actual_sha256
        == expected_sha256
    ), {
        "path": str(path),
        "expected": expected_sha256,
        "actual": actual_sha256,
    }

final_configuration = json.loads(
    RERANKER_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

final_reload_validation = json.loads(
    RERANKER_RELOAD_VALIDATION_PATH
    .read_text(
        encoding="utf-8"
    )
)

final_manifest_payload = json.loads(
    RERANKER_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

final_release_payload = json.loads(
    RERANKER_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

final_handoff_payload = json.loads(
    NB18_RERANKER_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

final_acceptance_payload = json.loads(
    RERANKER_ACCEPTANCE_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    final_configuration[
        "configuration_status"
    ]
    == "FROZEN_BEFORE_TEST_ACCESS"
)

assert (
    final_configuration[
        "configuration_sha256"
    ]
    == (
        "8924c47c96bc52080ac0306daf272185a"
        "0c7c8eeb07d15a963058530badfa738"
    )
)

assert (
    final_reload_validation[
        "reload_validation_status"
    ]
    == "PASS"
)

assert (
    final_manifest_payload[
        "manifest_status"
    ]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    final_release_payload["status"]
    == "FROZEN_FOR_CONTEXT_BUILDING"
)

assert (
    final_handoff_payload["status"]
    == "READY_FOR_NB18"
)

assert (
    final_handoff_payload[
        "target_notebook"
    ]
    == "18_context_builder.ipynb"
)

assert (
    final_acceptance_payload["status"]
    == "PASS_COMPLETE_FROZEN"
)

assert RERANKER_CONFIGURATION_FROZEN
assert TEST_QUERIES_ACCESSED

nb17_final_health = {
    "notebook": "17_reranking.ipynb",
    "nb17_objective": "PASS",
    "execution_health": "PASS",
    "syntax_health": "PASS",
    "markdown_rtl_health": "PASS",
    "code_cell_count": len(
        code_cells
    ),
    "code_cell_numbering": (
        "01_TO_22_PASS"
    ),
    "canonical_input_gate": "PASS",
    "pilot_gate": "PASS",
    "onnx_parity_gate": "PASS",
    "development_gate": "PASS",
    "source_slice_gate": "PASS",
    "bootstrap_gate": "PASS",
    "failure_analysis_gate": "PASS",
    "configuration_freeze_gate": "PASS",
    "independent_test_gate": "PASS",
    "reload_validation_gate": "PASS",
    "candidate_record_count": 3813,
    "development_query_count": 2664,
    "test_query_count": 1149,
    "candidate_depth": 50,
    "selected_configuration": {
        "model_id": (
            final_configuration[
                "reranker"
            ]["model_id"]
        ),
        "revision": (
            final_configuration[
                "reranker"
            ]["revision"]
        ),
        "runtime": "ONNX_INT8_AVX2",
        "batch_size": 4,
        "maximum_sequence_length": 512,
        "final_top_k": 10,
    },
    "development_metrics": (
        full_development_reranker_metrics
    ),
    "test_metrics": (
        test_reranker_metrics
    ),
    "configuration_sha256": (
        reranker_configuration_sha256
    ),
    "logical_reranker_output_sha256": (
        reranked_candidates_file_sha256
    ),
    "manifest": {
        "path": str(
            RERANKER_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_manifest_sha256
        ),
        "status": (
            "PASS_COMPLETE_FROZEN"
        ),
    },
    "release": {
        "path": str(
            RERANKER_RELEASE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_release_sha256
        ),
        "status": (
            "FROZEN_FOR_CONTEXT_BUILDING"
        ),
    },
    "nb18_handoff": {
        "path": str(
            NB18_RERANKER_HANDOFF_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_nb18_handoff_sha256
        ),
        "status": "READY_FOR_NB18",
    },
    "acceptance": {
        "path": str(
            RERANKER_ACCEPTANCE_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "sha256": (
            final_acceptance_sha256
        ),
        "status": (
            "PASS_COMPLETE_FROZEN"
        ),
    },
    "configuration_changed_after_test": (
        False
    ),
    "paid_api_calls": 0,
    "next_execution_notebook": (
        "18_context_builder.ipynb"
    ),
    "final_status": (
        "PASS_COMPLETE_RELOAD_VALIDATED_FROZEN"
    ),
}

print(
    json.dumps(
        nb17_final_health,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "\nNB17 FINAL HEALTH: PASS — "
    "COMPLETE / RELOAD-VALIDATED / "
    "FROZEN / READY FOR NB18"
)

{
  "notebook": "17_reranking.ipynb",
  "nb17_objective": "PASS",
  "execution_health": "PASS",
  "syntax_health": "PASS",
  "markdown_rtl_health": "PASS",
  "code_cell_count": 22,
  "code_cell_numbering": "01_TO_22_PASS",
  "canonical_input_gate": "PASS",
  "pilot_gate": "PASS",
  "onnx_parity_gate": "PASS",
  "development_gate": "PASS",
  "source_slice_gate": "PASS",
  "bootstrap_gate": "PASS",
  "failure_analysis_gate": "PASS",
  "configuration_freeze_gate": "PASS",
  "independent_test_gate": "PASS",
  "reload_validation_gate": "PASS",
  "candidate_record_count": 3813,
  "development_query_count": 2664,
  "test_query_count": 1149,
  "candidate_depth": 50,
  "selected_configuration": {
    "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "runtime": "ONNX_INT8_AVX2",
    "batch_size": 4,
    "maximum_sequence_length": 512,
    "final_top_k": 10
  },
  "development_metrics": {
    "hit_at_10": 0.8498498498498499,


<div dir="rtl" align="right">

# جمع‌بندی نهایی نوت‌بوک ۱۷ — بازرتبه‌بندی نتایج بازیابی

## وضعیت نهایی

نوت‌بوک `17_reranking.ipynb` با موفقیت کامل شد.

**وضعیت رسمی مرحله:**

`PASS_COMPLETE_RELOAD_VALIDATED_FROZEN`

تمام بررسی‌های اجرایی، نحوی، قراردادی، آماری و بازخوانی مستقل خروجی‌ها با موفقیت انجام شدند. پیکربندی نهایی بازرتبه‌بند پیش از دسترسی به مجموعه‌ی Test تثبیت شد و پس از ارزیابی Test هیچ تغییری در آن ایجاد نشد.

---

## هدف این مرحله

هدف نوت‌بوک ۱۷، بازرتبه‌بندی خروجی ثابت‌شده‌ی بازیابی ترکیبی NB16 بود تا مرتبط‌ترین قطعه‌ها با دقت بیشتری در رتبه‌های ابتدایی قرار گیرند.

این مرحله شامل موارد زیر بود:

- اعتبارسنجی ورودی‌های ثابت‌شده‌ی NB16؛
- بازسازی نگاشت دقیق شواهد طلایی به قطعه‌های مرجع؛
- ثبت و ارزیابی گزینه‌های بازرتبه‌بندی؛
- تثبیت دقیق نسخه‌ی مدل؛
- ارزیابی اجرای PyTorch و ONNX؛
- اجرای Pilot فقط روی Development؛
- انتخاب عمق مناسب بازرتبه‌بندی؛
- اجرای کامل Development؛
- تحلیل Bootstrap، Source Slice و موارد افت کیفیت؛
- تثبیت پیکربندی و ADR پیش از Test؛
- ارزیابی مستقل و یک‌باره روی Test؛
- تولید Artifactهای نسخه‌بندی‌شده؛
- بازخوانی مستقل و اعتبارسنجی نهایی خروجی‌ها.

---

## پیکربندی منتخب

| مؤلفه | مقدار نهایی |
|---|---|
| مدل | `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` |
| Revision | `1427fd652930e4ba29e8149678df786c240d8825` |
| Runtime | `ONNX_INT8_AVX2` |
| Candidate Depth | `50` |
| Batch Size | `4` |
| Maximum Sequence Length | `512` |
| Final Top-K | `10` |
| دستگاه اجرا | `CPU` |

عمق ۵۰ پس از ارزیابی Pilot انتخاب شد. عمق ۱۰۰ ارزش افزوده‌ای نسبت به عمق ۵۰ نشان نداد و در Pilot افت معناداری ایجاد کرد؛ بنابراین هزینه‌ی پردازشی بیشتر آن پذیرفته نشد.

---

## نتایج Development

| معیار | Hybrid Baseline | Reranker | تغییر |
|---|---:|---:|---:|
| Hit@5 | 0.711712 | 0.791667 | +0.079955 |
| Hit@10 | 0.785661 | 0.849850 | +0.064189 |
| nDCG@5 | 0.511560 | 0.573277 | +0.061717 |
| nDCG@10 | 0.538173 | 0.598834 | +0.060661 |
| Precision@5 | 0.186411 | 0.201577 | +0.015165 |
| Precision@10 | 0.107620 | 0.115015 | +0.007395 |
| Recall@5 | 0.546903 | 0.597787 | +0.050885 |
| Recall@10 | 0.621243 | 0.667674 | +0.046432 |
| Reciprocal Rank | 0.608217 | 0.691327 | +0.083110 |

بهبود `nDCG@10` روی Development برابر با `+0.060661` بود.

بازه‌ی اطمینان ۹۵٪ Bootstrap برای این بهبود:

`[0.051491, 0.070226]`

احتمال مثبت‌بودن تغییر:

`1.0`

بنابراین بهبود کیفیت روی Development از نظر آماری پایدار ارزیابی شد.

---

## نتایج مستقل Test

| معیار | Hybrid Baseline | Reranker | تغییر |
|---|---:|---:|---:|
| Hit@5 | 0.696258 | 0.749347 | +0.053090 |
| Hit@10 | 0.755440 | 0.803307 | +0.047868 |
| nDCG@5 | 0.498341 | 0.549614 | +0.051273 |
| nDCG@10 | 0.521665 | 0.573536 | +0.051871 |
| Precision@5 | 0.180505 | 0.189208 | +0.008703 |
| Precision@10 | 0.102089 | 0.108094 | +0.006005 |
| Recall@5 | 0.533661 | 0.570173 | +0.036512 |
| Recall@10 | 0.599858 | 0.634318 | +0.034460 |
| Reciprocal Rank | 0.589917 | 0.660238 | +0.070322 |

بهبود `nDCG@10` روی Test برابر با `+0.051871` بود.

بازه‌ی اطمینان ۹۵٪ Bootstrap:

`[0.037259, 0.066528]`

احتمال مثبت‌بودن تغییر:

`1.0`

این نتایج نشان می‌دهند که بهبود مشاهده‌شده روی Development به مجموعه‌ی مستقل Test نیز تعمیم یافته است.

---

## تحلیل بر اساس نوع منبع

بازرتبه‌بند روی هر سه نوع منبع بهبود مثبت `nDCG@10` ایجاد کرد:

| نوع منبع | تغییر nDCG@10 در Development | تغییر nDCG@10 در Test |
|---|---:|---:|
| کتاب | +0.053190 | +0.079510 |
| رونوشت | +0.084014 | +0.063258 |
| وب‌سایت | +0.026177 | +0.025849 |

هیچ Source Slice اصلی با افت `nDCG@10` مواجه نشد. کمترین بهبود مربوط به منابع وب‌سایت بود، اما همان بهبود نیز در Development و Test مثبت باقی ماند.

---

## تحلیل موارد افت

در Development:

- ۸۹۳ پرسش بهبود یافتند؛
- ۱۲۸۷ پرسش بدون تغییر باقی ماندند؛
- ۴۸۴ پرسش افت کردند.

وجود افت در بخشی از پرسش‌ها ثبت و تحلیل شد، اما بهبود کلی، بهبود تمام Source Sliceها و نتیجه‌ی Bootstrap نشان دادند که پیکربندی منتخب واجد شرایط انتشار است.

این تحلیل به‌عنوان محدودیت شناخته‌شده‌ی نسخه‌ی فعلی ثبت شده و هیچ تغییری پس از مشاهده‌ی Test در پیکربندی ایجاد نشده است.

---

## تمامیت و بازتولیدپذیری

- تعداد کل رکوردهای خروجی: `3813`
- تعداد پرسش‌های Development: `2664`
- تعداد پرسش‌های Test: `1149`
- عمق خروجی بازرتبه‌بندی‌شده: `50`
- تعداد API پولی: `0`
- تغییر پیکربندی پس از Test: `false`

شناسه‌ی محتوایی پیکربندی:

`8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738`

شناسه‌ی منطقی خروجی بازرتبه‌بندی:

`d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a`

اعتبارسنجی مستقل Reload نیز با وضعیت `PASS` کامل شد.

---

## خروجی‌های نهایی

- خروجی بازرتبه‌بندی‌شده:

  `data/phase2/retrieval/reranker_v1/reranked_candidates_v1.jsonl`

- پیکربندی ثابت‌شده:

  `artifacts/phase2/reranking/reranker_configuration_v1.json`

- گزارش تصمیم معماری:

  `reports/phase2/reranker_adr_v1.json`

- Manifest نهایی:

  `data/phase2/retrieval/reranker_v1/reranker_manifest.json`

- Release نهایی:

  `artifacts/phase2/reranking/reranker_release_v1.json`

- گزارش پذیرش:

  `reports/phase2/reranker_acceptance_v1.json`

- اعتبارسنجی Reload:

  `data/phase2/retrieval/reranker_v1/reranker_reload_validation_v1.json`

- قرارداد تحویل به مرحله‌ی بعد:

  `artifacts/phase2/reranking/nb18_reranker_handoff_v1.json`

---

## تصمیم نهایی

پیکربندی بازرتبه‌بند پذیرفته و ثابت شد.

Release نهایی با وضعیت زیر منتشر شده است:

`FROZEN_FOR_CONTEXT_BUILDING`

قرارداد تحویل مرحله‌ی بعد نیز دارای وضعیت زیر است:

`READY_FOR_NB18`

در نتیجه، نوت‌بوک ۱۷ کامل شده و ادامه‌ی تغییر یا تنظیم مدل بر اساس نتایج Test مجاز نیست.

---

## مرحله‌ی بعد

مرحله‌ی بعدی پروژه:

`18_context_builder.ipynb`

در NB18، خروجی ثابت‌شده‌ی بازرتبه‌بند برای ساخت Context نهایی مورد استفاده قرار می‌گیرد. تمرکز مرحله‌ی بعد بر انتخاب شواهد، مدیریت بودجه‌ی Context، حفظ تنوع منابع، حذف افزونگی و آماده‌سازی ورودی مستند برای مرحله‌ی تولید پاسخ خواهد بود.

---

## نتیجه نهایی

**NB17 FINAL HEALTH: PASS**

**COMPLETE / RELOAD-VALIDATED / FROZEN / READY FOR NB18**

</div>